# Reproducible ASM2d-TSN Surrogate Benchmark

This is the sole executable analysis artifact for the study. It reads immutable configuration and scientific inputs, generates mechanistic in-distribution and extrapolation datasets, evaluates 13 CPU-based statistical surrogate methods, applies positivity-preserving conservative projection, and writes a checksummed result bundle.

The configured `full` profile is the only article-eligible profile. Set `ICSOR_PROFILE=smoke` for a small infrastructure validation. Smoke outputs are always marked non-final and must never supply manuscript results.


In [ ]:
from __future__ import annotations

import contextlib
import copy
import hashlib
import importlib.metadata
import io
import json
import math
import os
import pickle
import platform
import re
import random
import shutil
import socket
import subprocess
import sys
import tempfile
import time
import warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import scipy
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from scipy.integrate import solve_ivp
from scipy.linalg import null_space
from scipy.optimize import least_squares
from scipy.stats.qmc import LatinHypercube, scale as qmc_scale
from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import MultiTaskElasticNet, MultiTaskLasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMRegressor was fitted with feature names",
    category=UserWarning,
)


ROOT = Path.cwd().resolve()
if not (ROOT / "config").is_dir() or not (ROOT / "data").is_dir():
    raise RuntimeError("Run the notebook from the project root containing config/ and data/.")


def read_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"Expected a JSON object in {path}.")
    return value


CONFIG = read_json(ROOT / "config" / "params.json")
PATHS = read_json(ROOT / "config" / "paths.json")
EXPECTED_TOP_LEVEL = {"run", "simulation", "projection", "evaluation"}
if set(CONFIG) != EXPECTED_TOP_LEVEL:
    raise ValueError(f"Configuration keys must be exactly {sorted(EXPECTED_TOP_LEVEL)}; got {sorted(CONFIG)}.")


def require_exact_keys(value: Mapping[str, Any], expected: set[str], location: str) -> None:
    actual = set(value)
    if actual != expected:
        missing = sorted(expected - actual)
        unknown = sorted(actual - expected)
        raise ValueError(f"Invalid keys at {location}: missing={missing}, unknown={unknown}.")


require_exact_keys(
    CONFIG["run"],
    {"run_id", "default_profile", "profile_env_var", "resume", "resume_policy", "threads", "solver_chains", "hardware_profile", "profiles"},
    "run",
)
require_exact_keys(
    CONFIG["simulation"],
    {"schema_version", "sampling", "operational_columns", "operational_ranges", "ood", "solver", "influent_state_ranges", "aeration_model", "process_types", "workbook"},
    "simulation",
)
require_exact_keys(
    CONFIG["simulation"]["sampling"],
    {"id_samples", "seed", "n_chains", "max_sample_attempts", "parallel_workers", "parallel_chunk_size"},
    "simulation.sampling",
)
require_exact_keys(CONFIG["simulation"]["ood"], {"candidate_mix", "other_variables", "regimes"}, "simulation.ood")
require_exact_keys(CONFIG["simulation"]["ood"]["candidate_mix"], {"one_variable", "two_variables"}, "simulation.ood.candidate_mix")
require_exact_keys(CONFIG["simulation"]["ood"]["regimes"], {"mild", "severe"}, "simulation.ood.regimes")
for regime_name, regime_config in CONFIG["simulation"]["ood"]["regimes"].items():
    require_exact_keys(regime_config, {"n_samples", "ranges"}, f"simulation.ood.regimes.{regime_name}")
    require_exact_keys(regime_config["ranges"], {"HRT", "Aeration", "S_NH4", "X_S"}, f"simulation.ood.regimes.{regime_name}.ranges")
solver_keys = {
    "lower_bound", "upper_bound", "initial_guess_floor", "warm_start_previous_weight",
    "warm_start_influent_weight", "initial_s_a_fraction", "initial_s_f_fraction",
    "initial_heterotroph_to_xs_ratio", "initial_pao_to_pp_ratio", "initial_aob_to_nh4_ratio",
    "initial_nob_to_no2_ratio", "multistart_s_a_fraction", "multistart_s_f_fraction",
    "multistart_heterotroph_to_xs_ratio", "multistart_pao_to_pp_ratio",
    "multistart_aob_to_nh4_ratio", "multistart_nob_to_no2_ratio", "dynamic_relaxation_days",
    "dynamic_absolute_tolerance", "dynamic_relative_tolerance", "dynamic_max_step",
    "residual_tolerance", "variable_tolerance", "gradient_tolerance",
    "acceptance_residual_max", "max_nfev",
}
require_exact_keys(CONFIG["simulation"]["solver"], solver_keys, "simulation.solver")
require_exact_keys(CONFIG["simulation"]["operational_ranges"], {"HRT", "Aeration"}, "simulation.operational_ranges")
require_exact_keys(CONFIG["simulation"]["aeration_model"], {"do_saturation", "kla_base", "kla_per_aeration"}, "simulation.aeration_model")
workbook_config = CONFIG["simulation"]["workbook"]
require_exact_keys(
    workbook_config,
    {"sheets", "dissolved_state_columns", "particulate_state_columns", "state_columns", "state_units", "processes", "parameters"},
    "simulation.workbook",
)
state_names = list(workbook_config["state_columns"])
if len(state_names) != 20 or set(CONFIG["simulation"]["influent_state_ranges"]) != set(state_names):
    raise ValueError("The influent-range schema must match the 20 workbook state columns.")
if set(workbook_config["state_units"]) != set(state_names):
    raise ValueError("The state-unit schema must match the 20 workbook state columns.")
if len(workbook_config["processes"]) != 28 or len(CONFIG["simulation"]["process_types"]) != 28:
    raise ValueError("The process schema must contain exactly 28 ordered processes.")
for index, process in enumerate(workbook_config["processes"], start=1):
    require_exact_keys(process, {"index", "name"}, f"simulation.workbook.processes[{index - 1}]")
    if int(process["index"]) != index or not str(process["name"]).strip():
        raise ValueError("Workbook process indices and names must be complete and ordered.")
for index, parameter in enumerate(workbook_config["parameters"]):
    require_exact_keys(parameter, {"category", "symbol", "excel_name", "description", "value", "unit"}, f"simulation.workbook.parameters[{index}]")
if len(workbook_config["parameters"]) != 81:
    raise ValueError("The workbook configuration must declare exactly 81 parameters.")

require_exact_keys(
    CONFIG["projection"],
    {"method", "rank_tolerance_policy", "primary_solver", "fallback_solver", "normal_equations", "eps", "eta", "tau", "svd_tolerance"},
    "projection",
)
require_exact_keys(CONFIG["evaluation"], {"seed", "nested_cv", "timing", "scaling_policies", "reporting", "models"}, "evaluation")
require_exact_keys(
    CONFIG["evaluation"]["nested_cv"],
    {"sample_sizes", "outer_folds", "inner_folds", "n_trials", "shuffle", "optimization_metric", "direction"},
    "evaluation.nested_cv",
)
require_exact_keys(CONFIG["evaluation"]["timing"], {"batch_size", "warmup_runs", "measured_runs"}, "evaluation.timing")
require_exact_keys(
    CONFIG["evaluation"]["reporting"],
    {"fold_summary", "interpretation", "confidence_intervals", "zero_variance_policy", "ood_normalization", "metrics"},
    "evaluation.reporting",
)
for policy_name, policy in CONFIG["evaluation"]["scaling_policies"].items():
    require_exact_keys(policy, {"features", "targets"}, f"evaluation.scaling_policies.{policy_name}")
require_exact_keys(PATHS, {"workbook", "results_root", "run_root_pattern"}, "paths")
for model_name, model_config in CONFIG["evaluation"]["models"].items():
    require_exact_keys(model_config, {"hyperparameters", "training_defaults", "search_space", "artifact_options"}, f"evaluation.models.{model_name}")
    require_exact_keys(model_config["hyperparameters"], {"random_seed", "scale_features", "scale_targets"}, f"evaluation.models.{model_name}.hyperparameters")
    require_exact_keys(model_config["artifact_options"], {"persist_model", "persist_metrics", "persist_optuna"}, f"evaluation.models.{model_name}.artifact_options")
    if not all(bool(value) for value in model_config["artifact_options"].values()):
        raise ValueError(f"All required artifact options must be enabled for {model_name}.")
    for parameter_name, parameter_spec in model_config["search_space"].items():
        parameter_type = parameter_spec.get("type")
        expected_spec = {"type", "choices"} if parameter_type == "categorical" else {"type", "low", "high", "log"}
        require_exact_keys(parameter_spec, expected_spec, f"evaluation.models.{model_name}.search_space.{parameter_name}")

if set(CONFIG["evaluation"]["models"]) != {
    "xgboost_regressor", "lightgbm_regressor", "catboost_regressor", "adaboost_regressor",
    "random_forest_regressor", "extra_trees_regressor", "svr_regressor", "knn_regressor",
    "pls_regressor", "multitask_elastic_net_regressor", "multitask_lasso_regressor",
    "ann_deep_regressor", "tabnet_regressor",
}:
    raise ValueError("The evaluation schema must declare exactly the 13 benchmark models.")


def deep_merge(base: Mapping[str, Any], override: Mapping[str, Any]) -> dict[str, Any]:
    merged = copy.deepcopy(dict(base))
    for key, value in override.items():
        if isinstance(value, Mapping) and isinstance(merged.get(key), Mapping):
            merged[key] = deep_merge(merged[key], value)
        else:
            merged[key] = copy.deepcopy(value)
    return merged


requested_profile = os.environ.get(
    str(CONFIG["run"].get("profile_env_var", "ICSOR_PROFILE")),
    str(CONFIG["run"].get("default_profile", "full")),
).strip().lower()
profiles = CONFIG["run"].get("profiles", {})
if requested_profile not in {"full", "smoke"}:
    raise ValueError("ICSOR_PROFILE must be 'full' or 'smoke'.")


def validate_profile_override(override: Mapping[str, Any], base: Mapping[str, Any], location: str) -> None:
    for key, value in override.items():
        if location == "profile" and key == "run_id":
            continue
        if key not in base:
            raise ValueError(f"Unknown profile override {location}.{key}.")
        if isinstance(value, Mapping):
            if not isinstance(base[key], Mapping):
                raise ValueError(f"Profile override {location}.{key} has the wrong type.")
            validate_profile_override(value, base[key], f"{location}.{key}")


for profile_name, profile_override in profiles.items():
    if profile_name not in {"full", "smoke"} or not isinstance(profile_override, Mapping):
        raise ValueError(f"Invalid run profile {profile_name!r}.")
    validate_profile_override(profile_override, CONFIG, "profile")

locked_sizes = [500, 1450, 2400, 3350, 4300, 5250, 6200, 7150, 8100, 9050, 10000]
locked_scaling = {
    "xgboost_regressor": (False, False), "lightgbm_regressor": (False, False),
    "catboost_regressor": (False, False), "adaboost_regressor": (False, True),
    "random_forest_regressor": (False, False), "extra_trees_regressor": (False, True),
    "svr_regressor": (True, True), "knn_regressor": (True, True),
    "pls_regressor": (True, True), "multitask_elastic_net_regressor": (True, True),
    "multitask_lasso_regressor": (True, True), "ann_deep_regressor": (True, True),
    "tabnet_regressor": (True, True),
}
sampling_contract = CONFIG["simulation"]["sampling"]
nested_contract = CONFIG["evaluation"]["nested_cv"]
timing_contract = CONFIG["evaluation"]["timing"]
if (
    int(sampling_contract["id_samples"]) != 10_000
    or int(sampling_contract["seed"]) != 42
    or int(sampling_contract["n_chains"]) != 12
    or int(sampling_contract["max_sample_attempts"]) != 25
    or list(nested_contract["sample_sizes"]) != locked_sizes
    or int(nested_contract["outer_folds"]) != 5
    or int(nested_contract["inner_folds"]) != 4
    or int(nested_contract["n_trials"]) != 100
    or str(nested_contract["optimization_metric"]) != "nMSE"
    or not bool(nested_contract["shuffle"])
    or str(nested_contract["direction"]) != "minimize"
    or CONFIG["run"]["hardware_profile"] != "cpu_multicore"
    or (int(timing_contract["batch_size"]), int(timing_contract["warmup_runs"]), int(timing_contract["measured_runs"])) != (512, 5, 30)
):
    raise ValueError("The full-profile sampling, nested-validation, or timing contract has changed.")
for regime_name in ("mild", "severe"):
    if int(CONFIG["simulation"]["ood"]["regimes"][regime_name]["n_samples"]) != 600:
        raise ValueError("Each full-profile extrapolation regime must contain 600 accepted states.")
for model_name, expected_scaling in locked_scaling.items():
    model_hyperparameters = CONFIG["evaluation"]["models"][model_name]["hyperparameters"]
    observed_scaling = (bool(model_hyperparameters["scale_features"]), bool(model_hyperparameters["scale_targets"]))
    declared_policy = CONFIG["evaluation"]["scaling_policies"][model_name]
    policy_scaling = (bool(declared_policy["features"]), bool(declared_policy["targets"]))
    if observed_scaling != expected_scaling or policy_scaling != expected_scaling:
        raise ValueError(f"Scaling policy mismatch for {model_name}.")
if set(CONFIG["evaluation"]["scaling_policies"]) != set(locked_scaling):
    raise ValueError("The scaling-policy schema does not match the 13-model benchmark.")
solver_contract = CONFIG["simulation"]["solver"]
if (
    float(solver_contract["lower_bound"]) != 1e-8
    or float(solver_contract["upper_bound"]) != 1500.0
    or float(solver_contract["residual_tolerance"]) != 1e-9
    or float(solver_contract["variable_tolerance"]) != 1e-9
    or float(solver_contract["gradient_tolerance"]) != 1e-9
    or int(solver_contract["max_nfev"]) != 1200
    or float(solver_contract["dynamic_relaxation_days"]) != 20.0
    or int(CONFIG["run"]["solver_chains"]) != int(sampling_contract["n_chains"])
):
    raise ValueError("The mechanistic solver contract has changed.")
if (
    CONFIG["projection"]["rank_tolerance_policy"] != "max_dimension_times_machine_epsilon_times_sigma_max"
    or CONFIG["projection"]["primary_solver"] != "qr"
    or CONFIG["projection"]["fallback_solver"] != "svd"
    or bool(CONFIG["projection"]["normal_equations"])
    or float(CONFIG["projection"]["eps"]) != 1e-8
    or float(CONFIG["projection"]["eta"]) != 1e-12
    or float(CONFIG["projection"]["tau"]) != 1e-10
    or float(CONFIG["projection"]["svd_tolerance"]) != 1e-12
):
    raise ValueError("The physical-projection contract has changed.")
if (
    float(CONFIG["simulation"]["ood"]["candidate_mix"]["one_variable"]) != 0.7
    or float(CONFIG["simulation"]["ood"]["candidate_mix"]["two_variables"]) != 0.3
):
    raise ValueError("The OOD perturbation-mixture contract has changed.")

ACTIVE = copy.deepcopy(CONFIG)
if requested_profile in profiles:
    profile_override = copy.deepcopy(profiles[requested_profile])
    profile_run_id = profile_override.pop("run_id", None)
    ACTIVE = deep_merge(ACTIVE, profile_override)
    if profile_run_id is not None:
        ACTIVE["run"]["run_id"] = profile_run_id
ACTIVE["run"]["profile"] = requested_profile
if ACTIVE["run"]["resume_policy"] != "matching_contract_only":
    raise ValueError("Only matching-contract resume is supported.")
if int(ACTIVE["run"]["threads"]) != int(ACTIVE["simulation"]["sampling"]["parallel_workers"]):
    raise ValueError("run.threads must match simulation.sampling.parallel_workers.")
if int(ACTIVE["run"]["solver_chains"]) != int(ACTIVE["simulation"]["sampling"]["n_chains"]):
    raise ValueError("run.solver_chains must match simulation.sampling.n_chains.")

run_id_default = "article_final_v2" if requested_profile == "full" else "article_smoke_13model_v1"
RUN_ID = os.environ.get("ICSOR_RUN_ID", str(ACTIVE["run"].get("run_id", run_id_default))).strip()
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", RUN_ID):
    raise ValueError("run_id may contain only letters, numbers, dot, underscore, and hyphen.")


def relative_input_path(key: str) -> Path:
    raw = Path(str(PATHS[key]))
    if raw.is_absolute() or ".." in raw.parts:
        raise ValueError(f"Configured path {key!r} must be a safe relative path.")
    resolved = (ROOT / raw).resolve()
    if ROOT not in resolved.parents and resolved != ROOT:
        raise ValueError(f"Configured path {key!r} escapes the project root.")
    return resolved


WORKBOOK_PATH = relative_input_path("workbook")
RESULTS_ROOT = relative_input_path("results_root")
if PATHS["run_root_pattern"] != "results/{run_id}":
    raise ValueError("The run-root pattern must be results/{run_id}.")
if RESULTS_ROOT != (ROOT / "results").resolve():
    raise ValueError("All analytical writes must use the project results/ directory.")
RUN_ROOT = (RESULTS_ROOT / RUN_ID).resolve()
if RESULTS_ROOT != RUN_ROOT and RESULTS_ROOT not in RUN_ROOT.parents:
    raise ValueError("The run output must remain beneath the configured results root.")
PREEXISTING_RUN_ENTRIES = sorted(item.name for item in RUN_ROOT.iterdir()) if RUN_ROOT.exists() else []
RUN_ROOT.mkdir(parents=True, exist_ok=True)

DIRS = {
    name: RUN_ROOT / name
    for name in (
        "inputs", "datasets", "matrices", "splits", "tuning", "predictions",
        "metrics", "timing", "models", "article", "logs"
    )
}
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_json(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def atomic_bytes(path: Path, payload: bytes) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "wb") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        with contextlib.suppress(FileNotFoundError):
            os.unlink(temporary)
    return path


def atomic_json(path: Path, value: Any) -> Path:
    return atomic_bytes(path, (json.dumps(value, indent=2, sort_keys=True, default=str) + "\n").encode("utf-8"))


def atomic_csv(path: Path, frame: pd.DataFrame, *, index: bool = False) -> Path:
    return atomic_bytes(path, frame.to_csv(index=index).encode("utf-8"))


def atomic_parquet(path: Path, frame: pd.DataFrame) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.parent / f".{path.name}.{os.getpid()}.tmp"
    try:
        frame.to_parquet(temporary, index=False)
        os.replace(temporary, path)
    finally:
        with contextlib.suppress(FileNotFoundError):
            temporary.unlink()
    return path


def atomic_npz(path: Path, **arrays: np.ndarray) -> Path:
    buffer = io.BytesIO()
    np.savez_compressed(buffer, **arrays)
    return atomic_bytes(path, buffer.getvalue())


def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


if sys.version_info[:2] != (3, 12):
    raise RuntimeError("The authoritative environment requires Python 3.12.")


np.random.seed(int(CONFIG["evaluation"]["seed"]))
CONFIG_HASH = sha256_json(ACTIVE)
WORKBOOK_HASH = sha256_file(WORKBOOK_PATH)
NOTEBOOK_HASH = sha256_file(ROOT / "main.ipynb")
LOCK_HASH = sha256_file(ROOT / "uv.lock")
PATHS_HASH = sha256_json(PATHS)
ARTICLE_ELIGIBLE = requested_profile == "full"

manifest_path = RUN_ROOT / "manifest.json"
ENVIRONMENT_CONTRACT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "hardware_profile": ACTIVE["run"]["hardware_profile"],
    "threads": int(ACTIVE["run"]["threads"]),
    "processor": platform.processor(),
    "physical_cores": psutil.cpu_count(logical=False),
    "logical_cores": psutil.cpu_count(logical=True),
    "ram_bytes": psutil.virtual_memory().total,
}
RUN_CONTRACT = {
    "config_sha256": CONFIG_HASH,
    "workbook_sha256": WORKBOOK_HASH,
    "notebook_sha256": NOTEBOOK_HASH,
    "dependency_lock_sha256": LOCK_HASH,
    "paths_sha256": PATHS_HASH,
    "environment_sha256": sha256_json(ENVIRONMENT_CONTRACT),
}
existing_manifest = read_json(manifest_path) if manifest_path.exists() else None
if existing_manifest is None and PREEXISTING_RUN_ENTRIES:
    raise RuntimeError(f"Refusing an orphaned nonempty run root: {PREEXISTING_RUN_ENTRIES}.")
if existing_manifest and existing_manifest.get("status") == "complete":
    raise RuntimeError("A completed run is immutable; choose a new run_id to rerun it.")
if existing_manifest and existing_manifest.get("contract") != RUN_CONTRACT:
    raise RuntimeError("Cannot resume a run created by a different configuration, input, notebook, lock, or environment contract.")

MANIFEST: dict[str, Any] = existing_manifest or {
    "schema_version": "1.0",
    "run_id": RUN_ID,
    "profile": requested_profile,
    "status": "running",
    "article_eligible": ARTICLE_ELIGIBLE,
    "started_utc": utc_now(),
    "contract": RUN_CONTRACT,
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "hostname": socket.gethostname(),
        "cpu": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_cores": psutil.cpu_count(logical=True),
        "ram_bytes": psutil.virtual_memory().total,
        "packages": {
            name: package_version(name)
            for name in (
                "numpy", "scipy", "pandas", "pyarrow", "openpyxl", "scikit-learn",
                "optuna", "xgboost", "lightgbm", "catboost", "matplotlib",
                "pytorch-tabnet", "torch"
            )
        },
    },
    "stages": {},
    "artifacts": {},
}


def save_manifest() -> None:
    MANIFEST["updated_utc"] = utc_now()
    atomic_json(manifest_path, MANIFEST)


def register_artifact(name: str, path: Path) -> None:
    resolved = path.resolve()
    if RUN_ROOT not in resolved.parents:
        raise ValueError("Only run-root artifacts may be registered.")
    MANIFEST["artifacts"][name] = {
        "path": resolved.relative_to(RUN_ROOT).as_posix(),
        "bytes": resolved.stat().st_size,
        "sha256": sha256_file(resolved),
    }
    save_manifest()


def stage_complete(name: str, **details: Any) -> None:
    MANIFEST["stages"][name] = {"status": "complete", "completed_utc": utc_now(), **details}
    save_manifest()


input_snapshots = {
    DIRS["inputs"] / "params.resolved.json": (json.dumps(ACTIVE, indent=2, sort_keys=True, default=str) + "\n").encode("utf-8"),
    DIRS["inputs"] / "paths.snapshot.json": (json.dumps(PATHS, indent=2, sort_keys=True, default=str) + "\n").encode("utf-8"),
    DIRS["inputs"] / "workbook.snapshot.xlsx": WORKBOOK_PATH.read_bytes(),
}
for snapshot_path, payload in input_snapshots.items():
    if snapshot_path.exists():
        if hashlib.sha256(snapshot_path.read_bytes()).digest() != hashlib.sha256(payload).digest():
            raise RuntimeError(f"Existing immutable input snapshot changed: {snapshot_path.name}.")
    else:
        atomic_bytes(snapshot_path, payload)
register_artifact("resolved_configuration", DIRS["inputs"] / "params.resolved.json")
register_artifact("paths_snapshot", DIRS["inputs"] / "paths.snapshot.json")
register_artifact("workbook_snapshot", DIRS["inputs"] / "workbook.snapshot.xlsx")
save_manifest()

print(f"Profile: {requested_profile}")
print(f"Run root: {RUN_ROOT.relative_to(ROOT)}")
print(f"Article eligible at startup: {ARTICLE_ELIGIBLE}")
print(f"Hardware profile: {ACTIVE['run']['hardware_profile']}")


## Mechanistic model and workbook contract

The following cell contains the complete ASM2d-TSN stoichiometry, composition handling, kinetic rates, initialization rules, and bounded steady-state solver. Reaction rates are evaluated without an artificial cap. The workbook is independently re-evaluated and compared with the assembled matrices before simulation begins.


In [ ]:
# Compatibility helpers used only by the embedded standalone definitions below.
SIM_PARAMS = copy.deepcopy(ACTIVE["simulation"])
SIM_PARAMS["hyperparameters"] = {
    "n_samples": int(ACTIVE["simulation"]["sampling"]["id_samples"]),
    "seed": int(ACTIVE["simulation"]["sampling"]["seed"]),
    "max_sample_attempts": int(ACTIVE["simulation"]["sampling"]["max_sample_attempts"]),
    "parallel_workers": int(ACTIVE["simulation"]["sampling"]["parallel_workers"]),
    "parallel_chunk_size": int(ACTIVE["simulation"]["sampling"]["parallel_chunk_size"]),
}


def get_repo_root(repo_root: str | Path | None = None) -> Path:
    return Path(repo_root).resolve() if repo_root is not None else ROOT


def load_json_file(path: str | Path) -> dict[str, Any]:
    return read_json(Path(path))


def load_pickle_file(path: str | Path) -> Any:
    with Path(path).open("rb") as handle:
        return pickle.load(handle)


def save_json_file(path: str | Path, data: dict[str, Any], *, indent: int = 2) -> Path:
    return atomic_json(Path(path), data)


def save_pickle_file(path: str | Path, data: Any) -> Path:
    return atomic_bytes(Path(path), pickle.dumps(data, protocol=5))


def compute_file_sha256(path: str | Path, *, chunk_size: int = 1024 * 1024) -> str:
    return sha256_file(Path(path), chunk_size=chunk_size)


def load_model_params(model_name: str, repo_root: str | Path | None = None) -> dict[str, Any]:
    if model_name != "asm2d_tsn_simulation":
        raise KeyError(model_name)
    return copy.deepcopy(SIM_PARAMS)


def load_paths_config(repo_root: str | Path | None = None) -> dict[str, Any]:
    return {
        "asm2d_tsn_reference_workbook": PATHS["workbook"],
        "asm2d_tsn_composition_cache_pattern": str(
            (DIRS["matrices"] / "composition_cache_{workbook_hash}.pkl").relative_to(ROOT)
        ),
        "asm2d_tsn_composition_cache_metadata_pattern": str(
            (DIRS["matrices"] / "composition_cache_{workbook_hash}.json").relative_to(ROOT)
        ),
        "asm2d_tsn_simulation_data_pattern": str((DIRS["datasets"] / "id.parquet").relative_to(ROOT)),
        "asm2d_tsn_simulation_metadata_pattern": str((DIRS["datasets"] / "id.metadata.json").relative_to(ROOT)),
    }


def render_simulation_artifact_paths(*args: Any, **kwargs: Any) -> tuple[Path, Path]:
    return DIRS["datasets"] / "id.parquet", DIRS["datasets"] / "id.metadata.json"


def save_simulation_artifacts(*args: Any, **kwargs: Any) -> tuple[Path, Path]:
    raise RuntimeError("The standalone workflow uses its atomic result-bundle persistence layer.")

MODEL_NAME = "asm2d_tsn_simulation"
WORKBOOK_PATH_KEY = "asm2d_tsn_reference_workbook"
DATA_PATTERN_KEY = "asm2d_tsn_simulation_data_pattern"
METADATA_PATTERN_KEY = "asm2d_tsn_simulation_metadata_pattern"
COMPOSITION_CACHE_PATTERN_KEY = "asm2d_tsn_composition_cache_pattern"
COMPOSITION_CACHE_METADATA_PATTERN_KEY = "asm2d_tsn_composition_cache_metadata_pattern"
STOICHIOMETRIC_SHEET_NAME = "stoichiometric_matrix"
COMPOSITION_SHEET_NAME = "composition_matrix"
PARAMETER_SHEET_NAME = "parameter_table"
PARAMETER_VALUE_COLUMN_INDEX = 5

_EXCEL_REFERENCE_PATTERN = re.compile(
    r"(?<![A-Za-z0-9_])(?:(?:'(?P<sheet_quoted>[^']+)')|(?P<sheet_unquoted>[A-Za-z_][A-Za-z0-9_]*))?!?\$?(?P<column>[A-Za-z]{1,3})\$?(?P<row>\d+)"
)

HEADER_FILL = PatternFill(fill_type="solid", fgColor="D8DEE6")
SECTION_FILL = PatternFill(fill_type="solid", fgColor="EEF2F5")
HEADER_FONT = Font(bold=True, color="22303C")

STOICHIOMETRIC_COEFFICIENTS: list[dict[str, dict[str, str]]] = [
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "-1/{Y_H}", "S_O": "1-1/{Y_H}", "X_H": "1"}},
    {"coefficients": {"S_A": "-1/{Y_H}", "S_O": "1-1/{Y_H}", "X_H": "1"}},
    {
        "coefficients": {
            "S_F": "-1/{Y_H}",
            "S_NO2": "(1-{Y_H})/((8/7)*{Y_H})",
            "S_NO3": "-((1-{Y_H})/((8/7)*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_F": "-1/{Y_H}",
            "S_N2": "(1-{Y_H})/(1.72*{Y_H})",
            "S_NO2": "-((1-{Y_H})/(1.72*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_A": "-1/{Y_H}",
            "S_NO2": "(1-{Y_H})/((8/7)*{Y_H})",
            "S_NO3": "-((1-{Y_H})/((8/7)*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_A": "-1/{Y_H}",
            "S_N2": "(1-{Y_H})/(1.72*{Y_H})",
            "S_NO2": "-((1-{Y_H})/(1.72*{Y_H}))",
            "X_H": "1",
        }
    },
    {"coefficients": {"S_A": "1", "S_F": "-1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_H": "-1"}},
    {"coefficients": {"S_A": "-1", "X_PP": "-{Y_PO4}", "X_PHA": "1"}},
    {"coefficients": {"S_O": "-{Y_PHA}", "X_PP": "1", "X_PHA": "-{Y_PHA}"}},
    {
        "coefficients": {
            "S_NO2": "{Y_PHA}/(8/7)",
            "S_NO3": "-({Y_PHA}/(8/7))",
            "X_PP": "1",
            "X_PHA": "-{Y_PHA}",
        }
    },
    {
        "coefficients": {
            "S_N2": "{Y_PHA}/1.72",
            "S_NO2": "-({Y_PHA}/1.72)",
            "X_PP": "1",
            "X_PHA": "-{Y_PHA}",
        }
    },
    {"coefficients": {"S_O": "1-1/{Y_PAO}", "X_PAO": "1", "X_PHA": "-1/{Y_PAO}"}},
    {
        "coefficients": {
            "S_NO2": "(1-{Y_PAO})/((8/7)*{Y_PAO})",
            "S_NO3": "-((1-{Y_PAO})/((8/7)*{Y_PAO}))",
            "X_PAO": "1",
            "X_PHA": "-1/{Y_PAO}",
        }
    },
    {
        "coefficients": {
            "S_N2": "(1-{Y_PAO})/(1.72*{Y_PAO})",
            "S_NO2": "-((1-{Y_PAO})/(1.72*{Y_PAO}))",
            "X_PAO": "1",
            "X_PHA": "-1/{Y_PAO}",
        }
    },
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_PAO": "-1"}},
    {"coefficients": {"X_PP": "-1"}},
    {"coefficients": {"S_A": "1", "X_PHA": "-1"}},
    {"coefficients": {"S_NO2": "1/{Y_AOB}", "S_O": "-((3.43-{Y_AOB})/{Y_AOB})", "X_AOB": "1"}},
    {"coefficients": {"S_NO2": "-1/{Y_NOB}", "S_NO3": "1/{Y_NOB}", "S_O": "-((1.14-{Y_NOB})/{Y_NOB})", "X_NOB": "1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_AOB": "-1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_NOB": "-1"}},
    {"coefficients": {"S_PO4": "-1", "X_MeOH": "-3.45", "X_MeP": "4.87"}},
    {"coefficients": {"S_PO4": "1", "X_MeOH": "3.45", "X_MeP": "-4.87"}},
]

COMPOSITION_FORMULAS: dict[str, dict[str, str]] = {
    "S_A": {"COD": "1"},
    "S_F": {"COD": "1", "TN": "{i_NSF}", "TKN": "{i_NSF}", "TP": "{i_PSF}"},
    "S_I": {"COD": "1", "TN": "{i_NSI}", "TKN": "{i_NSI}", "TP": "{i_PSI}"},
    "S_NH4": {"TN": "1", "TKN": "1"},
    "S_NO2": {"TN": "1"},
    "S_NO3": {"TN": "1"},
    "S_PO4": {"TP": "1"},
    "X_I": {"COD": "1", "TN": "{i_NXI}", "TKN": "{i_NXI}", "TP": "{i_PXI}", "TSS": "{i_TSS_XI}"},
    "X_S": {"COD": "1", "TN": "{i_NXS}", "TKN": "{i_NXS}", "TP": "{i_PXS}", "TSS": "{i_TSS_XS}"},
    "X_H": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_PAO": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_PP": {"TP": "1", "TSS": "{i_TSS_PP}"},
    "X_PHA": {"COD": "1", "TSS": "{i_TSS_PHA}"},
    "X_AOB": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_NOB": {"COD": "1", "TN": "{i_NBM}", "TKN": "{i_NBM}", "TP": "{i_PBM}", "TSS": "{i_TSS_BM}"},
    "X_MeOH": {"TSS": "1"},
    "X_MeP": {"TP": "{i_PMeP}", "TSS": "1"},
}

NITROGEN_CONTINUITY_TERMS = {
    "S_F": "{i_NSF}",
    "S_I": "{i_NSI}",
    "S_N2": "1",
    "S_NO2": "1",
    "S_NO3": "1",
    "X_I": "{i_NXI}",
    "X_S": "{i_NXS}",
    "X_H": "{i_NBM}",
    "X_PAO": "{i_NBM}",
    "X_AOB": "{i_NBM}",
    "X_NOB": "{i_NBM}",
}

PHOSPHORUS_CONTINUITY_TERMS = {
    "S_F": "{i_PSF}",
    "S_I": "{i_PSI}",
    "X_I": "{i_PXI}",
    "X_S": "{i_PXS}",
    "X_H": "{i_PBM}",
    "X_PAO": "{i_PBM}",
    "X_PP": "1",
    "X_AOB": "{i_PBM}",
    "X_NOB": "{i_PBM}",
    "X_MeP": "{i_PMeP}",
}

def load_asm2d_tsn_simulation_params(repo_root: str | Path | None = None) -> dict[str, Any]:
    """Load the configured ASM2D-TSN simulation definition."""

    return load_model_params(MODEL_NAME, repo_root)


def resolve_asm2d_tsn_workbook_path(
    repo_root: str | Path | None = None,
    *,
    paths_config: Mapping[str, Any] | None = None,
) -> Path:
    """Resolve the configured canonical workbook path."""

    root = get_repo_root(repo_root)
    config = dict(paths_config) if paths_config is not None else load_paths_config(root)
    return root / Path(config[WORKBOOK_PATH_KEY])


def resolve_asm2d_tsn_simulation_artifact_paths(
    repo_root: str | Path | None = None,
    *,
    timestamp: str | None = None,
    paths_config: Mapping[str, Any] | None = None,
) -> tuple[Path, Path, str]:
    """Resolve the configured ASM2D-TSN dataset and metadata output paths."""

    return render_simulation_artifact_paths(
        MODEL_NAME,
        repo_root=repo_root,
        timestamp=timestamp,
        paths_config=paths_config,
        data_pattern_key=DATA_PATTERN_KEY,
        metadata_pattern_key=METADATA_PATTERN_KEY,
    )


def resolve_asm2d_tsn_composition_cache_paths(
    workbook_hash: str,
    repo_root: str | Path | None = None,
    *,
    paths_config: Mapping[str, Any] | None = None,
) -> tuple[Path, Path]:
    """Resolve configured cache paths for one workbook-derived composition artifact."""

    root = get_repo_root(repo_root)
    config = dict(paths_config) if paths_config is not None else load_paths_config(root)
    matrix_path = root / Path(config[COMPOSITION_CACHE_PATTERN_KEY].format(workbook_hash=str(workbook_hash)))
    metadata_path = root / Path(
        config[COMPOSITION_CACHE_METADATA_PATTERN_KEY].format(workbook_hash=str(workbook_hash))
    )
    return matrix_path, metadata_path


def create_asm2d_tsn_workbook(
    workbook_path: str | Path | None = None,
    *,
    repo_root: str | Path | None = None,
    model_params: Mapping[str, Any] | None = None,
) -> Path:
    """Create the canonical ASM2D-TSN workbook with formula-driven matrices."""

    workbook = build_asm2d_tsn_workbook(model_params=model_params, repo_root=repo_root)
    output_path = Path(workbook_path) if workbook_path is not None else resolve_asm2d_tsn_workbook_path(repo_root)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    workbook.save(output_path)
    return output_path.resolve()


def build_asm2d_tsn_workbook(
    *,
    model_params: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
) -> Workbook:
    """Build the workbook object for the configured ASM2D-TSN reference model."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    workbook_config = _validate_workbook_config(params)
    parameter_refs = _build_parameter_reference_map(workbook_config["parameters"])

    workbook = Workbook()
    stoichiometric_sheet = workbook.active
    stoichiometric_sheet.title = STOICHIOMETRIC_SHEET_NAME
    composition_sheet = workbook.create_sheet(COMPOSITION_SHEET_NAME)
    parameter_sheet = workbook.create_sheet(PARAMETER_SHEET_NAME)

    _write_stoichiometric_sheet(stoichiometric_sheet, workbook_config, parameter_refs)
    _write_composition_sheet(composition_sheet, workbook_config, parameter_refs)
    _write_parameter_sheet(parameter_sheet, workbook_config["parameters"])

    for worksheet in workbook.worksheets:
        _auto_size_columns(worksheet)
        worksheet.auto_filter.ref = worksheet.dimensions

    return workbook


def _normalize_excel_reference(sheet_name: str, coordinate: str) -> str:
    return f"{str(sheet_name).strip().lower()}!{str(coordinate).replace('$', '').upper()}"


def _build_workbook_numeric_lookup(workbook) -> dict[str, float]:
    numeric_lookup: dict[str, float] = {}
    for worksheet in workbook.worksheets:
        for row in worksheet.iter_rows(
            min_row=1,
            max_row=worksheet.max_row,
            min_col=1,
            max_col=worksheet.max_column,
        ):
            for cell in row:
                cell_value = cell.value
                if isinstance(cell_value, bool):
                    continue
                if isinstance(cell_value, (int, float)):
                    numeric_lookup[_normalize_excel_reference(worksheet.title, cell.coordinate)] = float(cell_value)
    return numeric_lookup


def _evaluate_workbook_formula(
    formula: str,
    numeric_lookup: Mapping[str, float],
    *,
    current_sheet_name: str,
) -> float:
    expression = str(formula).strip()
    if expression.startswith("="):
        expression = expression[1:]

    def _replace_reference(match: re.Match[str]) -> str:
        sheet_name = match.group("sheet_quoted") or match.group("sheet_unquoted") or str(current_sheet_name)
        coordinate = f"{match.group('column').upper()}{match.group('row')}"
        lookup_key = _normalize_excel_reference(sheet_name, coordinate)
        if lookup_key not in numeric_lookup:
            raise KeyError(
                "Workbook formula references a non-numeric or missing cell: "
                f"{sheet_name}!{coordinate}."
            )
        return str(float(numeric_lookup[lookup_key]))

    resolved_expression = _EXCEL_REFERENCE_PATTERN.sub(_replace_reference, expression).replace("^", "**")
    return float(eval(resolved_expression, {"__builtins__": {}}, {}))


def _coerce_workbook_composition_value(
    cell_value: Any,
    numeric_lookup: Mapping[str, float],
    *,
    current_sheet_name: str,
) -> float:
    if cell_value is None:
        return 0.0
    if isinstance(cell_value, bool):
        return float(cell_value)
    if isinstance(cell_value, (int, float)):
        return float(cell_value)

    text_value = str(cell_value).strip()
    if not text_value:
        return 0.0
    if text_value.startswith("="):
        return _evaluate_workbook_formula(
            text_value,
            numeric_lookup,
            current_sheet_name=current_sheet_name,
        )

    try:
        return float(text_value)
    except ValueError as error:
        raise ValueError(f"Workbook composition_matrix contains a non-numeric value: {text_value!r}") from error


def _read_composition_matrix_from_workbook(
    workbook_path: str | Path,
    *,
    expected_state_columns: list[str],
) -> dict[str, Any]:
    workbook = load_workbook(filename=Path(workbook_path), data_only=False)
    try:
        if COMPOSITION_SHEET_NAME not in workbook.sheetnames:
            raise KeyError(f"Workbook is missing required sheet '{COMPOSITION_SHEET_NAME}'.")
        if PARAMETER_SHEET_NAME not in workbook.sheetnames:
            raise KeyError(f"Workbook is missing required sheet '{PARAMETER_SHEET_NAME}'.")

        worksheet = workbook[COMPOSITION_SHEET_NAME]
        header_values = [str(cell.value).strip() if cell.value is not None else "" for cell in worksheet[1]]
        if "state_variable" not in header_values:
            raise KeyError("Workbook composition_matrix must include a 'state_variable' header column.")

        state_column_number = header_values.index("state_variable") + 1
        reserved_headers = {"state_group", "state_variable", "unit"}
        composite_header_pairs = [
            (column_number, header_name)
            for column_number, header_name in enumerate(header_values, start=1)
            if header_name and header_name not in reserved_headers
        ]
        if not composite_header_pairs:
            raise ValueError("Workbook composition_matrix must define at least one composite output column.")

        measured_output_columns = [header_name for _, header_name in composite_header_pairs]
        _validate_unique_names(measured_output_columns, "composition_matrix output columns")

        numeric_lookup = _build_workbook_numeric_lookup(workbook)
        state_columns: list[str] = []
        coefficients_by_state: list[list[float]] = []
        for row_number in range(2, worksheet.max_row + 1):
            raw_state_name = worksheet.cell(row=row_number, column=state_column_number).value
            if raw_state_name is None:
                continue
            state_name = str(raw_state_name).strip()
            if not state_name:
                continue

            state_columns.append(state_name)
            row_coefficients: list[float] = []
            for column_number, _ in composite_header_pairs:
                raw_value = worksheet.cell(row=row_number, column=column_number).value
                row_coefficients.append(
                    _coerce_workbook_composition_value(
                        raw_value,
                        numeric_lookup,
                        current_sheet_name=worksheet.title,
                    )
                )
            coefficients_by_state.append(row_coefficients)

        if not state_columns:
            raise ValueError("Workbook composition_matrix must define at least one state_variable row.")
        _validate_unique_names(state_columns, "composition_matrix state_variable")

        if state_columns != expected_state_columns:
            raise ValueError(
                "Workbook composition_matrix state_variable rows must match configured workbook state_columns "
                "exactly and in order."
            )

        composition_matrix = np.asarray(coefficients_by_state, dtype=float).T
        expected_shape = (len(measured_output_columns), len(state_columns))
        if composition_matrix.shape != expected_shape:
            raise ValueError(
                "Workbook composition_matrix shape must match measured_output_columns x state_columns."
            )

        return {
            "state_columns": state_columns,
            "measured_output_columns": measured_output_columns,
            "composition_matrix": composition_matrix,
        }
    finally:
        workbook.close()


def _build_workbook_fingerprint(workbook_path: Path) -> dict[str, Any]:
    resolved_path = Path(workbook_path).resolve()
    stat_info = resolved_path.stat()
    return {
        "workbook_path": resolved_path.as_posix(),
        "workbook_sha256": compute_file_sha256(resolved_path),
        "workbook_mtime_ns": int(stat_info.st_mtime_ns),
        "workbook_size_bytes": int(stat_info.st_size),
    }


def _validate_cached_composition_payload(cache_payload: Mapping[str, Any]) -> None:
    required_keys = ("state_columns", "measured_output_columns", "composition_matrix")
    for required_key in required_keys:
        if required_key not in cache_payload:
            raise KeyError(f"Cached composition payload is missing required key '{required_key}'.")


def load_asm2d_tsn_workbook_composition(
    *,
    repo_root: str | Path | None = None,
    workbook_path: str | Path | None = None,
    model_params: Mapping[str, Any] | None = None,
    paths_config: Mapping[str, Any] | None = None,
    use_cache: bool = True,
) -> dict[str, Any]:
    """Load workbook-derived composition schema and coefficients, with fingerprinted cache reuse."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    workbook_config = _validate_workbook_config(params)
    expected_state_columns = list(workbook_config["state_columns"])
    resolved_workbook_path = (
        Path(workbook_path)
        if workbook_path is not None
        else resolve_asm2d_tsn_workbook_path(repo_root, paths_config=paths_config)
    )
    if not resolved_workbook_path.exists():
        raise FileNotFoundError(f"ASM2D-TSN workbook not found at {resolved_workbook_path}.")

    fingerprint = _build_workbook_fingerprint(resolved_workbook_path)
    cache_matrix_path, cache_metadata_path = resolve_asm2d_tsn_composition_cache_paths(
        fingerprint["workbook_sha256"],
        repo_root=repo_root,
        paths_config=paths_config,
    )

    if use_cache and cache_matrix_path.exists() and cache_metadata_path.exists():
        cache_metadata = load_json_file(cache_metadata_path)
        if (
            str(cache_metadata.get("workbook_sha256")) == str(fingerprint["workbook_sha256"])
            and int(cache_metadata.get("workbook_mtime_ns", -1)) == int(fingerprint["workbook_mtime_ns"])
            and int(cache_metadata.get("workbook_size_bytes", -1)) == int(fingerprint["workbook_size_bytes"])
        ):
            cache_payload = load_pickle_file(cache_matrix_path)
            _validate_cached_composition_payload(cache_payload)
            state_columns = [str(name) for name in cache_payload["state_columns"]]
            measured_output_columns = [str(name) for name in cache_payload["measured_output_columns"]]
            if state_columns != expected_state_columns:
                raise ValueError(
                    "Cached composition state_columns no longer match configured workbook state_columns."
                )
            composition_matrix = np.asarray(cache_payload["composition_matrix"], dtype=float)
            if composition_matrix.shape != (len(measured_output_columns), len(state_columns)):
                raise ValueError("Cached composition_matrix shape is invalid for cached schema.")
            return {
                "state_columns": state_columns,
                "measured_output_columns": measured_output_columns,
                "composition_matrix": composition_matrix,
                **fingerprint,
                "cache_source": "cache",
                "cache_paths": {
                    "composition_matrix": cache_matrix_path,
                    "composition_metadata": cache_metadata_path,
                },
            }

    parsed_composition = _read_composition_matrix_from_workbook(
        resolved_workbook_path,
        expected_state_columns=expected_state_columns,
    )
    composition_payload = {
        "state_columns": list(parsed_composition["state_columns"]),
        "measured_output_columns": list(parsed_composition["measured_output_columns"]),
        "composition_matrix": np.asarray(parsed_composition["composition_matrix"], dtype=float),
    }

    if use_cache:
        save_pickle_file(cache_matrix_path, composition_payload)
        save_json_file(
            cache_metadata_path,
            {
                "cache_schema_version": 1,
                "state_columns": composition_payload["state_columns"],
                "measured_output_columns": composition_payload["measured_output_columns"],
                **fingerprint,
            },
        )

    return {
        **composition_payload,
        **fingerprint,
        "cache_source": "workbook",
        "cache_paths": {
            "composition_matrix": cache_matrix_path,
            "composition_metadata": cache_metadata_path,
        },
    }


def get_asm2d_tsn_matrices(
    model_params: Mapping[str, Any] | None = None,
    *,
    repo_root: str | Path | None = None,
    paths_config: Mapping[str, Any] | None = None,
    use_composition_cache: bool = True,
) -> dict[str, Any]:
    """Build numeric Petersen and composition matrices for the configured ASM2D-TSN model."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    composition_bundle = load_asm2d_tsn_workbook_composition(
        repo_root=repo_root,
        model_params=params,
        paths_config=paths_config,
        use_cache=use_composition_cache,
    )
    measured_output_columns = list(composition_bundle["measured_output_columns"])
    runtime = _validate_runtime_structure(params, measured_output_columns=measured_output_columns)
    workbook_config = runtime["workbook_config"]
    parameter_values = _build_parameter_value_map(workbook_config["parameters"])
    state_columns = list(runtime["state_columns"])
    process_names = list(runtime["process_names"])
    process_types = list(runtime["process_types"])
    state_index = _build_state_index(state_columns)

    composition_state_columns = list(composition_bundle["state_columns"])
    if composition_state_columns != state_columns:
        raise ValueError(
            "Workbook composition_matrix state columns do not match the configured ASM2D-TSN state columns."
        )

    petersen_matrix = np.zeros((len(process_names), len(state_columns)), dtype=float)
    composition_matrix = np.asarray(composition_bundle["composition_matrix"], dtype=float)
    if composition_matrix.shape != (len(measured_output_columns), len(state_columns)):
        raise ValueError(
            "Workbook composition_matrix shape must match measured_output_columns x state_columns."
        )

    for row_index, process_definition in enumerate(STOICHIOMETRIC_COEFFICIENTS):
        row_values = petersen_matrix[row_index]
        direct_coefficients = process_definition["coefficients"]

        for state_name, expression in direct_coefficients.items():
            row_values[state_index[state_name]] = _evaluate_numeric_expression(expression, parameter_values)

        row_values[state_index["S_NH4"]] = -sum(
            row_values[state_index[state_name]] * _evaluate_numeric_expression(factor_expression, parameter_values)
            for state_name, factor_expression in NITROGEN_CONTINUITY_TERMS.items()
        )

        if "S_PO4" not in direct_coefficients:
            row_values[state_index["S_PO4"]] = -sum(
                row_values[state_index[state_name]] * _evaluate_numeric_expression(factor_expression, parameter_values)
                for state_name, factor_expression in PHOSPHORUS_CONTINUITY_TERMS.items()
            )

        row_values[state_index["S_ALK"]] = (
            row_values[state_index["S_NH4"]] / 14.0
            - row_values[state_index["S_NO2"]] / 14.0
            - row_values[state_index["S_NO3"]] / 14.0
            + row_values[state_index["S_PO4"]] / 31.0
        )

    return {
        "petersen_matrix": petersen_matrix,
        "composition_matrix": composition_matrix,
        "process_names": process_names,
        "process_types": process_types,
        "state_index": state_index,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "composition_workbook_path": str(composition_bundle["workbook_path"]),
        "composition_workbook_sha256": str(composition_bundle["workbook_sha256"]),
        "composition_workbook_mtime_ns": int(composition_bundle["workbook_mtime_ns"]),
        "composition_workbook_size_bytes": int(composition_bundle["workbook_size_bytes"]),
        "composition_cache_source": str(composition_bundle["cache_source"]),
        "composition_cache_paths": dict(composition_bundle["cache_paths"]),
    }


def build_asm2d_tsn_metadata(
    model_params: Mapping[str, Any],
    *,
    sample_count: int,
    random_seed: int,
    dataset_file: str | None = None,
    measured_output_columns: list[str] | None = None,
    composition_source: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
) -> dict[str, Any]:
    """Create the metadata contract for the ASM2D-TSN mixed-schema dataset."""

    if measured_output_columns is None:
        measured_output_columns = list(
            load_asm2d_tsn_workbook_composition(
                repo_root=repo_root,
                model_params=model_params,
            )["measured_output_columns"]
        )

    runtime = _validate_runtime_structure(model_params, measured_output_columns=measured_output_columns)
    state_columns = list(runtime["state_columns"])
    measured_output_columns = list(runtime["measured_output_columns"])
    process_names = list(runtime["process_names"])
    process_types = list(runtime["process_types"])
    operational_columns = list(runtime["operational_columns"])
    influent_fraction_columns = [f"In_{name}" for name in state_columns]
    influent_composite_columns = [f"In_{name}" for name in measured_output_columns]
    effluent_fraction_columns = [f"Out_{name}" for name in state_columns]
    dependent_columns = [f"Out_{name}" for name in measured_output_columns]

    return {
        "simulation_name": MODEL_NAME,
        "n_samples": sample_count,
        "random_seed": random_seed,
        "sampling_method": "latin_hypercube",
        "dependent_columns": dependent_columns,
        "independent_columns": operational_columns + influent_fraction_columns,
        "identifier_columns": [],
        "ignored_columns": influent_composite_columns + effluent_fraction_columns,
        "dataset_file": dataset_file,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "operational_columns": operational_columns,
        "influent_fraction_columns": influent_fraction_columns,
        "influent_composite_columns": influent_composite_columns,
        "effluent_fraction_columns": effluent_fraction_columns,
        "effluent_composite_columns": dependent_columns,
        "processes": process_names,
        "process_types": process_types,
        "petersen_matrix_shape": [len(process_names), len(state_columns)],
        "composition_matrix_shape": [len(measured_output_columns), len(state_columns)],
        "schema_version": str(model_params["schema_version"]),
        "composition_source": dict(composition_source or {}),
    }


def generate_asm2d_tsn_dataset(
    *,
    model_params: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
    n_samples: int | None = None,
    random_seed: int | None = None,
    parallel_workers: int | None = None,
    parallel_chunk_size: int | None = None,
    include_debug_data: bool = False,
    show_progress: bool = False,
    progress_description: str | None = None,
) -> tuple[pd.DataFrame, dict[str, Any], dict[str, Any]]:
    """Generate a mechanistic steady-state ASM2D-TSN dataset with input/output fractions and composites."""

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    matrix_bundle = get_asm2d_tsn_matrices(params, repo_root=repo_root)
    runtime = _validate_runtime_structure(
        params,
        measured_output_columns=list(matrix_bundle["measured_output_columns"]),
    )
    configured_hyperparameters = params["hyperparameters"]
    sample_count = int(n_samples if n_samples is not None else configured_hyperparameters["n_samples"])
    if sample_count < 0:
        raise ValueError("n_samples must be greater than or equal to 0.")

    seed = int(random_seed if random_seed is not None else configured_hyperparameters["seed"])
    requested_parallel_workers = int(
        parallel_workers if parallel_workers is not None else configured_hyperparameters.get("parallel_workers", 1)
    )
    requested_parallel_chunk_size = int(
        parallel_chunk_size
        if parallel_chunk_size is not None
        else configured_hyperparameters.get("parallel_chunk_size", sample_count or 1)
    )
    state_columns = list(runtime["state_columns"])
    operational_columns = list(runtime["operational_columns"])
    measured_output_columns = list(runtime["measured_output_columns"])

    influent_states = np.zeros((sample_count, len(state_columns)), dtype=float)
    operational = np.zeros((sample_count, len(operational_columns)), dtype=float)
    measured_outputs = np.zeros((sample_count, len(measured_output_columns)), dtype=float)
    effluent_states = np.zeros((sample_count, len(state_columns)), dtype=float)
    solver_diagnostic_records: list[dict[str, Any]] = []
    chunk_results: list[tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[dict[str, Any]]]] = []
    if sample_count > 0:
        worker_count = _resolve_parallel_workers(requested_parallel_workers, sample_count)
        chunk_size = min(_resolve_parallel_chunk_size(requested_parallel_chunk_size), sample_count)
        chunk_specs = [
            {
                "chunk_start": chunk_start,
                "chunk_size": min(chunk_size, sample_count - chunk_start),
                "chunk_seed": seed + chunk_index,
                "model_params": params,
                "matrix_bundle": matrix_bundle,
                "runtime": runtime,
                "collect_debug_data": include_debug_data,
            }
            for chunk_index, chunk_start in enumerate(range(0, sample_count, chunk_size))
        ]

        if worker_count == 1 or len(chunk_specs) == 1:
            progress_bar = tqdm(
                total=sample_count,
                desc=progress_description or "ASM2D-TSN simulation",
                unit="sample",
                disable=not show_progress,
            )
            try:
                chunk_results = [
                    _generate_asm2d_tsn_dataset_chunk(progress_bar=progress_bar, **chunk_spec)
                    for chunk_spec in chunk_specs
                ]
            finally:
                progress_bar.close()
        else:
            with ProcessPoolExecutor(max_workers=worker_count) as executor:
                future_map = {
                    executor.submit(_generate_asm2d_tsn_dataset_chunk, **chunk_spec): int(chunk_spec["chunk_size"])
                    for chunk_spec in chunk_specs
                }
                progress_bar = tqdm(
                    total=sample_count,
                    desc=progress_description or "ASM2D-TSN simulation",
                    unit="sample",
                    disable=not show_progress,
                )
                try:
                    for future in as_completed(future_map):
                        chunk_results.append(future.result())
                        progress_bar.update(future_map[future])
                finally:
                    progress_bar.close()

    for chunk_start, chunk_influent, chunk_operational, chunk_effluent, chunk_measured, chunk_diagnostics in chunk_results:
        chunk_end = chunk_start + len(chunk_influent)
        influent_states[chunk_start:chunk_end] = chunk_influent
        operational[chunk_start:chunk_end] = chunk_operational
        effluent_states[chunk_start:chunk_end] = chunk_effluent
        measured_outputs[chunk_start:chunk_end] = chunk_measured
        if include_debug_data:
            solver_diagnostic_records.extend(chunk_diagnostics)

    influent_df = pd.DataFrame(influent_states, columns=[f"In_{name}" for name in state_columns])
    influent_composite_df = pd.DataFrame(
        influent_states @ np.asarray(matrix_bundle["composition_matrix"], dtype=float).T,
        columns=[f"In_{name}" for name in measured_output_columns],
    )
    operational_df = pd.DataFrame(operational, columns=operational_columns)
    effluent_df = pd.DataFrame(effluent_states, columns=[f"Out_{name}" for name in state_columns])
    measured_df = pd.DataFrame(measured_outputs, columns=[f"Out_{name}" for name in measured_output_columns])
    dataset = pd.concat([operational_df, influent_df, influent_composite_df, effluent_df, measured_df], axis=1)

    metadata = build_asm2d_tsn_metadata(
        params,
        sample_count=sample_count,
        random_seed=seed,
        measured_output_columns=measured_output_columns,
        composition_source={
            "workbook_path": matrix_bundle["composition_workbook_path"],
            "workbook_sha256": matrix_bundle["composition_workbook_sha256"],
            "workbook_mtime_ns": matrix_bundle["composition_workbook_mtime_ns"],
            "workbook_size_bytes": matrix_bundle["composition_workbook_size_bytes"],
            "cache_source": matrix_bundle["composition_cache_source"],
        },
        repo_root=repo_root,
    )

    simulation_bundle = dict(matrix_bundle)
    if include_debug_data:
        solver_diagnostics = pd.DataFrame(solver_diagnostic_records).sort_values("sample_index").reset_index(drop=True)
        simulation_bundle["effluent_states"] = pd.DataFrame(effluent_states, columns=state_columns)
        simulation_bundle["solver_diagnostics"] = solver_diagnostics
        simulation_bundle["solver_summary"] = _summarize_asm2d_tsn_solver_diagnostics(
            solver_diagnostics,
            float(runtime["solver"]["acceptance_residual_max"]),
        )

    return dataset, metadata, simulation_bundle


def run_asm2d_tsn_simulation(
    *,
    save_artifacts: bool = True,
    repo_root: str | Path | None = None,
    n_samples: int | None = None,
    random_seed: int | None = None,
    parallel_workers: int | None = None,
    parallel_chunk_size: int | None = None,
    timestamp: str | None = None,
    include_debug_data: bool = False,
    show_progress: bool = False,
    progress_description: str | None = None,
) -> dict[str, Any]:
    """Run the ASM2D-TSN steady-state simulation and optionally persist artifacts."""

    params = load_asm2d_tsn_simulation_params(repo_root)
    simulation_bundle: dict[str, Any]
    dataset, metadata, simulation_bundle = generate_asm2d_tsn_dataset(
        model_params=params,
        repo_root=repo_root,
        n_samples=n_samples,
        random_seed=random_seed,
        parallel_workers=parallel_workers,
        parallel_chunk_size=parallel_chunk_size,
        include_debug_data=include_debug_data,
        show_progress=show_progress,
        progress_description=progress_description,
    )

    artifact_paths: dict[str, Path | None] = {
        "dataset_csv": None,
        "metadata_json": None,
    }

    if save_artifacts:
        dataset_path, metadata_path, persisted_metadata = save_simulation_artifacts(
            dataset,
            metadata,
            MODEL_NAME,
            repo_root=repo_root,
            timestamp=timestamp,
            data_pattern_key=DATA_PATTERN_KEY,
            metadata_pattern_key=METADATA_PATTERN_KEY,
        )
        metadata = persisted_metadata
        artifact_paths = {
            "dataset_csv": dataset_path,
            "metadata_json": metadata_path,
        }

    return {
        "dataset": dataset,
        "metadata": metadata,
        "petersen_matrix": simulation_bundle["petersen_matrix"],
        "composition_matrix": simulation_bundle["composition_matrix"],
        "matrix_bundle": simulation_bundle,
        "composite_matrix": simulation_bundle["composition_matrix"],
        "artifact_paths": artifact_paths,
        "effluent_states": simulation_bundle.get("effluent_states"),
        "solver_diagnostics": simulation_bundle.get("solver_diagnostics"),
        "solver_summary": simulation_bundle.get("solver_summary"),
    }


def sweep_asm2d_tsn_operating_space(
    *,
    model_params: Mapping[str, Any] | None = None,
    repo_root: str | Path | None = None,
    n_samples: int = 512,
    random_seed: int | None = None,
    show_progress: bool = False,
    progress_description: str | None = None,
) -> dict[str, Any]:
    """Sample the configured operating space and summarize ASM2D-TSN solver behavior."""

    if n_samples < 1:
        raise ValueError("n_samples must be at least 1.")

    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    matrix_bundle = get_asm2d_tsn_matrices(params, repo_root=repo_root)
    runtime = _validate_runtime_structure(
        params,
        measured_output_columns=list(matrix_bundle["measured_output_columns"]),
    )
    configured_hyperparameters = params["hyperparameters"]
    seed = int(random_seed if random_seed is not None else configured_hyperparameters["seed"])
    state_columns = list(runtime["state_columns"])
    operational_columns = list(runtime["operational_columns"])
    state_index = dict(matrix_bundle["state_index"])

    # Pre-generate a joint LHS design for all sweep points.
    all_columns = state_columns + operational_columns
    all_ranges = {**runtime["influent_state_ranges"], **runtime["operational_ranges"]}
    candidate_pool = _generate_lhs_candidate_pool(seed, n_samples, all_columns, all_ranges)
    n_state = len(state_columns)

    influent_samples = np.zeros((n_samples, len(state_columns)), dtype=float)
    operational_samples = np.zeros((n_samples, len(operational_columns)), dtype=float)
    effluent_samples = np.zeros((n_samples, len(state_columns)), dtype=float)
    diagnostic_records: list[dict[str, Any]] = []

    progress_bar = tqdm(
        total=n_samples,
        desc=progress_description or "ASM2D-TSN sweep",
        unit="sample",
        disable=not show_progress,
    )
    try:
        for sample_index in range(n_samples):
            sampled_state = candidate_pool[sample_index, :n_state]
            influent_state = _build_influent_state_sample(sampled_state)
            operating_point = candidate_pool[sample_index, n_state:]
            effluent_state, diagnostics = simulate_asm2d_tsn_steady_state(
                influent_state=influent_state,
                hrt_hours=float(operating_point[0]),
                aeration=float(operating_point[1]),
                model_params=params,
                matrix_bundle=matrix_bundle,
                previous_solution=None,
                enforce_acceptance=False,
            )
            influent_samples[sample_index] = influent_state
            operational_samples[sample_index] = operating_point
            effluent_samples[sample_index] = effluent_state
            diagnostic_record = dict(diagnostics)
            diagnostic_record["sample_index"] = sample_index
            diagnostic_record["HRT"] = float(operating_point[0])
            diagnostic_record["Aeration"] = float(operating_point[1])
            diagnostic_records.append(diagnostic_record)
            progress_bar.update(1)
    finally:
        progress_bar.close()

    solver_diagnostics = pd.DataFrame(diagnostic_records).sort_values("sample_index").reset_index(drop=True)
    summary = _summarize_asm2d_tsn_solver_diagnostics(
        solver_diagnostics,
        float(runtime["solver"]["acceptance_residual_max"]),
    )

    return {
        "influent_states": pd.DataFrame(influent_samples, columns=state_columns),
        "operating_conditions": pd.DataFrame(operational_samples, columns=operational_columns),
        "effluent_states": pd.DataFrame(effluent_samples, columns=state_columns),
        "solver_diagnostics": solver_diagnostics,
        "summary": summary,
        "matrix_bundle": matrix_bundle,
    }


def _validate_workbook_config(model_params: Mapping[str, Any]) -> dict[str, Any]:
    if "workbook" not in model_params:
        raise KeyError("asm2d_tsn_simulation must define a workbook section.")

    workbook_config = dict(model_params["workbook"])
    expected_sheets = [STOICHIOMETRIC_SHEET_NAME, COMPOSITION_SHEET_NAME, PARAMETER_SHEET_NAME]
    configured_sheets = list(workbook_config["sheets"])
    if configured_sheets != expected_sheets:
        raise ValueError("asm2d_tsn_simulation workbook sheets must match the required three-sheet contract.")

    dissolved_state_columns = list(workbook_config["dissolved_state_columns"])
    particulate_state_columns = list(workbook_config["particulate_state_columns"])
    state_columns = list(workbook_config["state_columns"])
    legacy_composite_variables = workbook_config.get("composite_variables")
    processes = list(workbook_config["processes"])
    parameter_rows = list(workbook_config["parameters"])
    state_units = dict(workbook_config["state_units"])

    _validate_unique_names(dissolved_state_columns, "dissolved_state_columns")
    _validate_unique_names(particulate_state_columns, "particulate_state_columns")
    _validate_unique_names(state_columns, "state_columns")
    _resolve_workbook_composite_variables(
        state_columns,
        legacy_composite_variables=(
            None if legacy_composite_variables is None else list(legacy_composite_variables)
        ),
    )

    if state_columns != dissolved_state_columns + particulate_state_columns:
        raise ValueError("asm2d_tsn_simulation state_columns must concatenate dissolved and particulate state columns.")

    missing_state_units = [state_name for state_name in state_columns if state_name not in state_units]
    if missing_state_units:
        missing_display = ", ".join(missing_state_units)
        raise KeyError(f"asm2d_tsn_simulation missing state units for: {missing_display}")

    if len(processes) != len(STOICHIOMETRIC_COEFFICIENTS):
        raise ValueError("asm2d_tsn_simulation workbook process count does not match the stoichiometric matrix definition.")

    process_indices = [int(process["index"]) for process in processes]
    if process_indices != list(range(1, len(processes) + 1)):
        raise ValueError("asm2d_tsn_simulation workbook processes must be sequentially indexed from 1.")

    parameter_names = [str(parameter_row["excel_name"]) for parameter_row in parameter_rows]
    _validate_unique_names(parameter_names, "parameter excel_name")

    for parameter_row in parameter_rows:
        for required_key in ("category", "symbol", "excel_name", "description", "value", "unit"):
            if required_key not in parameter_row:
                raise KeyError(f"asm2d_tsn_simulation parameter row missing '{required_key}'.")
        float(parameter_row["value"])

    return workbook_config


def _validate_runtime_structure(
    model_params: Mapping[str, Any],
    *,
    measured_output_columns: list[str] | None,
) -> dict[str, Any]:
    workbook_config = _validate_workbook_config(model_params)
    solver = _validate_solver_config(model_params)
    state_columns = list(workbook_config["state_columns"])
    if measured_output_columns is None:
        raise ValueError(
            "asm2d_tsn_simulation requires measured_output_columns derived from workbook composition_matrix."
        )
    measured_output_columns = [str(name) for name in measured_output_columns]
    process_names = [str(process["name"]) for process in workbook_config["processes"]]
    process_types = list(model_params["process_types"])
    operational_columns = list(model_params["operational_columns"])
    influent_state_ranges = dict(model_params["influent_state_ranges"])
    operational_ranges = dict(model_params["operational_ranges"])

    _validate_unique_names(measured_output_columns, "measured_output_columns")
    _validate_unique_names(process_names, "process names")
    _validate_unique_names(operational_columns, "operational_columns")

    if len(process_types) != len(process_names):
        raise ValueError("asm2d_tsn_simulation process_types must align with the configured process list.")

    missing_state_ranges = [state_name for state_name in state_columns if state_name not in influent_state_ranges]
    if missing_state_ranges:
        missing_display = ", ".join(missing_state_ranges)
        raise KeyError(f"asm2d_tsn_simulation missing influent_state_ranges for: {missing_display}")

    missing_operational_ranges = [name for name in operational_columns if name not in operational_ranges]
    if missing_operational_ranges:
        missing_display = ", ".join(missing_operational_ranges)
        raise KeyError(f"asm2d_tsn_simulation missing operational_ranges for: {missing_display}")

    return {
        "workbook_config": workbook_config,
        "solver": solver,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "process_names": process_names,
        "process_types": process_types,
        "operational_columns": operational_columns,
        "influent_state_ranges": influent_state_ranges,
        "operational_ranges": operational_ranges,
    }


def _validate_solver_config(model_params: Mapping[str, Any]) -> dict[str, Any]:
    if "solver" not in model_params:
        raise KeyError("asm2d_tsn_simulation must define a solver section.")

    solver = dict(model_params["solver"])
    required_keys = (
        "lower_bound",
        "upper_bound",
        "initial_guess_floor",
        "warm_start_previous_weight",
        "warm_start_influent_weight",
        "initial_s_a_fraction",
        "initial_s_f_fraction",
        "initial_heterotroph_to_xs_ratio",
        "initial_pao_to_pp_ratio",
        "initial_aob_to_nh4_ratio",
        "initial_nob_to_no2_ratio",
        "multistart_s_a_fraction",
        "multistart_s_f_fraction",
        "multistart_heterotroph_to_xs_ratio",
        "multistart_pao_to_pp_ratio",
        "multistart_aob_to_nh4_ratio",
        "multistart_nob_to_no2_ratio",
        "dynamic_relaxation_days",
        "dynamic_absolute_tolerance",
        "dynamic_relative_tolerance",
        "dynamic_max_step",
        "residual_tolerance",
        "variable_tolerance",
        "gradient_tolerance",
        "acceptance_residual_max",
        "max_nfev",
    )

    for key in required_keys:
        if key not in solver:
            raise KeyError(f"asm2d_tsn_simulation solver missing '{key}'.")
        float(solver[key])

    lower_bound = float(solver["lower_bound"])
    upper_bound = float(solver["upper_bound"])
    initial_guess_floor = float(solver["initial_guess_floor"])
    if lower_bound < 0.0:
        raise ValueError("asm2d_tsn_simulation solver lower_bound must be non-negative.")
    if upper_bound <= lower_bound:
        raise ValueError("asm2d_tsn_simulation solver upper_bound must exceed lower_bound.")
    if not (lower_bound <= initial_guess_floor <= upper_bound):
        raise ValueError(
            "asm2d_tsn_simulation solver initial_guess_floor must lie between lower_bound and upper_bound."
        )

    previous_weight = float(solver["warm_start_previous_weight"])
    influent_weight = float(solver["warm_start_influent_weight"])
    if previous_weight < 0.0 or influent_weight < 0.0:
        raise ValueError("asm2d_tsn_simulation warm-start weights must be non-negative.")
    if previous_weight + influent_weight <= 0.0:
        raise ValueError("asm2d_tsn_simulation warm-start weights must sum to a positive value.")

    return solver


def _validate_unique_names(names: list[str], name_type: str) -> None:
    if not names:
        raise ValueError(f"{name_type} must not be empty.")

    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        duplicate_display = ", ".join(duplicates)
        raise ValueError(f"asm2d_tsn_simulation {name_type} contains duplicates: {duplicate_display}")


def _build_parameter_reference_map(parameter_rows: list[Mapping[str, Any]]) -> dict[str, str]:
    value_column_letter = get_column_letter(PARAMETER_VALUE_COLUMN_INDEX)
    parameter_refs: dict[str, str] = {}

    for row_number, parameter_row in enumerate(parameter_rows, start=2):
        excel_name = str(parameter_row["excel_name"])
        parameter_refs[excel_name] = f"'{PARAMETER_SHEET_NAME}'!${value_column_letter}${row_number}"

    return parameter_refs


def _build_parameter_value_map(parameter_rows: list[Mapping[str, Any]]) -> dict[str, float]:
    return {str(parameter_row["excel_name"]): float(parameter_row["value"]) for parameter_row in parameter_rows}


def _evaluate_numeric_expression(expression: str | float | int, parameter_values: Mapping[str, float]) -> float:
    if isinstance(expression, (int, float)):
        return float(expression)

    formatted_expression = str(expression).format_map(parameter_values)
    return float(eval(formatted_expression, {"__builtins__": {}}, {}))


def _build_state_index(state_columns: list[str]) -> dict[str, int]:
    return {name: position for position, name in enumerate(state_columns)}


def _generate_lhs_candidate_pool(
    seed: int,
    n_points: int,
    ordered_names: list[str],
    ranges: Mapping[str, Any],
) -> np.ndarray:
    """Generate n_points Latin Hypercube samples scaled to the configured parameter ranges.

    For degenerate ranges where lower == upper the returned column is the constant
    lower-bound value.  An empty array is returned when n_points or the number of
    dimensions is zero.
    """
    from scipy.stats.qmc import LatinHypercube, scale as qmc_scale

    n_dims = len(ordered_names)
    if n_points == 0 or n_dims == 0:
        return np.zeros((n_points, n_dims), dtype=float)

    lower_bounds = np.array([float(ranges[name][0]) for name in ordered_names], dtype=float)
    upper_bounds = np.array([float(ranges[name][1]) for name in ordered_names], dtype=float)

    degenerate_mask = lower_bounds == upper_bounds
    active_indices = np.where(~degenerate_mask)[0]

    result = np.empty((n_points, n_dims), dtype=float)
    # Fill constant values for degenerate dimensions
    for idx in np.where(degenerate_mask)[0]:
        result[:, idx] = lower_bounds[idx]

    if len(active_indices) == 0:
        return result

    sampler = LatinHypercube(d=len(active_indices), seed=seed)
    unit_samples = sampler.random(n=n_points)
    scaled = qmc_scale(unit_samples, lower_bounds[active_indices], upper_bounds[active_indices])
    result[:, active_indices] = scaled
    return result


def _monod(numerator: float, half_saturation: float) -> float:
    return float(numerator) / max(float(numerator) + float(half_saturation), 1e-9)


def _ratio(numerator: float, denominator: float) -> float:
    return float(numerator) / max(float(denominator), 1e-9)


def _share(numerator: float, denominator_a: float, denominator_b: float) -> float:
    return float(numerator) / max(float(denominator_a) + float(denominator_b), 1e-9)


def _build_influent_state_sample(
    sampled_state: np.ndarray,
) -> np.ndarray:
    return np.maximum(sampled_state.copy(), 0.0)


def _build_initial_guess(
    influent_state: np.ndarray,
    hrt_hours: float,
    aeration: float,
    state_index: Mapping[str, int],
    model_params: Mapping[str, Any],
    previous_solution: np.ndarray | None = None,
) -> np.ndarray:
    solver = model_params["solver"]
    aeration_model = model_params["aeration_model"]
    lower_floor = float(solver["initial_guess_floor"])
    upper_bound = float(solver["upper_bound"])
    guess = np.clip(influent_state.copy(), lower_floor, upper_bound)

    if previous_solution is not None:
        previous_weight = float(solver["warm_start_previous_weight"])
        influent_weight = float(solver["warm_start_influent_weight"])
        weight_total = max(previous_weight + influent_weight, 1e-9)
        guess = np.clip(
            (previous_weight * previous_solution + influent_weight * guess) / weight_total,
            lower_floor,
            upper_bound,
        )

    guess[state_index["S_A"]] *= float(solver["initial_s_a_fraction"])
    guess[state_index["S_F"]] *= float(solver["initial_s_f_fraction"])
    guess[state_index["X_H"]] = max(
        guess[state_index["X_H"]],
        guess[state_index["X_S"]] * float(solver["initial_heterotroph_to_xs_ratio"]),
    )
    guess[state_index["X_PAO"]] = max(
        guess[state_index["X_PAO"]],
        guess[state_index["X_PP"]] * float(solver["initial_pao_to_pp_ratio"]),
    )
    guess[state_index["X_AOB"]] = max(
        guess[state_index["X_AOB"]],
        guess[state_index["S_NH4"]] * float(solver["initial_aob_to_nh4_ratio"]),
    )
    guess[state_index["X_NOB"]] = max(
        guess[state_index["X_NOB"]],
        guess[state_index["S_NO2"]] * float(solver["initial_nob_to_no2_ratio"]),
    )

    dilution_rate = 24.0 / max(float(hrt_hours), 1e-6)
    kla = float(aeration_model["kla_base"]) + float(aeration_model["kla_per_aeration"]) * max(float(aeration), 0.0)
    do_saturation = float(aeration_model["do_saturation"])
    guess[state_index["S_O"]] = np.clip(
        (dilution_rate * guess[state_index["S_O"]] + kla * do_saturation) / max(dilution_rate + kla, 1e-9),
        lower_floor,
        do_saturation,
    )

    return guess


def _build_multistart_guess(
    initial_guess: np.ndarray,
    influent_state: np.ndarray,
    state_index: Mapping[str, int],
    model_params: Mapping[str, Any],
) -> np.ndarray:
    solver = model_params["solver"]
    multistart_guess = initial_guess.copy()
    multistart_guess[state_index["S_A"]] *= float(solver["multistart_s_a_fraction"])
    multistart_guess[state_index["S_F"]] *= float(solver["multistart_s_f_fraction"])
    multistart_guess[state_index["X_H"]] = max(
        multistart_guess[state_index["X_H"]],
        influent_state[state_index["X_S"]] * float(solver["multistart_heterotroph_to_xs_ratio"]),
    )
    multistart_guess[state_index["X_PAO"]] = max(
        multistart_guess[state_index["X_PAO"]],
        influent_state[state_index["X_PP"]] * float(solver["multistart_pao_to_pp_ratio"]),
    )
    multistart_guess[state_index["X_AOB"]] = max(
        multistart_guess[state_index["X_AOB"]],
        influent_state[state_index["S_NH4"]] * float(solver["multistart_aob_to_nh4_ratio"]),
    )
    multistart_guess[state_index["X_NOB"]] = max(
        multistart_guess[state_index["X_NOB"]],
        influent_state[state_index["S_NO2"]] * float(solver["multistart_nob_to_no2_ratio"]),
    )
    return np.clip(
        multistart_guess,
        float(solver["lower_bound"]),
        float(solver["upper_bound"]),
    )


def _compute_process_rates(
    state: np.ndarray,
    model_params: Mapping[str, Any],
    state_index: Mapping[str, int],
    parameter_values: Mapping[str, float],
) -> np.ndarray:
    s_a = state[state_index["S_A"]]
    s_f = state[state_index["S_F"]]
    s_nh4 = state[state_index["S_NH4"]]
    s_no2 = state[state_index["S_NO2"]]
    s_no3 = state[state_index["S_NO3"]]
    s_po4 = state[state_index["S_PO4"]]
    s_nox = s_no2 + s_no3
    s_alk = state[state_index["S_ALK"]]
    s_o = state[state_index["S_O"]]
    x_s = state[state_index["X_S"]]
    x_h = state[state_index["X_H"]]
    x_pao = state[state_index["X_PAO"]]
    x_pp = state[state_index["X_PP"]]
    x_pha = state[state_index["X_PHA"]]
    x_aob = state[state_index["X_AOB"]]
    x_nob = state[state_index["X_NOB"]]
    x_meoh = state[state_index["X_MeOH"]]
    x_mep = state[state_index["X_MeP"]]

    xs_to_xh = _ratio(x_s, x_h)
    hydrolysis_availability = _monod(xs_to_xh, parameter_values["K_X"])
    nitrate_share = _share(s_no3, s_no3, s_no2)
    nitrite_share = _share(s_no2, s_no3, s_no2)

    oxygen_hyd = _monod(s_o, parameter_values["K_O_hyd"])
    oxygen_hyd_limitation = parameter_values["K_O_hyd"] / max(parameter_values["K_O_hyd"] + s_o, 1e-9)
    oxygen_h = _monod(s_o, parameter_values["K_O_H"])
    oxygen_h_limitation = parameter_values["K_O_H"] / max(parameter_values["K_O_H"] + s_o, 1e-9)
    oxygen_pao = _monod(s_o, parameter_values["K_O_PAO"])
    oxygen_pao_limitation = parameter_values["K_O_PAO"] / max(parameter_values["K_O_PAO"] + s_o, 1e-9)

    alk_h = _monod(s_alk, parameter_values["K_ALK_H"])
    alk_pao = _monod(s_alk, parameter_values["K_ALK_PAO"])
    alk_nit = _monod(s_alk, parameter_values["K_ALK_nit"])
    alk_chem = _monod(s_alk, parameter_values["K_ALK_chem"])

    ammonium_h = _monod(s_nh4, parameter_values["K_NH4_H"])
    phosphate_h = _monod(s_po4, parameter_values["K_PO4_H"])
    ammonium_pao = _monod(s_nh4, parameter_values["K_NH4_PAO"])
    phosphate_pao = _monod(s_po4, parameter_values["K_PO4_PAO"])
    phosphate_storage = _monod(s_po4, parameter_values["K_PS"])
    phosphate_nit = _monod(s_po4, parameter_values["K_PO4_nit"])

    substrate_f = _monod(s_f, parameter_values["K_F"])
    substrate_fe = _monod(s_f, parameter_values["K_fe"])
    substrate_a = _monod(s_a, parameter_values["K_A"])

    pao_ratio_pp = _ratio(x_pp, x_pao)
    pao_ratio_pha = _ratio(x_pha, x_pao)
    pp_capacity = max(parameter_values["K_MAX"] - pao_ratio_pp, 0.0)
    r_pp = _monod(pao_ratio_pp, parameter_values["K_PP"])
    r_pha = _monod(pao_ratio_pha, parameter_values["K_PHA"])
    c_pp = pp_capacity / max(parameter_values["K_IPP"] + pp_capacity, 1e-9)

    process_rates = np.array(
        [
            parameter_values["K_H"] * oxygen_hyd * hydrolysis_availability * x_h,
            parameter_values["eta_hyd_NO2"]
            * parameter_values["K_H"]
            * oxygen_hyd_limitation
            * _monod(s_no2, parameter_values["K_NO2_hyd"])
            * nitrite_share
            * hydrolysis_availability
            * x_h,
            parameter_values["eta_hyd_NO3"]
            * parameter_values["K_H"]
            * oxygen_hyd_limitation
            * _monod(s_no3, parameter_values["K_NO3_hyd"])
            * nitrate_share
            * hydrolysis_availability
            * x_h,
            parameter_values["eta_hyd_fe"]
            * parameter_values["K_H"]
            * oxygen_hyd_limitation
            * (parameter_values["K_NOX_hyd"] / max(parameter_values["K_NOX_hyd"] + s_nox, 1e-9))
            * hydrolysis_availability
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h
            * substrate_f
            * _share(s_f, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h
            * substrate_a
            * _share(s_a, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_f
            * _share(s_f, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_H"])
            * nitrate_share
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_f
            * _share(s_f, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_H"])
            * nitrite_share
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_a
            * _share(s_a, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_H"])
            * nitrate_share
            * x_h,
            parameter_values["mu_H"]
            * oxygen_h_limitation
            * substrate_a
            * _share(s_a, s_a, s_f)
            * ammonium_h
            * phosphate_h
            * alk_h
            * parameter_values["eta_H_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_H"])
            * nitrite_share
            * x_h,
            parameter_values["q_Fe"]
            * parameter_values["mu_H"]
            * oxygen_h_limitation
            * (parameter_values["K_NOX_H"] / max(parameter_values["K_NOX_H"] + s_nox, 1e-9))
            * substrate_fe
            * alk_h
            * x_h,
            parameter_values["b_H"] * x_h,
            parameter_values["q_PHA"] * substrate_a * alk_pao * r_pp * x_pao,
            parameter_values["q_PP"] * oxygen_pao * phosphate_storage * alk_pao * r_pha * c_pp * x_pao,
            parameter_values["q_PP"]
            * oxygen_pao_limitation
            * phosphate_storage
            * alk_pao
            * r_pha
            * c_pp
            * parameter_values["eta_PAO_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_PAO"])
            * nitrate_share
            * x_pao,
            parameter_values["q_PP"]
            * oxygen_pao_limitation
            * phosphate_storage
            * alk_pao
            * r_pha
            * c_pp
            * parameter_values["eta_PAO_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_PAO"])
            * nitrite_share
            * x_pao,
            parameter_values["mu_PAO"] * oxygen_pao * ammonium_pao * phosphate_pao * r_pha * alk_pao * x_pao,
            parameter_values["mu_PAO"]
            * oxygen_pao_limitation
            * ammonium_pao
            * phosphate_pao
            * r_pha
            * alk_pao
            * parameter_values["eta_PAO_NO3"]
            * _monod(s_no3, parameter_values["K_NO3_PAO"])
            * nitrate_share
            * x_pao,
            parameter_values["mu_PAO"]
            * oxygen_pao_limitation
            * ammonium_pao
            * phosphate_pao
            * r_pha
            * alk_pao
            * parameter_values["eta_PAO_NO2"]
            * _monod(s_no2, parameter_values["K_NO2_PAO"])
            * nitrite_share
            * x_pao,
            parameter_values["b_PAO"] * x_pao * alk_pao,
            parameter_values["b_PP"] * x_pp * alk_pao,
            parameter_values["b_PHA"] * x_pha * alk_pao,
            parameter_values["mu_AOB"]
            * _monod(s_o, parameter_values["K_O_AOB"])
            * _monod(s_nh4, parameter_values["K_NH4_AOB"])
            * phosphate_nit
            * alk_nit
            * x_aob,
            parameter_values["mu_NOB"]
            * _monod(s_o, parameter_values["K_O_NOB"])
            * _monod(s_no2, parameter_values["K_NO2_NOB"])
            * phosphate_nit
            * alk_nit
            * x_nob,
            parameter_values["b_AOB"] * x_aob,
            parameter_values["b_NOB"] * x_nob,
            parameter_values["k_PRE"] * s_po4 * x_meoh,
            parameter_values["k_RED"] * x_mep * alk_chem,
        ],
        dtype=float,
    )
    if not np.all(np.isfinite(process_rates)):
        raise FloatingPointError("ASM2d-TSN process rates must remain finite.")
    if np.any(process_rates < 0.0):
        raise FloatingPointError("ASM2d-TSN process rates must remain non-negative.")
    return process_rates


def _compute_aeration_flux(state: np.ndarray, aeration: float, state_index: Mapping[str, int], model_params: Mapping[str, Any]) -> float:
    aeration_model = model_params["aeration_model"]
    kla = float(aeration_model["kla_base"]) + float(aeration_model["kla_per_aeration"]) * max(float(aeration), 0.0)
    do_saturation = float(aeration_model["do_saturation"])
    return kla * (do_saturation - state[state_index["S_O"]])


def _steady_state_residuals(
    state: np.ndarray,
    influent_state: np.ndarray,
    hrt_hours: float,
    aeration: float,
    matrix_bundle: Mapping[str, Any],
    model_params: Mapping[str, Any],
    parameter_values: Mapping[str, float],
) -> np.ndarray:
    dilution_rate = 24.0 / max(float(hrt_hours), 1e-6)
    state_index = dict(matrix_bundle["state_index"])
    residual = dilution_rate * (influent_state - state)
    process_rates = _compute_process_rates(state, model_params, state_index, parameter_values)
    residual += process_rates @ np.asarray(matrix_bundle["petersen_matrix"], dtype=float)
    residual[state_index["S_O"]] += _compute_aeration_flux(state, aeration, state_index, model_params)
    return residual


def simulate_asm2d_tsn_steady_state(
    *,
    influent_state: np.ndarray,
    hrt_hours: float,
    aeration: float,
    model_params: Mapping[str, Any],
    matrix_bundle: Mapping[str, Any] | None = None,
    previous_solution: np.ndarray | None = None,
    enforce_acceptance: bool = True,
) -> tuple[np.ndarray, dict[str, float | bool | int]]:
    """Solve a single mechanistic steady-state ASM2D-TSN operating point."""

    matrix_bundle = matrix_bundle if matrix_bundle is not None else get_asm2d_tsn_matrices(model_params)
    resolved_measured_output_columns = (
        list(matrix_bundle["measured_output_columns"])
        if "measured_output_columns" in matrix_bundle
        else None
    )
    runtime = _validate_runtime_structure(
        model_params,
        measured_output_columns=resolved_measured_output_columns,
    )
    parameter_values = _build_parameter_value_map(runtime["workbook_config"]["parameters"])
    state_columns = list(runtime["state_columns"])
    state_index = dict(matrix_bundle["state_index"])
    solver = runtime["solver"]
    lower_bounds = np.full(len(state_columns), float(solver["lower_bound"]), dtype=float)
    upper_bounds = np.full(len(state_columns), float(solver["upper_bound"]), dtype=float)
    initial_guess = _build_initial_guess(
        influent_state,
        hrt_hours,
        aeration,
        state_index,
        model_params,
        previous_solution=previous_solution,
    )
    candidate_guesses = [initial_guess, _build_multistart_guess(initial_guess, influent_state, state_index, model_params)]
    candidate_labels = ["initial", "multistart"]

    best_result = None
    best_residual_max = np.inf
    best_result_label = "initial"
    candidate_residuals: dict[str, float] = {}
    candidate_diagnostics: dict[str, dict[str, float | bool | int]] = {}
    for candidate_label, candidate_guess in zip(candidate_labels, candidate_guesses, strict=True):
        result = least_squares(
            _steady_state_residuals,
            candidate_guess,
            bounds=(lower_bounds, upper_bounds),
            xtol=float(solver["variable_tolerance"]),
            ftol=float(solver["residual_tolerance"]),
            gtol=float(solver["gradient_tolerance"]),
            max_nfev=int(solver["max_nfev"]),
            args=(influent_state, hrt_hours, aeration, matrix_bundle, model_params, parameter_values),
        )
        residual_max = float(np.max(np.abs(result.fun)))
        candidate_residuals[candidate_label] = residual_max
        candidate_diagnostics[candidate_label] = {
            "success": bool(result.success),
            "status": int(result.status),
            "nfev": int(result.nfev),
            "residual_l2": float(np.linalg.norm(result.fun)),
            "residual_max": residual_max,
            "cost": float(result.cost),
            "optimality": float(result.optimality),
        }
        if residual_max < best_residual_max:
            best_result = result
            best_residual_max = residual_max
            best_result_label = candidate_label

    dynamic_relaxation_used = best_residual_max > float(solver["acceptance_residual_max"])
    dynamic_relaxation_improved = False
    dynamic_diagnostics: dict[str, Any] = {
        "attempted": dynamic_relaxation_used,
        "success": False,
        "status": -1,
        "nfev": 0,
        "njev": 0,
        "nlu": 0,
        "message": "not_attempted",
        "refit_success": False,
        "refit_status": -1,
        "refit_nfev": 0,
        "refit_residual_l2": np.nan,
        "refit_residual_max": np.nan,
    }
    if best_residual_max > float(solver["acceptance_residual_max"]):
        dynamic_result = solve_ivp(
            lambda _time, values: _steady_state_residuals(
                values,
                influent_state,
                hrt_hours,
                aeration,
                matrix_bundle,
                model_params,
                parameter_values,
            ),
            (0.0, float(solver["dynamic_relaxation_days"])),
            np.clip(candidate_guesses[-1], lower_bounds, upper_bounds),
            method="BDF",
            atol=float(solver["dynamic_absolute_tolerance"]),
            rtol=float(solver["dynamic_relative_tolerance"]),
            max_step=float(solver["dynamic_max_step"]),
        )
        dynamic_diagnostics.update(
            success=bool(dynamic_result.success),
            status=int(dynamic_result.status),
            nfev=int(dynamic_result.nfev),
            njev=int(getattr(dynamic_result, "njev", 0) or 0),
            nlu=int(getattr(dynamic_result, "nlu", 0) or 0),
            message=str(dynamic_result.message),
        )
        if dynamic_result.success:
            relaxed_guess = np.clip(dynamic_result.y[:, -1], lower_bounds, upper_bounds)
            result = least_squares(
                _steady_state_residuals,
                relaxed_guess,
                bounds=(lower_bounds, upper_bounds),
                xtol=float(solver["variable_tolerance"]),
                ftol=float(solver["residual_tolerance"]),
                gtol=float(solver["gradient_tolerance"]),
                max_nfev=int(solver["max_nfev"]),
                args=(influent_state, hrt_hours, aeration, matrix_bundle, model_params, parameter_values),
            )
            residual_max = float(np.max(np.abs(result.fun)))
            dynamic_diagnostics.update(
                refit_success=bool(result.success),
                refit_status=int(result.status),
                refit_nfev=int(result.nfev),
                refit_residual_l2=float(np.linalg.norm(result.fun)),
                refit_residual_max=residual_max,
            )
            if residual_max < best_residual_max:
                best_result = result
                best_residual_max = residual_max
                best_result_label = "dynamic_relaxation"
                dynamic_relaxation_improved = True

    assert best_result is not None
    result = best_result
    accepted = bool(result.success and best_residual_max <= float(solver["acceptance_residual_max"]))
    diagnostics: dict[str, float | bool | int] = {
        "success": bool(result.success),
        "accepted": accepted,
        "status": int(result.status),
        "nfev": int(result.nfev),
        "residual_l2": float(np.linalg.norm(result.fun)),
        "residual_max": best_residual_max,
        "acceptance_threshold": float(solver["acceptance_residual_max"]),
        **{f"initial_{key}": value for key, value in candidate_diagnostics["initial"].items()},
        **{f"multistart_{key}": value for key, value in candidate_diagnostics["multistart"].items()},
        **{f"bdf_{key}": value for key, value in dynamic_diagnostics.items()},
        "selected_strategy": best_result_label,
        "dynamic_relaxation_used": dynamic_relaxation_used,
        "dynamic_relaxation_improved": dynamic_relaxation_improved,
    }

    if enforce_acceptance and (not accepted):
        raise RuntimeError(
            "asm2d_tsn_simulation steady-state solve failed: "
            f"success={result.success}, status={result.status}, residual_max={diagnostics['residual_max']:.3e}"
        )

    return result.x, diagnostics


def _compute_measured_output_values(state: np.ndarray, matrix_bundle: Mapping[str, Any]) -> np.ndarray:
    return np.asarray(matrix_bundle["composition_matrix"], dtype=float) @ state


def _generate_asm2d_tsn_dataset_chunk(
    *,
    chunk_start: int,
    chunk_size: int,
    chunk_seed: int,
    model_params: Mapping[str, Any],
    matrix_bundle: Mapping[str, Any],
    runtime: Mapping[str, Any],
    collect_debug_data: bool = False,
    progress_bar=None,
) -> tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[dict[str, Any]]]:
    configured_hyperparameters = model_params["hyperparameters"]
    max_sample_attempts = int(configured_hyperparameters["max_sample_attempts"])
    state_columns = list(runtime["state_columns"])
    operational_columns = list(runtime["operational_columns"])
    measured_output_columns = list(runtime["measured_output_columns"])
    state_index = dict(matrix_bundle["state_index"])
    parameter_values = _build_parameter_value_map(runtime["workbook_config"]["parameters"])

    # Pre-generate a joint LHS candidate pool covering all retry slots.
    # Influent-state and operational columns are combined into a single LHS design
    # so their stratification is jointly controlled.  Sequential consumption means
    # each retry draws the next LHS point rather than an independent uniform draw.
    total_candidates = chunk_size * max_sample_attempts
    all_columns = state_columns + operational_columns
    all_ranges = {**runtime["influent_state_ranges"], **runtime["operational_ranges"]}
    candidate_pool = _generate_lhs_candidate_pool(chunk_seed, total_candidates, all_columns, all_ranges)
    n_state = len(state_columns)

    influent_states = np.zeros((chunk_size, len(state_columns)), dtype=float)
    operational = np.zeros((chunk_size, len(operational_columns)), dtype=float)
    effluent_states = np.zeros((chunk_size, len(state_columns)), dtype=float)
    measured_outputs = np.zeros((chunk_size, len(measured_output_columns)), dtype=float)
    solver_diagnostics: list[dict[str, Any]] = []

    previous_solution: np.ndarray | None = None
    for local_index in range(chunk_size):
        last_error: RuntimeError | None = None
        for attempt_index in range(max_sample_attempts):
            candidate_index = local_index * max_sample_attempts + attempt_index
            sampled_state = candidate_pool[candidate_index, :n_state]
            candidate_influent = _build_influent_state_sample(sampled_state)
            candidate_operational = candidate_pool[candidate_index, n_state:]

            try:
                effluent_state, diagnostics = simulate_asm2d_tsn_steady_state(
                    influent_state=candidate_influent,
                    hrt_hours=float(candidate_operational[0]),
                    aeration=float(candidate_operational[1]),
                    model_params=model_params,
                    matrix_bundle=matrix_bundle,
                    previous_solution=previous_solution,
                )
            except RuntimeError as error:
                last_error = error
                continue

            if not diagnostics["accepted"]:
                last_error = RuntimeError(
                    "asm2d_tsn_simulation steady-state solve did not satisfy the configured acceptance threshold."
                )
                continue

            if not np.all(np.isfinite(effluent_state)):
                last_error = RuntimeError("asm2d_tsn_simulation produced non-finite effluent states.")
                continue

            previous_solution = effluent_state
            influent_states[local_index] = candidate_influent
            operational[local_index] = candidate_operational
            effluent_states[local_index] = effluent_state
            measured_outputs[local_index] = _compute_measured_output_values(effluent_state, matrix_bundle)
            if collect_debug_data:
                diagnostic_record = dict(diagnostics)
                diagnostic_record["sample_index"] = chunk_start + local_index
                diagnostic_record["attempt_count"] = attempt_index + 1
                diagnostic_record["HRT"] = float(candidate_operational[0])
                diagnostic_record["Aeration"] = float(candidate_operational[1])
                solver_diagnostics.append(diagnostic_record)
            if progress_bar is not None:
                progress_bar.update(1)
            break
        else:
            raise RuntimeError(
                "asm2d_tsn_simulation failed to generate a valid sample after "
                f"{max_sample_attempts} attempts at sample index {chunk_start + local_index}."
            ) from last_error

    return chunk_start, influent_states, operational, effluent_states, measured_outputs, solver_diagnostics


def _summarize_asm2d_tsn_solver_diagnostics(
    solver_diagnostics: pd.DataFrame,
    acceptance_threshold: float,
) -> dict[str, Any]:
    if solver_diagnostics.empty:
        return {
            "sample_count": 0,
            "accepted_count": 0,
            "accepted_rate": 0.0,
            "acceptance_threshold": acceptance_threshold,
            "multistart_selected_rate": 0.0,
            "dynamic_relaxation_used_rate": 0.0,
            "dynamic_relaxation_improved_rate": 0.0,
            "residual_max_quantiles": {},
            "nfev_quantiles": {},
            "selected_strategy_counts": {},
        }

    residual_max_quantiles = {
        "q50": float(solver_diagnostics["residual_max"].quantile(0.50)),
        "q90": float(solver_diagnostics["residual_max"].quantile(0.90)),
        "q95": float(solver_diagnostics["residual_max"].quantile(0.95)),
        "q99": float(solver_diagnostics["residual_max"].quantile(0.99)),
        "max": float(solver_diagnostics["residual_max"].max()),
    }
    nfev_quantiles = {
        "q50": float(solver_diagnostics["nfev"].quantile(0.50)),
        "q90": float(solver_diagnostics["nfev"].quantile(0.90)),
        "q95": float(solver_diagnostics["nfev"].quantile(0.95)),
        "max": float(solver_diagnostics["nfev"].max()),
    }
    return {
        "sample_count": int(len(solver_diagnostics)),
        "accepted_count": int(solver_diagnostics["accepted"].sum()),
        "accepted_rate": float(solver_diagnostics["accepted"].mean()),
        "acceptance_threshold": float(acceptance_threshold),
        "multistart_selected_rate": float((solver_diagnostics["selected_strategy"] == "multistart").mean()),
        "dynamic_relaxation_used_rate": float(solver_diagnostics["dynamic_relaxation_used"].mean()),
        "dynamic_relaxation_improved_rate": float(solver_diagnostics["dynamic_relaxation_improved"].mean()),
        "mean_attempt_count": float(solver_diagnostics.get("attempt_count", pd.Series([1])).mean()),
        "max_attempt_count": int(solver_diagnostics.get("attempt_count", pd.Series([1])).max()),
        "residual_max_quantiles": residual_max_quantiles,
        "nfev_quantiles": nfev_quantiles,
        "selected_strategy_counts": {
            str(name): int(count)
            for name, count in solver_diagnostics["selected_strategy"].value_counts().items()
        },
    }


def _resolve_parallel_workers(requested_workers: int, sample_count: int) -> int:
    if requested_workers < 0:
        raise ValueError("parallel_workers must be greater than or equal to 0.")

    if sample_count <= 1:
        return 1

    available_workers = os.cpu_count() or 1
    if requested_workers == 0:
        requested_workers = max(available_workers - 1, 1)

    return min(max(requested_workers, 1), available_workers, sample_count)


def _resolve_parallel_chunk_size(requested_chunk_size: int) -> int:
    if requested_chunk_size < 1:
        raise ValueError("parallel_chunk_size must be at least 1.")

    return requested_chunk_size


def _write_parameter_sheet(worksheet, parameter_rows: list[Mapping[str, Any]]) -> None:
    worksheet.freeze_panes = "A2"
    headers = ["category", "symbol", "excel_name", "description", "value", "unit"]
    _write_header_row(worksheet, headers)

    previous_category = None
    for row_number, parameter_row in enumerate(parameter_rows, start=2):
        current_category = str(parameter_row["category"])
        row_values = [
            current_category,
            str(parameter_row["symbol"]),
            str(parameter_row["excel_name"]),
            str(parameter_row["description"]),
            float(parameter_row["value"]),
            str(parameter_row["unit"]),
        ]

        for column_number, value in enumerate(row_values, start=1):
            cell = worksheet.cell(row=row_number, column=column_number, value=value)
            if column_number == PARAMETER_VALUE_COLUMN_INDEX:
                cell.number_format = "0.###############"

        if current_category != previous_category:
            for column_number in range(1, len(headers) + 1):
                worksheet.cell(row=row_number, column=column_number).fill = SECTION_FILL
        previous_category = current_category


def _write_stoichiometric_sheet(
    worksheet,
    workbook_config: Mapping[str, Any],
    parameter_refs: Mapping[str, str],
) -> None:
    worksheet.freeze_panes = "C2"
    state_columns = list(workbook_config["state_columns"])
    processes = list(workbook_config["processes"])
    headers = ["process_index", "process"] + state_columns
    _write_header_row(worksheet, headers)
    state_column_index = {state_name: position for position, state_name in enumerate(state_columns, start=3)}

    for row_number, process in enumerate(processes, start=2):
        worksheet.cell(row=row_number, column=1, value=int(process["index"]))
        worksheet.cell(row=row_number, column=2, value=str(process["name"]))
        direct_coefficients = STOICHIOMETRIC_COEFFICIENTS[row_number - 2]["coefficients"]

        for state_name in state_columns:
            column_number = state_column_index[state_name]
            cell = worksheet.cell(row=row_number, column=column_number)

            if state_name in direct_coefficients:
                cell.value = _format_formula(direct_coefficients[state_name], parameter_refs)
                continue

            if state_name == "S_NH4":
                cell.value = _build_weighted_formula(
                    row_number,
                    state_column_index,
                    NITROGEN_CONTINUITY_TERMS,
                    parameter_refs,
                    negate=True,
                )
                continue

            if state_name == "S_PO4":
                cell.value = _build_weighted_formula(
                    row_number,
                    state_column_index,
                    PHOSPHORUS_CONTINUITY_TERMS,
                    parameter_refs,
                    negate=True,
                )
                continue

            if state_name == "S_ALK":
                cell.value = _build_alkalinity_formula(row_number, state_column_index)
                continue

def _write_composition_sheet(
    worksheet,
    workbook_config: Mapping[str, Any],
    parameter_refs: Mapping[str, str],
) -> None:
    worksheet.freeze_panes = "D2"
    dissolved_state_columns = list(workbook_config["dissolved_state_columns"])
    particulate_state_columns = list(workbook_config["particulate_state_columns"])
    state_columns = list(workbook_config["state_columns"])
    legacy_composite_variables = workbook_config.get("composite_variables")
    composite_variables = _resolve_workbook_composite_variables(
        state_columns,
        legacy_composite_variables=(
            None if legacy_composite_variables is None else list(legacy_composite_variables)
        ),
    )
    state_units = dict(workbook_config["state_units"])
    headers = ["state_group", "state_variable", "unit"] + composite_variables
    _write_header_row(worksheet, headers)
    composite_column_index = {name: position for position, name in enumerate(composite_variables, start=4)}

    for row_number, state_name in enumerate(state_columns, start=2):
        state_group = "Dissolved" if state_name in dissolved_state_columns else "Particulate"
        worksheet.cell(row=row_number, column=1, value=state_group)
        worksheet.cell(row=row_number, column=2, value=state_name)
        worksheet.cell(row=row_number, column=3, value=state_units[state_name])

        for composite_name, expression in COMPOSITION_FORMULAS.get(state_name, {}).items():
            worksheet.cell(
                row=row_number,
                column=composite_column_index[composite_name],
                value=_format_formula(expression, parameter_refs),
            )


def _resolve_workbook_composite_variables(
    state_columns: list[str],
    *,
    legacy_composite_variables: list[str] | None,
) -> list[str]:
    composite_variables: list[str] = []
    for state_name in state_columns:
        for composite_name in COMPOSITION_FORMULAS.get(state_name, {}):
            normalized_name = str(composite_name)
            if normalized_name not in composite_variables:
                composite_variables.append(normalized_name)

    if not composite_variables:
        raise ValueError(
            "asm2d_tsn_simulation composition formulas do not define any composite output columns."
        )

    if legacy_composite_variables is not None:
        normalized_legacy = [str(name) for name in legacy_composite_variables]
        _validate_unique_names(normalized_legacy, "composite_variables")
        missing_formulas = sorted(name for name in normalized_legacy if name not in composite_variables)
        if missing_formulas:
            missing_display = ", ".join(missing_formulas)
            raise ValueError(
                "asm2d_tsn_simulation legacy workbook composite_variables contains outputs without "
                f"composition formulas: {missing_display}"
            )

    return composite_variables


def _write_header_row(worksheet, headers: list[str]) -> None:
    for column_number, header in enumerate(headers, start=1):
        cell = worksheet.cell(row=1, column=column_number, value=header)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")


def _format_formula(expression: str | float | int, parameter_refs: Mapping[str, str]) -> str:
    if isinstance(expression, (int, float)):
        return f"={expression:g}"

    formatted_expression = str(expression).format_map(parameter_refs)
    if formatted_expression.startswith("="):
        return formatted_expression

    return f"={formatted_expression}"


def _build_weighted_formula(
    row_number: int,
    state_column_index: Mapping[str, int],
    factor_terms: Mapping[str, str],
    parameter_refs: Mapping[str, str],
    *,
    negate: bool,
) -> str:
    terms: list[str] = []
    for state_name, factor_expression in factor_terms.items():
        cell_reference = f"{get_column_letter(state_column_index[state_name])}{row_number}"
        formatted_factor = factor_expression.format_map(parameter_refs)
        if formatted_factor == "1":
            terms.append(cell_reference)
        else:
            terms.append(f"{cell_reference}*({formatted_factor})")

    if not terms:
        return "=0"

    expression = "+".join(terms)
    if negate:
        return f"=-({expression})"

    return f"={expression}"


def _build_alkalinity_formula(row_number: int, state_column_index: Mapping[str, int]) -> str:
    ammonium_ref = f"{get_column_letter(state_column_index['S_NH4'])}{row_number}"
    nitrite_ref = f"{get_column_letter(state_column_index['S_NO2'])}{row_number}"
    nitrate_ref = f"{get_column_letter(state_column_index['S_NO3'])}{row_number}"
    phosphate_ref = f"{get_column_letter(state_column_index['S_PO4'])}{row_number}"
    return f"={ammonium_ref}/14-{nitrite_ref}/14-{nitrate_ref}/14+{phosphate_ref}/31"


def _auto_size_columns(worksheet) -> None:
    for column_cells in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(column_cells[0].column)
        for cell in column_cells:
            if cell.value is None:
                continue
            max_length = max(max_length, len(str(cell.value)))
        worksheet.column_dimensions[column_letter].width = min(max(max_length + 2, 12), 48)


__all__ = [
    "build_asm2d_tsn_workbook",
    "build_asm2d_tsn_metadata",
    "create_asm2d_tsn_workbook",
    "generate_asm2d_tsn_dataset",
    "get_asm2d_tsn_matrices",
    "load_asm2d_tsn_workbook_composition",
    "load_asm2d_tsn_simulation_params",
    "resolve_asm2d_tsn_composition_cache_paths",
    "resolve_asm2d_tsn_simulation_artifact_paths",
    "resolve_asm2d_tsn_workbook_path",
    "run_asm2d_tsn_simulation",
    "simulate_asm2d_tsn_steady_state",
    "sweep_asm2d_tsn_operating_space",
]


In [ ]:
def evaluate_workbook_cell(workbook, sheet_name: str, coordinate: str, cache: dict[str, float], stack: set[str]) -> float:
    key = _normalize_excel_reference(sheet_name, coordinate)
    if key in cache:
        return cache[key]
    if key in stack:
        raise ValueError(f"Circular workbook formula reference at {key}.")
    stack.add(key)
    cell = workbook[sheet_name][coordinate]
    value = cell.value
    if value is None or value == "":
        result = 0.0
    elif isinstance(value, bool):
        result = float(value)
    elif isinstance(value, (int, float)):
        result = float(value)
    else:
        expression = str(value).strip()
        if not expression.startswith("="):
            result = float(expression)
        else:
            expression = expression[1:]

            def replace_reference(match: re.Match[str]) -> str:
                referenced_sheet = match.group("sheet_quoted") or match.group("sheet_unquoted") or sheet_name
                referenced_coordinate = f"{match.group('column').upper()}{match.group('row')}"
                resolved = evaluate_workbook_cell(
                    workbook, referenced_sheet, referenced_coordinate, cache, stack
                )
                return repr(float(resolved))

            resolved_expression = _EXCEL_REFERENCE_PATTERN.sub(replace_reference, expression).replace("^", "**")
            result = float(eval(resolved_expression, {"__builtins__": {}}, {}))
    stack.remove(key)
    cache[key] = result
    return result


def reevaluate_workbook_matrices(path: Path, state_columns: Sequence[str]) -> tuple[np.ndarray, np.ndarray, list[str], list[str], list[dict[str, Any]], dict[str, str]]:
    workbook = load_workbook(path, data_only=False, read_only=False)
    try:
        stoich_sheet = workbook["stoichiometric_matrix"]
        composition_sheet = workbook["composition_matrix"]
        parameter_sheet = workbook["parameter_table"]
        if parameter_sheet.max_row - 1 != 81 or parameter_sheet.max_column != 6:
            raise ValueError("The parameter table must contain exactly 81 rows and six declared fields.")
        parameter_headers = [str(parameter_sheet.cell(1, col).value).strip() for col in range(1, 7)]
        if parameter_headers != ["category", "symbol", "excel_name", "description", "value", "unit"]:
            raise ValueError("The parameter-table schema is invalid.")
        workbook_parameters = [
            {
                "category": str(parameter_sheet.cell(row, 1).value).strip(),
                "symbol": str(parameter_sheet.cell(row, 2).value).strip(),
                "excel_name": str(parameter_sheet.cell(row, 3).value).strip(),
                "description": str(parameter_sheet.cell(row, 4).value).strip(),
                "value": float(parameter_sheet.cell(row, 5).value),
                "unit": str(parameter_sheet.cell(row, 6).value).strip(),
            }
            for row in range(2, 83)
        ]
        if any(not np.isfinite(parameter["value"]) or not all(str(parameter[field]).strip() for field in ("category", "symbol", "excel_name", "description", "unit")) for parameter in workbook_parameters):
            raise ValueError("Workbook parameters must have finite values and complete metadata.")
        cache: dict[str, float] = {}
        stoich_headers = [str(stoich_sheet.cell(1, col).value).strip() for col in range(3, 23)]
        if stoich_headers != list(state_columns):
            raise ValueError("Workbook stoichiometric state order does not match the declared state order.")
        process_names = [str(stoich_sheet.cell(row, 2).value).strip() for row in range(2, 30)]
        stoich = np.asarray(
            [
                [evaluate_workbook_cell(workbook, stoich_sheet.title, stoich_sheet.cell(row, col).coordinate, cache, set())
                 for col in range(3, 23)]
                for row in range(2, 30)
            ],
            dtype=np.float64,
        )
        composition_headers = [str(composition_sheet.cell(1, col).value).strip() for col in range(4, 8)]
        composition_states = [str(composition_sheet.cell(row, 2).value).strip() for row in range(2, 22)]
        state_units = {
            str(composition_sheet.cell(row, 2).value).strip(): str(composition_sheet.cell(row, 3).value).strip()
            for row in range(2, 22)
        }
        if composition_states != list(state_columns):
            raise ValueError("Workbook composition state order does not match the declared state order.")
        composition = np.asarray(
            [
                [evaluate_workbook_cell(workbook, composition_sheet.title, composition_sheet.cell(row, col).coordinate, cache, set())
                 for row in range(2, 22)]
                for col in range(4, 8)
            ],
            dtype=np.float64,
        )
    finally:
        workbook.close()
    return stoich, composition, process_names, composition_headers, workbook_parameters, state_units


def read_cached_workbook_matrices(path: Path) -> tuple[np.ndarray, np.ndarray]:
    workbook = load_workbook(path, data_only=True, read_only=True)
    formula_workbook = load_workbook(path, data_only=False, read_only=True)
    try:
        stoich_sheet = workbook["stoichiometric_matrix"]
        composition_sheet = workbook["composition_matrix"]
        stoich_formula_sheet = formula_workbook["stoichiometric_matrix"]
        composition_formula_sheet = formula_workbook["composition_matrix"]
        for row in range(2, 30):
            for col in range(3, 23):
                formula = stoich_formula_sheet.cell(row, col).value
                cached = stoich_sheet.cell(row, col).value
                if isinstance(formula, str) and formula.startswith("=") and cached is None:
                    raise ValueError(f"Missing cached result for stoichiometric formula {stoich_sheet.cell(row, col).coordinate}.")
        for row in range(2, 22):
            for col in range(4, 8):
                formula = composition_formula_sheet.cell(row, col).value
                cached = composition_sheet.cell(row, col).value
                if isinstance(formula, str) and formula.startswith("=") and cached is None:
                    raise ValueError(f"Missing cached result for composition formula {composition_sheet.cell(row, col).coordinate}.")
        stoich = np.asarray(
            [[stoich_sheet.cell(row, col).value or 0.0 for col in range(3, 23)] for row in range(2, 30)],
            dtype=np.float64,
        )
        composition = np.asarray(
            [[composition_sheet.cell(row, col).value or 0.0 for row in range(2, 22)] for col in range(4, 8)],
            dtype=np.float64,
        )
    finally:
        workbook.close()
        formula_workbook.close()
    if not np.all(np.isfinite(stoich)) or not np.all(np.isfinite(composition)):
        raise ValueError("The workbook must contain finite cached results for every supported formula.")
    return stoich, composition


MATRIX_BUNDLE = get_asm2d_tsn_matrices(
    SIM_PARAMS,
    repo_root=ROOT,
    paths_config=load_paths_config(ROOT),
    use_composition_cache=False,
)


In [ ]:
STATE_COLUMNS = list(MATRIX_BUNDLE["state_columns"])
OPERATIONAL_COLUMNS = list(SIM_PARAMS["operational_columns"])
MEASURED_COLUMNS = list(MATRIX_BUNDLE["measured_output_columns"])
PETERSEN = np.asarray(MATRIX_BUNDLE["petersen_matrix"], dtype=np.float64)
COMPOSITION = np.asarray(MATRIX_BUNDLE["composition_matrix"], dtype=np.float64)
workbook_petersen, workbook_composition, workbook_processes, workbook_composites, workbook_parameters, workbook_state_units = reevaluate_workbook_matrices(
    WORKBOOK_PATH, STATE_COLUMNS
)
cached_petersen, cached_composition = read_cached_workbook_matrices(WORKBOOK_PATH)
if PETERSEN.shape != (28, 20) or COMPOSITION.shape != (4, 20):
    raise ValueError("The mechanistic matrices must have shapes 28x20 and 4x20.")
if not np.allclose(PETERSEN, workbook_petersen, atol=1e-12, rtol=1e-12):
    raise ValueError("Re-evaluated workbook stoichiometry differs from the assembled Petersen matrix.")
if not np.allclose(COMPOSITION, workbook_composition, atol=1e-12, rtol=1e-12):
    raise ValueError("Re-evaluated workbook composition differs from the assembled composition matrix.")
if not np.allclose(cached_petersen, workbook_petersen, atol=1e-12, rtol=1e-12):
    raise ValueError("Cached workbook stoichiometry differs from the independently re-evaluated formulas.")
if not np.allclose(cached_composition, workbook_composition, atol=1e-12, rtol=1e-12):
    raise ValueError("Cached workbook composition differs from the independently re-evaluated formulas.")
if workbook_processes != list(MATRIX_BUNDLE["process_names"]) or workbook_composites != MEASURED_COLUMNS:
    raise ValueError("Workbook process or composite ordering differs from the declared contract.")
configured_parameters = ACTIVE["simulation"]["workbook"]["parameters"]
if len(configured_parameters) != len(workbook_parameters):
    raise ValueError("Workbook and configuration parameter counts differ.")
for index, (workbook_parameter, configured_parameter) in enumerate(zip(workbook_parameters, configured_parameters, strict=True)):
    for field in ("category", "symbol", "excel_name", "description", "unit"):
        if str(workbook_parameter[field]) != str(configured_parameter[field]):
            raise ValueError(f"Workbook parameter {index + 1} field {field} differs from the configuration.")
    if not np.isclose(float(workbook_parameter["value"]), float(configured_parameter["value"]), rtol=0.0, atol=0.0):
        raise ValueError(f"Workbook parameter {index + 1} value differs from the configuration.")
if workbook_state_units != ACTIVE["simulation"]["workbook"]["state_units"]:
    raise ValueError("Workbook state units differ from the configuration.")

singular_values = np.linalg.svd(PETERSEN, compute_uv=False)
rank_tolerance = PETERSEN.shape[0] * np.finfo(np.float64).eps * singular_values[0]
PETERSEN_RANK = int(np.sum(singular_values > rank_tolerance))
A_MATRIX = null_space(PETERSEN, rcond=rank_tolerance / singular_values[0]).T
Z_MATRIX = null_space(A_MATRIX)
if PETERSEN_RANK != 15 or A_MATRIX.shape != (5, 20) or Z_MATRIX.shape != (20, 15):
    raise ValueError("Expected rank 15 with five invariants and a 15-dimensional feasible direction space.")
invariant_error = float(np.max(np.abs(A_MATRIX @ PETERSEN.T)))
aeration_error = float(np.max(np.abs(A_MATRIX[:, STATE_COLUMNS.index("S_O")])))
if invariant_error > 1e-12:
    raise ValueError(f"Invariant construction failed: max |A nu^T|={invariant_error:.3e}.")
if aeration_error > 1e-12:
    raise ValueError(f"The invariant operator does not annihilate the aeration basis: {aeration_error:.3e}.")

matrix_path = atomic_npz(
    DIRS["matrices"] / "operators.npz",
    petersen=PETERSEN,
    composition=COMPOSITION,
    invariant=A_MATRIX,
    null_basis=Z_MATRIX,
    singular_values=singular_values,
)
matrix_summary_path = atomic_json(
    DIRS["matrices"] / "validation.json",
    {
        "petersen_shape": list(PETERSEN.shape),
        "composition_shape": list(COMPOSITION.shape),
        "rank": PETERSEN_RANK,
        "rank_tolerance": rank_tolerance,
        "max_abs_A_nu_transpose": invariant_error,
        "max_abs_A_aeration_basis": aeration_error,
        "cached_formula_crosscheck": True,
        "state_columns": STATE_COLUMNS,
        "process_names": workbook_processes,
        "composite_columns": MEASURED_COLUMNS,
        "workbook_sha256": WORKBOOK_HASH,
        "parameter_rows_validated": len(workbook_parameters),
        "state_units": workbook_state_units,
    },
)
register_artifact("operators", matrix_path)
register_artifact("matrix_validation", matrix_summary_path)
stage_complete("workbook_and_matrices", rank=PETERSEN_RANK, invariant_error=invariant_error)
print(f"Validated workbook; rank={PETERSEN_RANK}, max |A nu^T|={invariant_error:.3e}")


# Fixed midpoint kinetics fixture and an exercised 20-day BDF fallback.
fixture_state = np.asarray(
    [
        np.mean(ACTIVE["simulation"]["influent_state_ranges"][name])
        for name in STATE_COLUMNS
    ],
    dtype=np.float64,
)
fixture_runtime = _validate_runtime_structure(
    SIM_PARAMS,
    measured_output_columns=MEASURED_COLUMNS,
)
fixture_parameters = _build_parameter_value_map(fixture_runtime["workbook_config"]["parameters"])
fixture_rates = _compute_process_rates(
    fixture_state,
    SIM_PARAMS,
    MATRIX_BUNDLE["state_index"],
    fixture_parameters,
)
expected_fixture_rates = np.asarray(
    [
        92.69796111901375, 9.101254364412256, 28.764458238142446, 2.471945629840367,
        128.93214239104688, 52.085812791737965, 60.01205173110546, 18.988188243045084,
        24.243578317608947, 7.6708197020559545, 36.83730535582387, 23.0,
        144.17449029658616, 0.7426197345146636, 0.026884334631318736, 0.10936763362852317,
        9.873012234382074, 0.3574229883638925, 1.4540254381544506, 6.496178718400941,
        2.198706643151088, 3.098177542621987, 1.9064642665015905, 0.603245727879159,
        0.8500000000000001, 0.7225, 60.0, 3.5894428152492663,
    ],
    dtype=np.float64,
)
if not np.allclose(fixture_rates, expected_fixture_rates, rtol=1e-12, atol=1e-12):
    raise AssertionError("The fixed mechanistic rate fixture changed.")
fallback_params = copy.deepcopy(SIM_PARAMS)
fallback_params["solver"]["max_nfev"] = 1
fallback_params["solver"]["acceptance_residual_max"] = 0.0
_, fallback_diagnostics = simulate_asm2d_tsn_steady_state(
    influent_state=fixture_state,
    hrt_hours=21.0,
    aeration=1.5,
    model_params=fallback_params,
    matrix_bundle=MATRIX_BUNDLE,
    enforce_acceptance=False,
)
if not (
    fallback_diagnostics["dynamic_relaxation_used"]
    and fallback_diagnostics["bdf_attempted"]
    and fallback_diagnostics["bdf_success"]
    and int(fallback_diagnostics["bdf_nfev"]) > 0
):
    raise AssertionError("The declared BDF fallback path was not exercised successfully.")
stage_complete("mechanistic_tests", known_rate_fixture=True, bdf_fallback=True)


## Deterministic mechanistic datasets

Candidates are generated in indexed Latin-hypercube batches and assigned round-robin to fixed warm-start chains. Thread completion order cannot change accepted-row order. All attempted candidates, including rejected solves, are retained under the run root.


In [ ]:
def stable_seed(label: str) -> int:
    base = int(ACTIVE["simulation"]["sampling"]["seed"])
    digest = hashlib.sha256(f"{base}:{label}".encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def make_candidate_batch(n_points: int, *, label: str, batch_index: int, regime: str | None = None) -> pd.DataFrame:
    all_columns = STATE_COLUMNS + OPERATIONAL_COLUMNS
    all_ranges = {
        **ACTIVE["simulation"]["influent_state_ranges"],
        **ACTIVE["simulation"]["operational_ranges"],
    }
    lower = np.asarray([all_ranges[name][0] for name in all_columns], dtype=np.float64)
    upper = np.asarray([all_ranges[name][1] for name in all_columns], dtype=np.float64)
    seed = int(ACTIVE["simulation"]["sampling"]["seed"]) if label == "id" and batch_index == 0 else stable_seed(f"{label}:batch:{batch_index}")
    samples = qmc_scale(LatinHypercube(d=len(all_columns), seed=seed).random(n_points), lower, upper)
    frame = pd.DataFrame(samples, columns=all_columns)
    frame["ood_regime"] = regime if regime is not None else "id"
    frame["perturbed_variables"] = ""
    if regime is not None:
        regime_config = ACTIVE["simulation"]["ood"]["regimes"][regime]
        ood_ranges = regime_config["ranges"]
        rng = np.random.default_rng(stable_seed(f"{label}:selection:{batch_index}"))
        one_fraction = float(ACTIVE["simulation"]["ood"]["candidate_mix"]["one_variable"])
        one_count = int(round(n_points * one_fraction))
        perturbation_counts = np.asarray([1] * one_count + [2] * (n_points - one_count), dtype=int)
        rng.shuffle(perturbation_counts)
        eligible = list(ood_ranges)
        for row_index, count in enumerate(perturbation_counts):
            selected = sorted(rng.choice(eligible, size=int(count), replace=False).tolist())
            frame.at[row_index, "perturbed_variables"] = ";".join(selected)
            for variable in selected:
                lo, hi = map(float, ood_ranges[variable])
                frame.at[row_index, variable] = rng.uniform(lo, hi)
    return frame


def solve_candidate_chain(
    chain_id: int,
    candidate_rows: list[tuple[int, dict[str, Any]]],
    previous_solution: np.ndarray | None,
) -> tuple[list[dict[str, Any]], np.ndarray | None]:
    records: list[dict[str, Any]] = []
    for candidate_id, row in candidate_rows:
        influent = np.asarray([row[name] for name in STATE_COLUMNS], dtype=np.float64)
        started = time.perf_counter()
        failure_reason = ""
        try:
            effluent, diagnostics = simulate_asm2d_tsn_steady_state(
                influent_state=influent,
                hrt_hours=float(row["HRT"]),
                aeration=float(row["Aeration"]),
                model_params=SIM_PARAMS,
                matrix_bundle=MATRIX_BUNDLE,
                previous_solution=previous_solution,
                enforce_acceptance=False,
            )
            finite = bool(np.all(np.isfinite(effluent)))
            accepted = bool(diagnostics["accepted"] and finite)
            if accepted:
                previous_solution = np.asarray(effluent, dtype=np.float64)
            elif not finite:
                failure_reason = "non_finite_effluent"
            else:
                failure_reason = (
                    f"solver_rejected: success={diagnostics['success']}, "
                    f"residual_max={float(diagnostics['residual_max']):.6e}"
                )
        except Exception as error:
            effluent = np.full(len(STATE_COLUMNS), np.nan, dtype=np.float64)
            diagnostics = {
                "success": False,
                "accepted": False,
                "status": -1,
                "nfev": -1,
                "residual_l2": np.nan,
                "residual_max": np.nan,
                "selected_strategy": "exception",
                "dynamic_relaxation_used": False,
                "dynamic_relaxation_improved": False,
            }
            accepted = False
            failure_reason = f"{type(error).__name__}: {error}"
        record = {
            "candidate_id": int(candidate_id),
            "chain_id": int(chain_id),
            "accepted": accepted,
            "failure_reason": failure_reason,
            "solve_seconds": time.perf_counter() - started,
            "ood_regime": row["ood_regime"],
            "perturbed_variables": row["perturbed_variables"],
            **{name: float(row[name]) for name in OPERATIONAL_COLUMNS},
            **{f"In_{name}": float(row[name]) for name in STATE_COLUMNS},
            **{f"Out_{name}": float(value) for name, value in zip(STATE_COLUMNS, effluent, strict=True)},
            **{f"solver_{key}": value for key, value in diagnostics.items()},
        }
        records.append(record)
    return records, previous_solution


def generate_dataset(target_count: int, *, label: str, regime: str | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    sampling = ACTIVE["simulation"]["sampling"]
    max_candidates = target_count * int(sampling["max_sample_attempts"])
    n_chains = int(sampling["n_chains"])
    workers = max(1, min(int(sampling["parallel_workers"]), n_chains))
    partial_attempts_path = DIRS["datasets"] / f"{label}.attempts.partial.parquet"
    partial_accepted_path = DIRS["datasets"] / f"{label}.accepted.partial.parquet"
    partial_state_path = DIRS["datasets"] / f"{label}.generation_state.json"

    accepted_records: list[dict[str, Any]] = []
    attempt_records: list[dict[str, Any]] = []
    next_candidate_id = 0
    batch_index = 0
    if bool(ACTIVE["run"].get("resume", True)) and partial_state_path.exists():
        state = read_json(partial_state_path)
        expected_state = {
            "label": label,
            "regime": regime,
            "target_count": target_count,
            "config_sha256": CONFIG_HASH,
        }
        if any(state.get(key) != value for key, value in expected_state.items()):
            raise RuntimeError(f"Partial {label} generation state does not match the active contract.")
        if partial_attempts_path.exists():
            partial_attempts = pd.read_parquet(partial_attempts_path).iloc[: int(state["attempt_count"])]
            attempt_records = partial_attempts.to_dict(orient="records")
        if partial_accepted_path.exists():
            partial_accepted = pd.read_parquet(partial_accepted_path).iloc[: int(state["accepted_count"])]
            accepted_records = partial_accepted.to_dict(orient="records")
        if len(attempt_records) != int(state["attempt_count"]) or len(accepted_records) != int(state["accepted_count"]):
            raise RuntimeError(f"Partial {label} generation files are incomplete.")
        next_candidate_id = int(state["next_candidate_id"])
        batch_index = int(state["batch_index"])

    chain_warm_starts: list[np.ndarray | None] = [None] * n_chains
    for record in sorted(accepted_records, key=lambda value: int(value["candidate_id"])):
        chain_id = int(record["chain_id"])
        chain_warm_starts[chain_id] = np.asarray(
            [record[f"Out_{name}"] for name in STATE_COLUMNS], dtype=np.float64
        )

    while len(accepted_records) < target_count and next_candidate_id < max_candidates:
        remaining = target_count - len(accepted_records)
        pool_size = target_count
        pool_index = next_candidate_id // pool_size
        pool_offset = next_candidate_id % pool_size
        batch_capacity = n_chains * int(sampling["parallel_chunk_size"])
        batch_count = min(
            max(remaining, n_chains),
            batch_capacity,
            pool_size - pool_offset,
            max_candidates - next_candidate_id,
        )
        candidate_pool = make_candidate_batch(pool_size, label=label, batch_index=pool_index, regime=regime)
        candidates = candidate_pool.iloc[pool_offset : pool_offset + batch_count].reset_index(drop=True)
        indexed = [(next_candidate_id + i, row.to_dict()) for i, (_, row) in enumerate(candidates.iterrows())]
        chains = [
            [item for item in indexed if int(item[0]) % n_chains == chain_id]
            for chain_id in range(n_chains)
        ]
        chain_results: list[dict[str, Any]] = []
        with ThreadPoolExecutor(max_workers=workers) as executor:
            future_map = {
                executor.submit(
                    solve_candidate_chain,
                    chain_id,
                    rows,
                    chain_warm_starts[chain_id],
                ): chain_id
                for chain_id, rows in enumerate(chains)
                if rows
            }
            for future in as_completed(future_map):
                records, final_warm_start = future.result()
                chain_id = future_map[future]
                chain_warm_starts[chain_id] = final_warm_start
                chain_results.extend(records)
        chain_results.sort(key=lambda record: int(record["candidate_id"]))
        attempt_records.extend(chain_results)
        for record in chain_results:
            if record["accepted"] and len(accepted_records) < target_count:
                accepted_records.append(record)
        next_candidate_id += batch_count
        batch_index += 1

        atomic_parquet(partial_attempts_path, pd.DataFrame(attempt_records))
        if accepted_records:
            atomic_parquet(partial_accepted_path, pd.DataFrame(accepted_records))
        atomic_json(
            partial_state_path,
            {
                "label": label,
                "regime": regime,
                "target_count": target_count,
                "config_sha256": CONFIG_HASH,
                "attempt_count": len(attempt_records),
                "accepted_count": len(accepted_records),
                "next_candidate_id": next_candidate_id,
                "batch_index": batch_index,
            },
        )

    if len(accepted_records) != target_count:
        raise RuntimeError(f"Generated only {len(accepted_records)} accepted {label} states from {next_candidate_id} candidates.")
    accepted = pd.DataFrame(accepted_records).copy()
    accepted.insert(0, "sample_id", [f"{label}_{index:06d}" for index in range(target_count)])
    accepted = accepted.drop(columns=["solve_seconds"], errors="ignore")
    attempts = pd.DataFrame(attempt_records).copy()
    accepted_identifier = dict(zip(accepted["candidate_id"], accepted["sample_id"], strict=True))
    attempts.insert(1, "accepted_sample_id", attempts["candidate_id"].map(accepted_identifier).fillna(""))
    attempts.insert(2, "selected_for_dataset", attempts["candidate_id"].isin(accepted_identifier))
    input_values = accepted[[f"In_{name}" for name in STATE_COLUMNS]].to_numpy(np.float64)
    output_values = accepted[[f"Out_{name}" for name in STATE_COLUMNS]].to_numpy(np.float64)
    input_composites = input_values @ COMPOSITION.T
    output_composites = output_values @ COMPOSITION.T
    for column_index, name in enumerate(MEASURED_COLUMNS):
        accepted[f"In_{name}"] = input_composites[:, column_index]
        accepted[f"Out_{name}"] = output_composites[:, column_index]
    for partial_path in (partial_attempts_path, partial_accepted_path, partial_state_path):
        with contextlib.suppress(FileNotFoundError):
            partial_path.unlink()
    return accepted, attempts


def load_or_generate_dataset(label: str, target_count: int, regime: str | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    dataset_path = DIRS["datasets"] / f"{label}.parquet"
    attempts_path = DIRS["datasets"] / f"{label}.attempts.parquet"
    if bool(ACTIVE["run"].get("resume", True)) and dataset_path.exists() and attempts_path.exists():
        dataset = pd.read_parquet(dataset_path)
        attempts = pd.read_parquet(attempts_path)
        if len(dataset) != target_count:
            raise RuntimeError(f"Resumed {label} dataset has the wrong accepted-row count.")
        return dataset, attempts
    dataset, attempts = generate_dataset(target_count, label=label, regime=regime)
    atomic_parquet(dataset_path, dataset)
    atomic_parquet(attempts_path, attempts)
    register_artifact(f"dataset_{label}", dataset_path)
    register_artifact(f"attempts_{label}", attempts_path)
    return dataset, attempts


ID_TARGET = int(ACTIVE["simulation"]["sampling"]["id_samples"])
ID_DATA, ID_ATTEMPTS = load_or_generate_dataset("id", ID_TARGET)
if ID_DATA[[f"Out_{name}" for name in STATE_COLUMNS]].isna().any().any():
    raise ValueError("Accepted in-distribution data contain non-finite effluent states.")
id_in = ID_DATA[[f"In_{name}" for name in STATE_COLUMNS]].to_numpy(np.float64)
id_out = ID_DATA[[f"Out_{name}" for name in STATE_COLUMNS]].to_numpy(np.float64)
ground_residual = (A_MATRIX @ id_out.T - A_MATRIX @ id_in.T).T
ground_l2 = np.linalg.norm(ground_residual, axis=1)
if float(ground_l2.max()) > float(ACTIVE["projection"]["tau"]):
    raise ValueError("Accepted mechanistic states violate the declared invariant tolerance.")
if float(id_out.min()) < -float(ACTIVE["projection"]["tau"]):
    raise ValueError("Accepted mechanistic states violate non-negativity.")
dataset_validation = {
    "accepted": len(ID_DATA),
    "attempted": len(ID_ATTEMPTS),
    "acceptance_rate": len(ID_DATA) / len(ID_ATTEMPTS),
    "maximum_ground_truth_conservation_l2": float(ground_l2.max()),
    "minimum_ground_truth_component": float(id_out.min()),
}
atomic_json(DIRS["datasets"] / "id.validation.json", dataset_validation)
register_artifact("id_validation", DIRS["datasets"] / "id.validation.json")
stage_complete("id_dataset", **dataset_validation)
print(dataset_validation)


## Positivity-preserving conservative projection

For each sample, the strictly positive provisional target is `max(raw, epsilon)`. QR solves the weighted least-squares system `W_s Z q = W_s (c_+ - c_in)` and the conservative candidate is `c_in + Z q`. Ill-conditioned or rank-deficient systems use an SVD fallback at the declared relative cutoff. Positivity backtracking follows the conservative line from the influent state; projected components are never clipped.


In [ ]:
@dataclass(frozen=True)
class ProjectionResult:
    values: np.ndarray
    alpha: np.ndarray
    solver: np.ndarray
    conservation_l2: np.ndarray
    minimum_component: np.ndarray


def project_one(raw: np.ndarray, influent: np.ndarray) -> tuple[np.ndarray, float, str, float, float]:
    eps = float(ACTIVE["projection"]["eps"])
    eta = float(ACTIVE["projection"]["eta"])
    tau = float(ACTIVE["projection"]["tau"])
    svd_cutoff = float(ACTIVE["projection"]["svd_tolerance"])
    raw = np.asarray(raw, dtype=np.float64)
    influent = np.asarray(influent, dtype=np.float64)
    if raw.shape != (20,) or influent.shape != (20,):
        raise ValueError("Projection inputs must be 20-component vectors.")
    if np.any(~np.isfinite(raw)) or np.any(~np.isfinite(influent)) or np.any(influent < 0.0):
        raise ValueError("Projection requires finite raw values and a non-negative influent state.")
    positive_target = np.maximum(raw, eps)
    scaling = 1.0 / positive_target
    scaled_basis = scaling[:, None] * Z_MATRIX
    target_change = scaling * (positive_target - influent)
    solver = "qr"
    try:
        q_matrix, r_matrix = np.linalg.qr(scaled_basis, mode="reduced")
        condition = np.linalg.cond(r_matrix)
        if not np.isfinite(condition) or condition >= 1.0 / svd_cutoff:
            raise np.linalg.LinAlgError("Scaled QR system is ill-conditioned.")
        coefficients = np.linalg.solve(r_matrix, q_matrix.T @ target_change)
    except np.linalg.LinAlgError:
        solver = "svd"
        coefficients, _, rank, _ = scipy.linalg.lstsq(
            scaled_basis, target_change, cond=svd_cutoff, lapack_driver="gelsd"
        )
        if rank < Z_MATRIX.shape[1]:
            warnings.warn("The scaled projection used a rank-reduced SVD solution.", RuntimeWarning)
    candidate = influent + Z_MATRIX @ coefficients
    direction = candidate - influent
    alpha = 1.0
    if float(candidate.min()) < 0.0:
        decreasing = direction < 0.0
        if not np.any(decreasing):
            raise RuntimeError("Negative candidate has no decreasing component for backtracking.")
        alpha = float((1.0 - eta) * np.min(influent[decreasing] / (-direction[decreasing])))
        alpha = min(1.0, max(0.0, alpha))
    projected = influent + alpha * direction
    if np.any(~np.isfinite(candidate)) or np.any(~np.isfinite(projected)):
        raise RuntimeError("Projection produced a non-finite candidate or projected state.")
    conservation_l2 = float(np.linalg.norm(A_MATRIX @ projected - A_MATRIX @ influent))
    minimum = float(projected.min())
    if not np.isfinite(conservation_l2) or not np.isfinite(minimum) or conservation_l2 > tau or minimum < -tau:
        raise RuntimeError(
            f"Projected sample failed physical acceptance: residual={conservation_l2:.3e}, min={minimum:.3e}."
        )
    return projected, alpha, solver, conservation_l2, minimum


def project_batch(raw: np.ndarray, influent: np.ndarray) -> ProjectionResult:
    raw_array = np.asarray(raw, dtype=np.float64)
    influent_array = np.asarray(influent, dtype=np.float64)
    if raw_array.shape != influent_array.shape or raw_array.ndim != 2 or raw_array.shape[1] != 20:
        raise ValueError("Projection batches must be matching n-by-20 arrays.")
    rows = [project_one(raw_row, in_row) for raw_row, in_row in zip(raw_array, influent_array, strict=True)]
    return ProjectionResult(
        values=np.vstack([row[0] for row in rows]),
        alpha=np.asarray([row[1] for row in rows]),
        solver=np.asarray([row[2] for row in rows], dtype=object),
        conservation_l2=np.asarray([row[3] for row in rows]),
        minimum_component=np.asarray([row[4] for row in rows]),
    )


# Internal projection tests exercise identity, negative-output, zero-influent, and SVD-reference cases.
test_influent = np.maximum(id_in[0], 0.0)
test_raw = id_out[0].copy()
test_projected, test_alpha, _, test_residual, test_minimum = project_one(test_raw, test_influent)
assert test_residual <= float(ACTIVE["projection"]["tau"])
assert test_minimum >= -float(ACTIVE["projection"]["tau"])
assert np.allclose(test_projected, test_raw, atol=1e-8, rtol=1e-10)
negative_raw = test_raw.copy()
negative_raw[[1, 7, 12]] *= -1.0
negative_projected, negative_alpha, _, negative_residual, negative_minimum = project_one(negative_raw, test_influent)
assert 0.0 <= negative_alpha <= 1.0
assert negative_residual <= float(ACTIVE["projection"]["tau"])
assert negative_minimum >= -float(ACTIVE["projection"]["tau"])
zero_influent = test_influent.copy()
zero_influent[0] = 0.0
zero_projected, zero_alpha, _, zero_residual, zero_minimum = project_one(negative_raw, zero_influent)
assert 0.0 <= zero_alpha <= 1.0 and zero_residual <= float(ACTIVE["projection"]["tau"])
assert zero_minimum >= -float(ACTIVE["projection"]["tau"])

# The QR result is checked against the declared SVD least-squares fallback on a feasible case.
positive_reference = np.maximum(test_raw, float(ACTIVE["projection"]["eps"]))
reference_scaling = 1.0 / positive_reference
reference_basis = reference_scaling[:, None] * Z_MATRIX
reference_rhs = reference_scaling * (positive_reference - test_influent)
reference_coefficients, _, _, _ = scipy.linalg.lstsq(
    reference_basis,
    reference_rhs,
    cond=float(ACTIVE["projection"]["svd_tolerance"]),
    lapack_driver="gelsd",
)
reference_candidate = test_influent + Z_MATRIX @ reference_coefficients
assert np.allclose(test_projected, reference_candidate, atol=1e-8, rtol=1e-10)

stage_complete("projection_tests", cases=4, qr_svd_agreement=True, feasible_identity=True)
print("Projection property tests passed.")


## Persistent nested folds and surrogate adapters

The master permutation fixes nested membership and outer-fold roles before model fitting. Preprocessing is fitted only on the applicable training rows. Each adapter returns the same physical-unit 20-component output contract.


In [ ]:
FEATURE_COLUMNS = OPERATIONAL_COLUMNS + [f"In_{name}" for name in STATE_COLUMNS]
TARGET_COLUMNS = [f"Out_{name}" for name in STATE_COLUMNS]
INPUT_COMPONENT_COLUMNS = [f"In_{name}" for name in STATE_COLUMNS]
SAMPLE_SIZES = [int(value) for value in ACTIVE["evaluation"]["nested_cv"]["sample_sizes"]]
OUTER_FOLDS = int(ACTIVE["evaluation"]["nested_cv"]["outer_folds"])
INNER_FOLDS = int(ACTIVE["evaluation"]["nested_cv"]["inner_folds"])
N_TRIALS = int(ACTIVE["evaluation"]["nested_cv"]["n_trials"])
if SAMPLE_SIZES[-1] != len(ID_DATA) or any(size % OUTER_FOLDS for size in SAMPLE_SIZES):
    raise ValueError("Every nested size must be outer-fold balanced and the largest must equal the ID dataset.")

split_path = DIRS["splits"] / "master_assignments.parquet"
if bool(ACTIVE["run"].get("resume", True)) and split_path.exists():
    SPLITS = pd.read_parquet(split_path)
else:
    rng = np.random.default_rng(int(ACTIVE["evaluation"]["seed"]))
    order = rng.permutation(len(ID_DATA))
    fold_by_position = np.arange(len(ID_DATA), dtype=int) % OUTER_FOLDS
    outer_by_row = np.empty(len(ID_DATA), dtype=int)
    position_by_row = np.empty(len(ID_DATA), dtype=int)
    outer_by_row[order] = fold_by_position
    position_by_row[order] = np.arange(len(ID_DATA), dtype=int)
    first_size_by_row = np.asarray(
        [next(size for size in SAMPLE_SIZES if position < size) for position in position_by_row], dtype=int
    )
    SPLITS = pd.DataFrame(
        {
            "sample_id": ID_DATA["sample_id"].astype(str),
            "row_index": np.arange(len(ID_DATA), dtype=int),
            "master_position": position_by_row,
            "outer_fold": outer_by_row,
            "first_nested_size": first_size_by_row,
        }
    ).sort_values("master_position").reset_index(drop=True)
    atomic_parquet(split_path, SPLITS)
    register_artifact("master_splits", split_path)

for size in SAMPLE_SIZES:
    subset = SPLITS[SPLITS["master_position"] < size]
    counts = subset["outer_fold"].value_counts().sort_index().to_numpy()
    if not np.array_equal(counts, np.full(OUTER_FOLDS, size // OUTER_FOLDS)):
        raise AssertionError(f"Nested size {size} is not exactly outer-fold balanced.")
for smaller, larger in zip(SAMPLE_SIZES[:-1], SAMPLE_SIZES[1:], strict=True):
    small_ids = set(SPLITS.loc[SPLITS["master_position"] < smaller, "sample_id"])
    large_ids = set(SPLITS.loc[SPLITS["master_position"] < larger, "sample_id"])
    if not small_ids < large_ids:
        raise AssertionError("Nested datasets must be strict subsets.")


def inner_fold_ids(row_indices: np.ndarray, *, label: str) -> np.ndarray:
    indices = np.asarray(row_indices, dtype=int)
    rng = np.random.default_rng(stable_seed(f"inner:{label}"))
    permutation = rng.permutation(len(indices))
    by_position = np.arange(len(indices), dtype=int) % INNER_FOLDS
    result = np.empty(len(indices), dtype=int)
    result[permutation] = by_position
    return result


MODEL_LABELS = {
    "xgboost_regressor": "XGBoost",
    "lightgbm_regressor": "LightGBM",
    "catboost_regressor": "CatBoost",
    "adaboost_regressor": "AdaBoost",
    "random_forest_regressor": "Random Forest",
    "extra_trees_regressor": "Extra Trees",
    "svr_regressor": "SVR",
    "knn_regressor": "k-NN",
    "pls_regressor": "PLS",
    "multitask_elastic_net_regressor": "Multi-task Elastic Net",
    "multitask_lasso_regressor": "Multi-task Lasso",
    "ann_deep_regressor": "MLP",
    "tabnet_regressor": "TabNet",
}
MODEL_CATEGORIES = {
    "xgboost_regressor": "Boosting",
    "lightgbm_regressor": "Boosting",
    "catboost_regressor": "Boosting",
    "adaboost_regressor": "Boosting",
    "random_forest_regressor": "Randomized tree ensembles",
    "extra_trees_regressor": "Randomized tree ensembles",
    "svr_regressor": "Kernel",
    "knn_regressor": "Instance based",
    "pls_regressor": "Interpretable models",
    "multitask_elastic_net_regressor": "Interpretable models",
    "multitask_lasso_regressor": "Interpretable models",
    "ann_deep_regressor": "Neural networks",
    "tabnet_regressor": "Neural networks",
}
INTERPRETABLE_MODEL_NAMES = {
    "pls_regressor", "multitask_elastic_net_regressor", "multitask_lasso_regressor"
}
SCALING_POLICY = {
    name: (bool(policy["features"]), bool(policy["targets"]))
    for name, policy in ACTIVE["evaluation"]["scaling_policies"].items()
}
MODEL_NAMES = list(ACTIVE["evaluation"]["models"])
MODEL_WORKERS = int(ACTIVE["run"]["threads"])


class TabNetRegressorAdapter:
    """Sklearn-style CPU adapter for joint multi-output TabNet regression."""

    def __init__(self, **params: Any) -> None:
        self.params = copy.deepcopy(dict(params))
        self.model: Any | None = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "TabNetRegressorAdapter":
        import torch
        from pytorch_tabnet.tab_model import TabNetRegressor

        resolved = copy.deepcopy(self.params)
        width = int(resolved.pop("width"))
        learning_rate = float(resolved.pop("learning_rate"))
        max_epochs = int(resolved.pop("max_epochs"))
        batch_size = int(resolved.pop("batch_size"))
        virtual_batch_size = int(resolved.pop("virtual_batch_size"))
        seed = int(resolved.get("seed", 42))
        random.seed(seed)
        np.random.seed(seed)
        torch.set_num_threads(MODEL_WORKERS)
        with contextlib.suppress(RuntimeError):
            torch.set_num_interop_threads(1)
        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True)
        self.model = TabNetRegressor(
            n_d=width,
            n_a=width,
            optimizer_fn=torch.optim.Adam,
            optimizer_params={"lr": learning_rate},
            **resolved,
        )
        self.model.fit(
            np.asarray(X, dtype=np.float32),
            np.asarray(y, dtype=np.float32),
            max_epochs=max_epochs,
            patience=0,
            batch_size=batch_size,
            virtual_batch_size=min(virtual_batch_size, batch_size),
            num_workers=0,
            drop_last=False,
            pin_memory=False,
            compute_importance=False,
        )
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("TabNetRegressorAdapter has not been fitted.")
        return np.asarray(self.model.predict(np.asarray(X, dtype=np.float32)), dtype=np.float64)


@dataclass
class FittedSurrogate:
    name: str
    estimator: Any
    feature_scaler: StandardScaler | None
    target_scaler: StandardScaler | None

    def predict(self, X: np.ndarray) -> np.ndarray:
        values = np.asarray(X, dtype=np.float64)
        if self.feature_scaler is not None:
            values = self.feature_scaler.transform(values)
        with joblib.parallel_backend("threading", n_jobs=MODEL_WORKERS):
            predicted = np.asarray(self.estimator.predict(values), dtype=np.float64)
        if predicted.ndim == 1:
            predicted = predicted[:, None]
        if self.target_scaler is not None:
            predicted = self.target_scaler.inverse_transform(predicted)
        predicted = np.asarray(predicted, dtype=np.float64)
        if predicted.shape != (len(values), len(STATE_COLUMNS)) or not np.all(np.isfinite(predicted)):
            raise RuntimeError(f"{self.name} returned an invalid prediction array {predicted.shape}.")
        return predicted


def normalized_metrics(y_true: np.ndarray, y_pred: np.ndarray, sigma: np.ndarray) -> dict[str, float]:
    sigma = np.asarray(sigma, dtype=np.float64)
    if sigma.shape != (20,) or np.any(~np.isfinite(sigma)) or np.any(sigma <= np.finfo(np.float64).eps):
        raise ValueError("All 20 training-target standard deviations must be finite and nonzero.")
    standardized = (np.asarray(y_pred) - np.asarray(y_true)) / sigma
    nmse = float(np.mean(np.square(standardized)))
    component_r2 = r2_score(y_true, y_pred, multioutput="raw_values", force_finite=False)
    if np.any(~np.isfinite(component_r2)):
        raise ValueError("Component R2 is undefined because a validation target is constant.")
    return {
        "nMSE": nmse,
        "nRMSE": float(math.sqrt(nmse)),
        "nMAE": float(np.mean(np.abs(standardized))),
        "macro_R2": float(np.mean(component_r2)),
    }


def sanitize_defaults(name: str, supplied: Mapping[str, Any]) -> dict[str, Any]:
    params = copy.deepcopy(dict(supplied))
    if name == "xgboost_regressor":
        params.update(random_state=42, n_jobs=1, verbosity=0, device="cpu")
    elif name == "lightgbm_regressor":
        params.update(random_state=42, n_jobs=1, verbosity=-1, subsample_freq=1, device_type="cpu")
    elif name == "catboost_regressor":
        params.update(loss_function="RMSE", random_seed=42, verbose=False, allow_writing_files=False, thread_count=1, task_type="CPU")
    elif name == "adaboost_regressor":
        params.update(random_state=42)
    elif name in {"random_forest_regressor", "extra_trees_regressor"}:
        params.update(random_state=42, n_jobs=MODEL_WORKERS)
    elif name == "knn_regressor":
        params.update(n_jobs=MODEL_WORKERS)
    elif name in {"multitask_elastic_net_regressor", "multitask_lasso_regressor"}:
        params.update(random_state=42)
    elif name == "ann_deep_regressor":
        if isinstance(params.get("hidden_layer_sizes"), list):
            params["hidden_layer_sizes"] = tuple(params["hidden_layer_sizes"])
        params.update(random_state=42, verbose=False)
    elif name == "tabnet_regressor":
        params.update(seed=42, device_name="cpu", verbose=0)
    return params


def build_estimator(name: str, params: Mapping[str, Any]) -> Any:
    resolved = sanitize_defaults(name, params)
    if name == "xgboost_regressor":
        from xgboost import XGBRegressor
        return MultiOutputRegressor(XGBRegressor(**resolved), n_jobs=MODEL_WORKERS)
    if name == "lightgbm_regressor":
        from lightgbm import LGBMRegressor
        return MultiOutputRegressor(LGBMRegressor(**resolved), n_jobs=MODEL_WORKERS)
    if name == "catboost_regressor":
        from catboost import CatBoostRegressor
        return MultiOutputRegressor(CatBoostRegressor(**resolved), n_jobs=MODEL_WORKERS)
    if name == "adaboost_regressor":
        return MultiOutputRegressor(AdaBoostRegressor(**resolved), n_jobs=MODEL_WORKERS)
    if name == "random_forest_regressor":
        return RandomForestRegressor(**resolved)
    if name == "extra_trees_regressor":
        return ExtraTreesRegressor(**resolved)
    if name == "svr_regressor":
        return MultiOutputRegressor(SVR(**resolved), n_jobs=MODEL_WORKERS)
    if name == "knn_regressor":
        return KNeighborsRegressor(**resolved)
    if name == "pls_regressor":
        return PLSRegression(**resolved)
    if name == "multitask_elastic_net_regressor":
        return MultiTaskElasticNet(**resolved)
    if name == "multitask_lasso_regressor":
        return MultiTaskLasso(**resolved)
    if name == "ann_deep_regressor":
        return MLPRegressor(**resolved)
    if name == "tabnet_regressor":
        return TabNetRegressorAdapter(**resolved)
    raise KeyError(name)


def fit_surrogate(name: str, X: np.ndarray, y: np.ndarray, params: Mapping[str, Any]) -> FittedSurrogate:
    X_values = np.asarray(X, dtype=np.float64)
    y_values = np.asarray(y, dtype=np.float64)
    scale_features, scale_targets = SCALING_POLICY[name]
    feature_scaler = StandardScaler().fit(X_values) if scale_features else None
    target_scaler = StandardScaler().fit(y_values) if scale_targets else None
    transformed_X = feature_scaler.transform(X_values) if feature_scaler is not None else X_values
    transformed_y = target_scaler.transform(y_values) if target_scaler is not None else y_values
    if name in {"multitask_elastic_net_regressor", "multitask_lasso_regressor"}:
        transformed_X = np.asfortranarray(transformed_X, dtype=np.float64)
        transformed_y = np.asfortranarray(transformed_y, dtype=np.float64)
    fitted = build_estimator(name, params)
    with joblib.parallel_backend("threading", n_jobs=MODEL_WORKERS):
        fitted.fit(transformed_X, transformed_y)
    return FittedSurrogate(name, fitted, feature_scaler, target_scaler)


def standardized_coefficient_frame(fitted: FittedSurrogate) -> pd.DataFrame:
    if fitted.name not in INTERPRETABLE_MODEL_NAMES:
        raise ValueError(f"{fitted.name} is not in the prespecified interpretable-model group.")
    coefficient = np.asarray(fitted.estimator.coef_, dtype=np.float64)
    expected = (len(FEATURE_COLUMNS), len(STATE_COLUMNS))
    if coefficient.shape == expected[::-1]:
        coefficient = coefficient.T
    if coefficient.shape != expected:
        raise RuntimeError(f"Unexpected coefficient shape for {fitted.name}: {coefficient.shape}.")
    frame = pd.DataFrame(coefficient, columns=STATE_COLUMNS)
    frame.insert(0, "feature", FEATURE_COLUMNS)
    frame["row_l2_norm"] = np.linalg.norm(coefficient, axis=1)
    frame["active_at_1e-12"] = frame["row_l2_norm"] > 1e-12
    frame.insert(0, "coefficient_scale", "standardized_feature_to_standardized_target")
    frame.insert(0, "model_category", MODEL_CATEGORIES[fitted.name])
    frame.insert(0, "model_key", fitted.name)
    frame.insert(0, "model", MODEL_LABELS[fitted.name])
    return frame


stage_complete("split_tests", nested_sizes=SAMPLE_SIZES, outer_folds=OUTER_FOLDS)
print(f"Prepared {len(SAMPLE_SIZES)} nested sizes and model adapters: {[MODEL_LABELS[name] for name in MODEL_NAMES]}")


## Nested hyperparameter selection

Each benchmark method receives a separate seeded TPE study inside each outer training partition at the largest data scale. Study databases and every trial are persisted so interrupted searches can resume without changing evaluated trials.


In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def suggest_parameters(trial: optuna.Trial, search_space: Mapping[str, Any]) -> dict[str, Any]:
    values: dict[str, Any] = {}
    for name, spec_value in search_space.items():
        spec = dict(spec_value)
        kind = str(spec["type"])
        if kind == "int":
            values[name] = trial.suggest_int(name, int(spec["low"]), int(spec["high"]), log=bool(spec.get("log", False)))
        elif kind == "float":
            values[name] = trial.suggest_float(
                name, float(spec["low"]), float(spec["high"]), log=bool(spec.get("log", False))
            )
        elif kind == "categorical":
            values[name] = trial.suggest_categorical(name, list(spec["choices"]))
        else:
            raise ValueError(f"Unsupported search-space type {kind!r} for {name}.")
    return values


def tune_model(
    name: str,
    row_indices: np.ndarray,
    *,
    study_label: str,
) -> tuple[dict[str, Any], float, Path]:
    model_config = ACTIVE["evaluation"]["models"][name]
    study_dir = DIRS["tuning"] / name
    study_dir.mkdir(parents=True, exist_ok=True)
    database_path = study_dir / f"{study_label}.sqlite3"
    storage_url = "sqlite:///" + database_path.resolve().as_posix()
    sampler_state_path = study_dir / f"{study_label}.sampler.pkl"
    if sampler_state_path.exists():
        with sampler_state_path.open("rb") as handle:
            sampler = pickle.load(handle)
        if not isinstance(sampler, optuna.samplers.TPESampler):
            raise RuntimeError("Persisted Optuna sampler has an invalid type.")
    else:
        sampler = optuna.samplers.TPESampler(seed=int(ACTIVE["evaluation"]["seed"]))
    study = optuna.create_study(
        study_name=f"{name}_{study_label}",
        direction=str(ACTIVE["evaluation"]["nested_cv"]["direction"]),
        sampler=sampler,
        pruner=optuna.pruners.NopPruner(),
        storage=storage_url,
        load_if_exists=True,
    )
    if study.trials and not sampler_state_path.exists():
        raise RuntimeError("Cannot resume an Optuna study without its persisted sampler state.")
    indices = np.asarray(row_indices, dtype=int)
    fold_ids = inner_fold_ids(indices, label=study_label)
    assignment_path = DIRS["splits"] / f"inner_{study_label}.parquet"
    expected_assignment = pd.DataFrame({"row_index": indices, "inner_fold": fold_ids})
    if assignment_path.exists():
        observed_assignment = pd.read_parquet(assignment_path)
        if not observed_assignment.equals(expected_assignment):
            raise RuntimeError(f"Persisted inner assignment differs for {study_label}.")
    else:
        atomic_parquet(assignment_path, expected_assignment)

    X_all = ID_DATA.loc[indices, FEATURE_COLUMNS].to_numpy(np.float64)
    y_all = ID_DATA.loc[indices, TARGET_COLUMNS].to_numpy(np.float64)

    def objective(trial: optuna.Trial) -> float:
        parameters = sanitize_defaults(name, model_config["training_defaults"])
        parameters.update(suggest_parameters(trial, model_config["search_space"]))
        scores: list[float] = []
        for inner_fold in range(INNER_FOLDS):
            train_mask = fold_ids != inner_fold
            validation_mask = fold_ids == inner_fold
            train_y = y_all[train_mask]
            sigma = train_y.std(axis=0, ddof=0)
            fitted = fit_surrogate(name, X_all[train_mask], train_y, parameters)
            predicted = fitted.predict(X_all[validation_mask])
            scores.append(normalized_metrics(y_all[validation_mask], predicted, sigma)["nMSE"])
        return float(np.mean(scores))

    timing_path = study_dir / f"{study_label}.timing.json"
    prior_elapsed = float(read_json(timing_path)["search_seconds"]) if timing_path.exists() else 0.0
    completed_trials = [trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE]
    remaining_trials = max(0, N_TRIALS - len(completed_trials))
    elapsed = prior_elapsed
    if remaining_trials:
        started = time.perf_counter()
        def persist_sampler(current_study: optuna.Study, _trial: optuna.trial.FrozenTrial) -> None:
            atomic_bytes(sampler_state_path, pickle.dumps(sampler, protocol=5))
            cumulative = sum(
                trial.duration.total_seconds()
                for trial in current_study.trials
                if trial.duration is not None and trial.state.is_finished()
            )
            atomic_json(
                timing_path,
                {"search_seconds": cumulative, "completed_trials": len([trial for trial in current_study.trials if trial.state.is_finished()])},
            )

        study.optimize(
            objective,
            n_trials=remaining_trials,
            timeout=None,
            n_jobs=1,
            gc_after_trial=True,
            callbacks=[persist_sampler],
        )
        _wall_elapsed = time.perf_counter() - started
        elapsed = sum(
            trial.duration.total_seconds()
            for trial in study.trials
            if trial.duration is not None and trial.state.is_finished()
        )
        atomic_json(timing_path, {"search_seconds": elapsed, "completed_trials": len([trial for trial in study.trials if trial.state.is_finished()])})
    completed_trials = [trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE]
    if len(completed_trials) != N_TRIALS:
        raise RuntimeError(f"Study {name}/{study_label} has {len(completed_trials)} complete trials; expected {N_TRIALS}.")
    trials_path = study_dir / f"{study_label}.trials.csv"
    atomic_csv(trials_path, study.trials_dataframe(), index=False)
    selected = sanitize_defaults(name, model_config["training_defaults"])
    selected.update(study.best_params)
    atomic_json(study_dir / f"{study_label}.selected.json", selected)
    return selected, elapsed, trials_path


print("Optuna orchestration is ready; studies execute in the evaluation cell.")


## Paired in-distribution benchmark

Raw and projected predictions are scored on identical outer-validation rows. Row-level predictions, physical diagnostics, component metrics, composite metrics, setup/search timings, and repeated inference timings are written as each model-size-fold unit completes.


In [ ]:
def physical_diagnostics(
    raw: np.ndarray,
    projected: ProjectionResult,
    influent: np.ndarray,
    sigma: np.ndarray,
) -> pd.DataFrame:
    tau = float(ACTIVE["projection"]["tau"])
    raw_residual = (A_MATRIX @ raw.T - A_MATRIX @ influent.T).T
    projected_residual = (A_MATRIX @ projected.values.T - A_MATRIX @ influent.T).T
    raw_negative = np.minimum(raw / sigma, 0.0)
    projected_negative = np.minimum(projected.values / sigma, 0.0)
    displacement = np.linalg.norm((projected.values - raw) / sigma, axis=1)
    return pd.DataFrame(
        {
            "raw_conservation_l2": np.linalg.norm(raw_residual, axis=1),
            "projected_conservation_l2": np.linalg.norm(projected_residual, axis=1),
            "raw_negative_standardized_l1": np.sum(np.abs(raw_negative), axis=1),
            "projected_negative_standardized_l1": np.sum(np.abs(projected_negative), axis=1),
            "raw_mass_violation": np.linalg.norm(raw_residual, axis=1) > tau,
            "projected_mass_violation": np.linalg.norm(projected_residual, axis=1) > tau,
            "raw_nonnegative_violation": np.min(raw, axis=1) < -tau,
            "projected_nonnegative_violation": np.min(projected.values, axis=1) < -tau,
            "standardized_displacement_l2": displacement,
            "backtracking_alpha": projected.alpha,
            "backtracking_active": projected.alpha < 1.0,
            "projection_solver": projected.solver,
            "raw_minimum_component": np.min(raw, axis=1),
            "projected_minimum_component": np.min(projected.values, axis=1),
        }
    )


def component_metric_rows(
    y_true: np.ndarray,
    raw: np.ndarray,
    projected: np.ndarray,
    sigma: np.ndarray,
    metadata: Mapping[str, Any],
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    units = ACTIVE["simulation"]["workbook"]["state_units"]
    displacement = projected - raw
    for index, component in enumerate(STATE_COLUMNS):
        scale = float(sigma[index])
        if not np.isfinite(scale) or scale <= np.finfo(np.float64).eps:
            raise ValueError(f"Target {component} has zero or invalid training variance.")
        for prediction_type, values in (("raw", raw), ("projected", projected)):
            error = values[:, index] - y_true[:, index]
            standardized_error = error / scale
            negative = np.minimum(values[:, index], 0.0)
            rows.append(
                {
                    **metadata,
                    "prediction_type": prediction_type,
                    "component": component,
                    "unit": units[component],
                    "MSE": float(np.mean(np.square(error))),
                    "RMSE": float(np.sqrt(np.mean(np.square(error)))),
                    "MAE": float(np.mean(np.abs(error))),
                    "bias": float(np.mean(error)),
                    "maximum_absolute_error": float(np.max(np.abs(error))),
                    "R2": float(r2_score(y_true[:, index], values[:, index], force_finite=False)),
                    "nMSE_component": float(np.mean(np.square(standardized_error))),
                    "nRMSE_component": float(np.sqrt(np.mean(np.square(standardized_error)))),
                    "nMAE_component": float(np.mean(np.abs(standardized_error))),
                    "negative_count": int(np.sum(values[:, index] < 0.0)),
                    "negative_magnitude_mean": float(np.mean(np.abs(negative))),
                    "projection_displacement_mean": float(np.mean(np.abs(displacement[:, index]))) if prediction_type == "projected" else np.nan,
                    "projection_displacement_standardized_mean": float(np.mean(np.abs(displacement[:, index]) / scale)) if prediction_type == "projected" else np.nan,
                }
            )
    return rows


def composite_metric_rows(
    y_true: np.ndarray,
    raw: np.ndarray,
    projected: np.ndarray,
    metadata: Mapping[str, Any],
) -> list[dict[str, Any]]:
    true_composite = y_true @ COMPOSITION.T
    composite_units = {"COD": "g COD/m^3", "TN": "g N/m^3", "TP": "g P/m^3", "TSS": "g TSS/m^3"}
    rows: list[dict[str, Any]] = []
    for prediction_type, values in (("raw", raw @ COMPOSITION.T), ("projected", projected @ COMPOSITION.T)):
        for index, composite in enumerate(MEASURED_COLUMNS):
            error = values[:, index] - true_composite[:, index]
            rows.append(
                {
                    **metadata,
                    "prediction_type": prediction_type,
                    "composite": composite,
                    "unit": composite_units[composite],
                    "MSE": float(np.mean(np.square(error))),
                    "RMSE": float(np.sqrt(np.mean(np.square(error)))),
                    "MAE": float(np.mean(np.abs(error))),
                    "bias": float(np.mean(error)),
                    "maximum_absolute_error": float(np.max(np.abs(error))),
                    "R2": float(r2_score(true_composite[:, index], values[:, index], force_finite=False)),
                }
            )
    return rows


def timed_inference(
    fitted: FittedSurrogate,
    X_validation: np.ndarray,
    influent_validation: np.ndarray,
) -> tuple[dict[str, float], pd.DataFrame]:
    timing_config = ACTIVE["evaluation"]["timing"]
    batch_size = int(timing_config["batch_size"])
    repeats = int(timing_config["measured_runs"])
    warmups = int(timing_config["warmup_runs"])
    cycle = np.arange(batch_size, dtype=int) % len(X_validation)
    X_batch = X_validation[cycle]
    influent_batch = influent_validation[cycle]
    for _ in range(warmups):
        _ = fitted.predict(X_batch)
    raw_times: list[float] = []
    reference_prediction: np.ndarray | None = None
    maximum_repeated_difference = 0.0
    for _ in range(repeats):
        started = time.perf_counter()
        repeated_prediction = fitted.predict(X_batch)
        raw_times.append(time.perf_counter() - started)
        if reference_prediction is None:
            reference_prediction = repeated_prediction.copy()
        else:
            maximum_repeated_difference = max(
                maximum_repeated_difference,
                float(np.max(np.abs(repeated_prediction - reference_prediction))),
            )
    if maximum_repeated_difference > 1e-12:
        raise RuntimeError(f"Repeated {fitted.name} predictions differ by {maximum_repeated_difference:.3e}.")
    fixed_raw = fitted.predict(X_batch)
    for _ in range(warmups):
        _ = project_batch(fixed_raw, influent_batch)
    projection_times: list[float] = []
    for _ in range(repeats):
        started = time.perf_counter()
        _ = project_batch(fixed_raw, influent_batch)
        projection_times.append(time.perf_counter() - started)
    for _ in range(warmups):
        warm_raw = fitted.predict(X_batch)
        _ = project_batch(warm_raw, influent_batch)
    end_to_end_times: list[float] = []
    for _ in range(repeats):
        started = time.perf_counter()
        current_raw = fitted.predict(X_batch)
        _ = project_batch(current_raw, influent_batch)
        end_to_end_times.append(time.perf_counter() - started)
    repetitions = pd.DataFrame(
        {
            "repetition": np.arange(repeats, dtype=int),
            "raw_seconds": raw_times,
            "projection_seconds": projection_times,
            "end_to_end_seconds": end_to_end_times,
        }
    )
    summary = {
        "raw_latency_ms_per_sample": float(np.median(raw_times) * 1000.0 / batch_size),
        "projection_latency_ms_per_sample": float(np.median(projection_times) * 1000.0 / batch_size),
        "end_to_end_latency_ms_per_sample": float(np.median(end_to_end_times) * 1000.0 / batch_size),
        "maximum_repeated_prediction_difference": maximum_repeated_difference,
    }
    return summary, repetitions


X_ID = ID_DATA[FEATURE_COLUMNS].to_numpy(np.float64)
Y_ID = ID_DATA[TARGET_COLUMNS].to_numpy(np.float64)
IN_ID = ID_DATA[INPUT_COMPONENT_COLUMNS].to_numpy(np.float64)
row_index_by_sample = dict(zip(ID_DATA["sample_id"].astype(str), np.arange(len(ID_DATA)), strict=True))

metric_records: list[dict[str, Any]] = []
timing_records: list[dict[str, Any]] = []
component_records: list[dict[str, Any]] = []
composite_records: list[dict[str, Any]] = []
selected_outer: dict[tuple[str, int], dict[str, Any]] = {}
search_seconds: dict[tuple[str, int], float] = {}

for model_name in MODEL_NAMES:
    for outer_fold in range(OUTER_FOLDS):
        maximum_subset = SPLITS[SPLITS["master_position"] < SAMPLE_SIZES[-1]]
        outer_training_indices = maximum_subset.loc[
            maximum_subset["outer_fold"] != outer_fold, "row_index"
        ].to_numpy(dtype=int)
        selected, elapsed_search, _ = tune_model(
            model_name,
            outer_training_indices,
            study_label=f"outer_{outer_fold}",
        )
        selected_outer[(model_name, outer_fold)] = selected
        search_seconds[(model_name, outer_fold)] = elapsed_search

        for sample_size in SAMPLE_SIZES:
            unit_name = f"{model_name}_n{sample_size}_fold{outer_fold}"
            metric_path = DIRS["metrics"] / "units" / f"{unit_name}.json"
            prediction_path = DIRS["predictions"] / "id" / f"{unit_name}.parquet"
            timing_path = DIRS["timing"] / f"{unit_name}.parquet"
            if bool(ACTIVE["run"].get("resume", True)) and metric_path.exists() and prediction_path.exists() and timing_path.exists():
                saved = read_json(metric_path)
                metric_records.extend(saved["metrics"])
                timing_records.append(saved["timing"])
                component_records.extend(saved["component_metrics"])
                composite_records.extend(saved["composite_metrics"])
                continue
            subset = SPLITS[SPLITS["master_position"] < sample_size]
            train_indices = subset.loc[subset["outer_fold"] != outer_fold, "row_index"].to_numpy(dtype=int)
            validation_indices = subset.loc[subset["outer_fold"] == outer_fold, "row_index"].to_numpy(dtype=int)
            if set(train_indices) & set(validation_indices):
                raise AssertionError("Outer training and validation rows overlap.")
            sigma = Y_ID[train_indices].std(axis=0, ddof=0)
            with threadpool_limits(limits=1):
                started = time.perf_counter()
                fitted = fit_surrogate(model_name, X_ID[train_indices], Y_ID[train_indices], selected)
                setup_seconds = time.perf_counter() - started
                raw = fitted.predict(X_ID[validation_indices])
            projected = project_batch(raw, IN_ID[validation_indices])
            diagnostics = physical_diagnostics(raw, projected, IN_ID[validation_indices], sigma)
            raw_metrics = normalized_metrics(Y_ID[validation_indices], raw, sigma)
            projected_metrics = normalized_metrics(Y_ID[validation_indices], projected.values, sigma)
            metadata = {
                "model": MODEL_LABELS[model_name],
                "model_key": model_name,
                "model_category": MODEL_CATEGORIES[model_name],
                "sample_size": sample_size,
                "outer_fold": outer_fold,
                "split": "id_validation",
            }
            unit_metrics = [
                {**metadata, "prediction_type": "raw", **raw_metrics},
                {**metadata, "prediction_type": "projected", **projected_metrics},
            ]
            component_unit = component_metric_rows(
                Y_ID[validation_indices], raw, projected.values, sigma, metadata
            )
            composite_unit = composite_metric_rows(
                Y_ID[validation_indices], raw, projected.values, metadata
            )
            with threadpool_limits(limits=1):
                timing_summary, timing_repetitions = timed_inference(
                    fitted, X_ID[validation_indices], IN_ID[validation_indices]
                )
            timing_summary = {
                **metadata,
                "setup_seconds": setup_seconds,
                "search_seconds": elapsed_search,
                **timing_summary,
            }
            prediction_frame = pd.DataFrame(
                {
                    "sample_id": ID_DATA.loc[validation_indices, "sample_id"].to_numpy(),
                    "row_index": validation_indices,
                    **{f"true_{name}": Y_ID[validation_indices, index] for index, name in enumerate(STATE_COLUMNS)},
                    **{f"raw_{name}": raw[:, index] for index, name in enumerate(STATE_COLUMNS)},
                    **{f"projected_{name}": projected.values[:, index] for index, name in enumerate(STATE_COLUMNS)},
                    **{f"influent_{name}": IN_ID[validation_indices, index] for index, name in enumerate(STATE_COLUMNS)},
                }
            )
            prediction_frame = pd.concat([prediction_frame, diagnostics], axis=1)
            atomic_parquet(prediction_path, prediction_frame)
            timing_repetitions = timing_repetitions.assign(**metadata)
            atomic_parquet(timing_path, timing_repetitions)
            atomic_json(
                metric_path,
                {
                    "metrics": unit_metrics,
                    "timing": timing_summary,
                    "component_metrics": component_unit,
                    "composite_metrics": composite_unit,
                    "physical_summary": {
                        column: float(diagnostics[column].mean())
                        for column in diagnostics.select_dtypes(include=[np.number, bool]).columns
                    },
                },
            )
            metric_records.extend(unit_metrics)
            timing_records.append(timing_summary)
            component_records.extend(component_unit)
            composite_records.extend(composite_unit)
            print(f"Completed {unit_name}")

ID_METRICS = pd.DataFrame(metric_records)
ID_TIMING = pd.DataFrame(timing_records)
COMPONENT_METRICS = pd.DataFrame(component_records)
COMPOSITE_METRICS = pd.DataFrame(composite_records)
atomic_csv(DIRS["metrics"] / "id_fold_metrics.csv", ID_METRICS)
atomic_csv(DIRS["timing"] / "id_fold_timing.csv", ID_TIMING)
atomic_csv(DIRS["metrics"] / "id_component_metrics.csv", COMPONENT_METRICS)
atomic_csv(DIRS["metrics"] / "id_composite_metrics.csv", COMPOSITE_METRICS)
stage_complete(
    "id_benchmark",
    models=len(MODEL_NAMES),
    sizes=len(SAMPLE_SIZES),
    folds=OUTER_FOLDS,
    units=len(MODEL_NAMES) * len(SAMPLE_SIZES) * OUTER_FOLDS,
)


## Frozen full-data refits and extrapolation

Final benchmark-model settings are selected using only the complete in-distribution pool. Only after every setting is frozen are the mild and severe mechanistic targets generated and opened for evaluation.


In [ ]:
full_indices = np.arange(len(ID_DATA), dtype=int)
selected_full: dict[str, dict[str, Any]] = {}
full_search_seconds: dict[str, float] = {}
for model_name in MODEL_NAMES:
    selected, elapsed, _ = tune_model(model_name, full_indices, study_label="full_id")
    selected_full[model_name] = selected
    full_search_seconds[model_name] = elapsed
atomic_json(DIRS["tuning"] / "full_id_selected.json", selected_full)
stage_complete("full_id_selection", models=len(selected_full))

FULL_MODEL_PATHS: dict[str, Path] = {}
full_setup_seconds: dict[str, float] = {}
for model_name in MODEL_NAMES:
    started = time.perf_counter()
    with threadpool_limits(limits=1):
        fitted = fit_surrogate(model_name, X_ID, Y_ID, selected_full[model_name])
    full_setup_seconds[model_name] = time.perf_counter() - started
    model_path = DIRS["models"] / f"{model_name}.joblib"
    temporary_model_path = model_path.with_suffix(".tmp")
    try:
        joblib.dump(fitted, temporary_model_path, compress=3)
        os.replace(temporary_model_path, model_path)
    except Exception:
        with contextlib.suppress(FileNotFoundError):
            temporary_model_path.unlink()
        raise
    register_artifact(f"final_model_{model_name}", model_path)
    FULL_MODEL_PATHS[model_name] = model_path
    if model_name in INTERPRETABLE_MODEL_NAMES:
        coefficient_path = DIRS["models"] / "interpretability" / f"{model_name}_standardized_coefficients.csv"
        atomic_csv(coefficient_path, standardized_coefficient_frame(fitted))
        register_artifact(f"standardized_coefficients_{model_name}", coefficient_path)
    del fitted
stage_complete("full_id_refits", models=len(MODEL_NAMES))

# OOD response columns are generated and revealed only after every full-ID setting,
# preprocessing transform and full-data refit is frozen.
OOD_DATASETS: dict[str, pd.DataFrame] = {}
OOD_ATTEMPTS: dict[str, pd.DataFrame] = {}
for regime in ("mild", "severe"):
    target = int(ACTIVE["simulation"]["ood"]["regimes"][regime]["n_samples"])
    OOD_DATASETS[regime], OOD_ATTEMPTS[regime] = load_or_generate_dataset(
        f"ood_{regime}", target, regime=regime
    )

OOD_COMBINED = pd.concat(
    [frame.assign(regime=regime) for regime, frame in OOD_DATASETS.items()], ignore_index=True
)
X_OOD = OOD_COMBINED[FEATURE_COLUMNS].to_numpy(np.float64)
Y_OOD = OOD_COMBINED[TARGET_COLUMNS].to_numpy(np.float64)
IN_OOD = OOD_COMBINED[INPUT_COMPONENT_COLUMNS].to_numpy(np.float64)
FULL_SIGMA = Y_ID.std(axis=0, ddof=0)
ood_metric_records: list[dict[str, Any]] = []
ood_component_records: list[dict[str, Any]] = []
ood_composite_records: list[dict[str, Any]] = []
ood_timing_records: list[dict[str, Any]] = []

for model_name in MODEL_NAMES:
    fitted = joblib.load(FULL_MODEL_PATHS[model_name])
    for regime in ("mild", "severe"):
        mask = OOD_COMBINED["regime"].to_numpy() == regime
        with threadpool_limits(limits=1):
            raw = fitted.predict(X_OOD[mask])
        projected = project_batch(raw, IN_OOD[mask])
        diagnostics = physical_diagnostics(raw, projected, IN_OOD[mask], FULL_SIGMA)
        with threadpool_limits(limits=1):
            ood_timing_summary, ood_timing_repetitions = timed_inference(
                fitted, X_OOD[mask], IN_OOD[mask]
            )
        metadata = {
            "model": MODEL_LABELS[model_name],
            "model_key": model_name,
            "model_category": MODEL_CATEGORIES[model_name],
            "regime": regime,
            "split": "ood",
        }
        raw_metrics = normalized_metrics(Y_OOD[mask], raw, FULL_SIGMA)
        projected_metrics = normalized_metrics(Y_OOD[mask], projected.values, FULL_SIGMA)
        for prediction_type, metrics in (("raw", raw_metrics), ("projected", projected_metrics)):
            ood_metric_records.append(
                {
                    **metadata,
                    "prediction_type": prediction_type,
                    "setup_seconds": full_setup_seconds[model_name],
                    "search_seconds": full_search_seconds[model_name],
                    **ood_timing_summary,
                    **metrics,
                    "mass_violation_rate": float(diagnostics[f"{prediction_type}_mass_violation"].mean()),
                    "nonnegative_violation_rate": float(diagnostics[f"{prediction_type}_nonnegative_violation"].mean()),
                }
            )
        ood_component_records.extend(
            component_metric_rows(Y_OOD[mask], raw, projected.values, FULL_SIGMA, metadata)
        )
        ood_composite_records.extend(
            composite_metric_rows(Y_OOD[mask], raw, projected.values, metadata)
        )
        timing_metadata = {
            **metadata,
            "setup_seconds": full_setup_seconds[model_name],
            "search_seconds": full_search_seconds[model_name],
            **ood_timing_summary,
        }
        ood_timing_records.append(timing_metadata)
        atomic_parquet(
            DIRS["timing"] / f"ood_{model_name}_{regime}.parquet",
            ood_timing_repetitions.assign(**metadata),
        )
        prediction_frame = pd.DataFrame(
            {
                "sample_id": OOD_COMBINED.loc[mask, "sample_id"].to_numpy(),
                "regime": regime,
                **{f"true_{name}": Y_OOD[mask, index] for index, name in enumerate(STATE_COLUMNS)},
                **{f"raw_{name}": raw[:, index] for index, name in enumerate(STATE_COLUMNS)},
                **{f"projected_{name}": projected.values[:, index] for index, name in enumerate(STATE_COLUMNS)},
                **{f"influent_{name}": IN_OOD[mask, index] for index, name in enumerate(STATE_COLUMNS)},
            }
        )
        prediction_frame = pd.concat([prediction_frame, diagnostics], axis=1)
        atomic_parquet(DIRS["predictions"] / "ood" / f"{model_name}_{regime}.parquet", prediction_frame)
    del fitted
    print(f"Completed final OOD evaluation for {MODEL_LABELS[model_name]}")

OOD_METRICS = pd.DataFrame(ood_metric_records)
OOD_COMPONENT_METRICS = pd.DataFrame(ood_component_records)
OOD_COMPOSITE_METRICS = pd.DataFrame(ood_composite_records)
OOD_TIMING = pd.DataFrame(ood_timing_records)
atomic_csv(DIRS["metrics"] / "ood_metrics.csv", OOD_METRICS)
atomic_csv(DIRS["metrics"] / "ood_component_metrics.csv", OOD_COMPONENT_METRICS)
atomic_csv(DIRS["metrics"] / "ood_composite_metrics.csv", OOD_COMPOSITE_METRICS)
atomic_csv(DIRS["timing"] / "ood_timing.csv", OOD_TIMING)

ood_validation_rows = []
for regime in ("mild", "severe"):
    data = OOD_DATASETS[regime]
    attempts = OOD_ATTEMPTS[regime]
    outputs = data[TARGET_COLUMNS].to_numpy(np.float64)
    inputs = data[INPUT_COMPONENT_COLUMNS].to_numpy(np.float64)
    residual = np.linalg.norm((A_MATRIX @ outputs.T - A_MATRIX @ inputs.T).T, axis=1)
    ood_validation_rows.append(
        {
            "set": regime,
            "attempted": len(attempts),
            "accepted": len(data),
            "acceptance_rate": len(data) / len(attempts),
            "maximum_conservation_l2": float(residual.max()),
            "minimum_component": float(outputs.min()),
            "one_variable_candidate_rate": float(
                attempts["perturbed_variables"].astype(str).str.count(";").eq(0).mean()
            ),
            "two_variable_candidate_rate": float(
                attempts["perturbed_variables"].astype(str).str.count(";").eq(1).mean()
            ),
            "one_variable_accepted_rate": float(
                data["perturbed_variables"].astype(str).str.count(";").eq(0).mean()
            ),
            "two_variable_accepted_rate": float(
                data["perturbed_variables"].astype(str).str.count(";").eq(1).mean()
            ),
        }
    )
OOD_VALIDATION = pd.DataFrame(ood_validation_rows)
atomic_csv(DIRS["datasets"] / "ood_validation.csv", OOD_VALIDATION)
stage_complete("ood_benchmark", regimes=["mild", "severe"], models=len(MODEL_NAMES))


## Publication tables, figures, and fold-variability summaries

All reported means and sample standard deviations describe variation across the five fixed outer folds; they are not confidence intervals or evidence of statistical significance. Every publication asset is emitted together with its tabular source.


In [ ]:
ARTICLE_SOURCE = DIRS["article"] / "source_data"
ARTICLE_SOURCE.mkdir(parents=True, exist_ok=True)


def summarize_folds(frame: pd.DataFrame, groups: list[str], values: list[str]) -> pd.DataFrame:
    summary = frame.groupby(groups, sort=False)[values].agg(["mean", "std"]).reset_index()
    summary.columns = [
        column if isinstance(column, str) else "_".join(part for part in column if part)
        for column in summary.columns.to_flat_index()
    ]
    counts = frame.groupby(groups, sort=False).size().rename("fold_count").reset_index()
    return summary.merge(counts, on=groups, how="left", validate="one_to_one")


ID_FOLD_SUMMARY = summarize_folds(
    ID_METRICS,
    ["model", "model_key", "model_category", "sample_size", "prediction_type"],
    ["nMSE", "nRMSE", "nMAE", "macro_R2"],
)
TIMING_FOLD_SUMMARY = summarize_folds(
    ID_TIMING,
    ["model", "model_key", "model_category", "sample_size"],
    [
        "setup_seconds", "search_seconds", "raw_latency_ms_per_sample",
        "projection_latency_ms_per_sample", "end_to_end_latency_ms_per_sample",
        "maximum_repeated_prediction_difference",
    ],
)

physical_rows: list[dict[str, Any]] = []
for prediction_path in sorted((DIRS["predictions"] / "id").glob("*.parquet")):
    match = re.fullmatch(r"(.+)_n(\d+)_fold(\d+)", prediction_path.stem)
    if match is None:
        raise ValueError(f"Unexpected ID prediction filename {prediction_path.name}.")
    model_key, size_text, fold_text = match.groups()
    frame = pd.read_parquet(prediction_path)
    for prediction_type in ("raw", "projected"):
        conservation = frame[f"{prediction_type}_conservation_l2"].to_numpy(np.float64)
        negative_l1 = frame[f"{prediction_type}_negative_standardized_l1"].to_numpy(np.float64)
        physical_rows.append(
            {
                "model": MODEL_LABELS[model_key],
                "model_key": model_key,
                "model_category": MODEL_CATEGORIES[model_key],
                "sample_size": int(size_text),
                "outer_fold": int(fold_text),
                "prediction_type": prediction_type,
                "conservation_l2_mean": float(np.mean(conservation)),
                "conservation_l2_std": float(np.std(conservation, ddof=1)),
                "conservation_l2_median": float(np.median(conservation)),
                "conservation_l2_q95": float(np.quantile(conservation, 0.95)),
                "conservation_l2_max": float(np.max(conservation)),
                "negative_standardized_l1_mean": float(np.mean(negative_l1)),
                "negative_standardized_l1_q95": float(np.quantile(negative_l1, 0.95)),
                "mass_violation_rate": float(frame[f"{prediction_type}_mass_violation"].mean()),
                "nonnegative_violation_rate": float(frame[f"{prediction_type}_nonnegative_violation"].mean()),
                "standardized_displacement_l2_mean": float(frame["standardized_displacement_l2"].mean()),
                "backtracking_rate": float(frame["backtracking_active"].mean()),
                "backtracking_alpha_mean": float(frame["backtracking_alpha"].mean()),
                "backtracking_alpha_min": float(frame["backtracking_alpha"].min()),
            }
        )
ID_PHYSICAL_FOLD = pd.DataFrame(physical_rows)
ID_PHYSICAL_SUMMARY = summarize_folds(
    ID_PHYSICAL_FOLD,
    ["model", "model_key", "model_category", "sample_size", "prediction_type"],
    [
        "conservation_l2_mean", "conservation_l2_std", "conservation_l2_median",
        "conservation_l2_q95", "conservation_l2_max", "negative_standardized_l1_mean",
        "negative_standardized_l1_q95", "mass_violation_rate", "nonnegative_violation_rate",
        "standardized_displacement_l2_mean", "backtracking_rate", "backtracking_alpha_mean",
        "backtracking_alpha_min",
    ],
)

ood_physical_rows: list[dict[str, Any]] = []
for model_key in MODEL_NAMES:
    for regime in ("mild", "severe"):
        frame = pd.read_parquet(DIRS["predictions"] / "ood" / f"{model_key}_{regime}.parquet")
        for prediction_type in ("raw", "projected"):
            conservation = frame[f"{prediction_type}_conservation_l2"].to_numpy(np.float64)
            ood_physical_rows.append(
                {
                    "model": MODEL_LABELS[model_key],
                    "model_key": model_key,
                    "model_category": MODEL_CATEGORIES[model_key],
                    "regime": regime,
                    "prediction_type": prediction_type,
                    "conservation_l2_mean": float(np.mean(conservation)),
                    "conservation_l2_std": float(np.std(conservation, ddof=1)),
                    "conservation_l2_median": float(np.median(conservation)),
                    "conservation_l2_q95": float(np.quantile(conservation, 0.95)),
                    "conservation_l2_max": float(np.max(conservation)),
                    "mass_violation_rate": float(frame[f"{prediction_type}_mass_violation"].mean()),
                    "nonnegative_violation_rate": float(frame[f"{prediction_type}_nonnegative_violation"].mean()),
                    "standardized_displacement_l2_mean": float(frame["standardized_displacement_l2"].mean()),
                    "backtracking_rate": float(frame["backtracking_active"].mean()),
                    "backtracking_alpha_mean": float(frame["backtracking_alpha"].mean()),
                    "backtracking_alpha_min": float(frame["backtracking_alpha"].min()),
                }
            )
OOD_PHYSICAL = pd.DataFrame(ood_physical_rows)

summary_outputs = {
    "id_fold_summary.csv": ID_FOLD_SUMMARY,
    "id_physical_fold.csv": ID_PHYSICAL_FOLD,
    "id_physical_summary.csv": ID_PHYSICAL_SUMMARY,
    "timing_fold_summary.csv": TIMING_FOLD_SUMMARY,
    "ood_physical_summary.csv": OOD_PHYSICAL,
}
for filename, frame in summary_outputs.items():
    atomic_csv(DIRS["metrics"] / filename, frame)
    atomic_csv(ARTICLE_SOURCE / filename, frame)
for model_name in sorted(INTERPRETABLE_MODEL_NAMES):
    coefficient_frame = pd.read_csv(
        DIRS["models"] / "interpretability" / f"{model_name}_standardized_coefficients.csv"
    )
    atomic_csv(
        ARTICLE_SOURCE / f"{model_name}_standardized_coefficients.csv",
        coefficient_frame,
    )

largest_id = ID_FOLD_SUMMARY[ID_FOLD_SUMMARY["sample_size"] == SAMPLE_SIZES[-1]].copy()
largest_timing = TIMING_FOLD_SUMMARY[TIMING_FOLD_SUMMARY["sample_size"] == SAMPLE_SIZES[-1]].copy()
publication_tables = {
    "table_dataset_validation": pd.concat(
        [pd.DataFrame([{"set": "in-distribution", **dataset_validation}]), OOD_VALIDATION],
        ignore_index=True,
        sort=False,
    ),
    "table_largest_id_metrics": largest_id,
    "table_projection_physics": ID_PHYSICAL_SUMMARY[
        ID_PHYSICAL_SUMMARY["sample_size"] == SAMPLE_SIZES[-1]
    ].copy(),
    "table_ood_metrics": OOD_METRICS.merge(
        OOD_PHYSICAL,
        on=["model", "model_key", "model_category", "regime", "prediction_type"],
        how="left",
        validate="one_to_one",
        suffixes=("", "_physical"),
    ),
    "table_timing": largest_timing,
}
for table_name, table_frame in publication_tables.items():
    atomic_csv(DIRS["article"] / f"{table_name}.csv", table_frame)
    latex = table_frame.to_latex(index=False, escape=True, float_format=lambda value: f"{value:.6g}")
    atomic_bytes(DIRS["article"] / f"{table_name}.tex", latex.encode("utf-8"))

atomic_csv(ARTICLE_SOURCE / "id_fold_metrics.csv", ID_METRICS)
atomic_csv(ARTICLE_SOURCE / "id_component_metrics.csv", COMPONENT_METRICS)
atomic_csv(ARTICLE_SOURCE / "id_composite_metrics.csv", COMPOSITE_METRICS)
atomic_csv(ARTICLE_SOURCE / "ood_metrics.csv", OOD_METRICS)
atomic_csv(ARTICLE_SOURCE / "ood_component_metrics.csv", OOD_COMPONENT_METRICS)
atomic_csv(ARTICLE_SOURCE / "ood_composite_metrics.csv", OOD_COMPOSITE_METRICS)
atomic_csv(ARTICLE_SOURCE / "ood_timing.csv", OOD_TIMING)
atomic_csv(ARTICLE_SOURCE / "ood_dataset_validation.csv", OOD_VALIDATION)

mpl.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 130,
        "savefig.bbox": "tight",
    }
)
MODEL_COLORS = {
    "xgboost_regressor": "#577590",
    "lightgbm_regressor": "#264653",
    "catboost_regressor": "#E76F51",
    "adaboost_regressor": "#F4A261",
    "random_forest_regressor": "#8D99AE",
    "extra_trees_regressor": "#6B705C",
    "svr_regressor": "#BC4749",
    "knn_regressor": "#E9C46A",
    "pls_regressor": "#5C6770",
    "multitask_elastic_net_regressor": "#2A9D8F",
    "multitask_lasso_regressor": "#4D908E",
    "ann_deep_regressor": "#6D597A",
    "tabnet_regressor": "#9C6644",
}
MODEL_MARKERS = {
    "xgboost_regressor": "o",
    "lightgbm_regressor": "s",
    "catboost_regressor": "^",
    "adaboost_regressor": "v",
    "random_forest_regressor": "D",
    "extra_trees_regressor": "d",
    "svr_regressor": "P",
    "knn_regressor": ">",
    "pls_regressor": "<",
    "multitask_elastic_net_regressor": "h",
    "multitask_lasso_regressor": "+",
    "ann_deep_regressor": "p",
    "tabnet_regressor": "*",
}


def save_figure(fig: mpl.figure.Figure, stem: str) -> None:
    for extension in ("pdf", "png"):
        destination = DIRS["article"] / f"{stem}.{extension}"
        temporary = destination.parent / f".{destination.name}.{os.getpid()}.tmp"
        try:
            fig.savefig(temporary, format=extension, dpi=300 if extension == "png" else None)
            os.replace(temporary, destination)
        finally:
            with contextlib.suppress(FileNotFoundError):
                temporary.unlink()


figure1, axis1 = plt.subplots(figsize=(8.4, 5.2))
raw_learning = ID_FOLD_SUMMARY[ID_FOLD_SUMMARY["prediction_type"] == "raw"]
for model_key in MODEL_NAMES:
    rows = raw_learning[raw_learning["model_key"] == model_key].sort_values("sample_size")
    axis1.plot(rows["sample_size"], rows["nRMSE_mean"], marker=MODEL_MARKERS[model_key], linewidth=1.25, markersize=4, label=MODEL_LABELS[model_key], color=MODEL_COLORS[model_key])
    axis1.fill_between(rows["sample_size"], rows["nRMSE_mean"] - rows["nRMSE_std"], rows["nRMSE_mean"] + rows["nRMSE_std"], color=MODEL_COLORS[model_key], alpha=0.08)
axis1.set(xlabel="Nested sample total", ylabel="nRMSE", title="In-distribution learning curves (raw predictions)")
axis1.legend(ncol=3, fontsize=7, frameon=False)
save_figure(figure1, "figure_learning_curves")
plt.close(figure1)

delta = ID_FOLD_SUMMARY.pivot(index=["model", "model_key", "sample_size"], columns="prediction_type", values="nRMSE_mean").reset_index()
delta["delta_nRMSE"] = delta["projected"] - delta["raw"]
delta_matrix = np.vstack([
    delta[delta["model_key"] == model_key].sort_values("sample_size")["delta_nRMSE"].to_numpy()
    for model_key in MODEL_NAMES
])
figure2, axis2 = plt.subplots(figsize=(8.3, 4.6))
limit = max(float(np.max(np.abs(delta_matrix))), np.finfo(float).eps)
image2 = axis2.imshow(delta_matrix, aspect="auto", cmap="coolwarm", vmin=-limit, vmax=limit)
axis2.set_xticks(np.arange(len(SAMPLE_SIZES)), [str(value) for value in SAMPLE_SIZES], rotation=45, ha="right")
axis2.set_yticks(np.arange(len(MODEL_NAMES)), [MODEL_LABELS[name] for name in MODEL_NAMES])
axis2.set(xlabel="Nested sample total", title="Projection change in nRMSE (projected minus raw)")
figure2.colorbar(image2, ax=axis2, label="Delta nRMSE")
save_figure(figure2, "figure_projection_change")
plt.close(figure2)
atomic_csv(ARTICLE_SOURCE / "figure_projection_change.csv", delta)

severe = OOD_METRICS[OOD_METRICS["regime"] == "severe"].copy()
severe_pivot = severe.pivot(index="model_key", columns="prediction_type", values="nRMSE").reindex(MODEL_NAMES)
x_positions = np.arange(len(MODEL_NAMES))
figure3, axis3 = plt.subplots(figsize=(10.4, 5.2))
axis3.bar(x_positions - 0.19, severe_pivot["raw"], width=0.38, label="Raw", color="#4c78a8")
axis3.bar(x_positions + 0.19, severe_pivot["projected"], width=0.38, label="Projected", color="#f58518")
axis3.set_xticks(x_positions, [MODEL_LABELS[name] for name in MODEL_NAMES], rotation=45, ha="right")
axis3.set(ylabel="nRMSE", title="Severe extrapolation performance")
axis3.legend(frameon=False)
save_figure(figure3, "figure_severe_extrapolation")
plt.close(figure3)
atomic_csv(ARTICLE_SOURCE / "figure_severe_extrapolation.csv", severe)

timing_ordered = largest_timing.set_index("model_key").reindex(MODEL_NAMES)
figure4, axis4 = plt.subplots(figsize=(10.4, 5.0))
axis4.bar(x_positions, timing_ordered["raw_latency_ms_per_sample_mean"], color=[MODEL_COLORS[name] for name in MODEL_NAMES])
axis4.errorbar(x_positions, timing_ordered["raw_latency_ms_per_sample_mean"], yerr=timing_ordered["raw_latency_ms_per_sample_std"], fmt="none", ecolor="black", capsize=2, linewidth=0.8)
axis4.set_xticks(x_positions, [MODEL_LABELS[name] for name in MODEL_NAMES], rotation=45, ha="right")
axis4.set(ylabel="Milliseconds per sample", title="Complete 20-component inference latency")
save_figure(figure4, "figure_inference_latency")
plt.close(figure4)
atomic_csv(ARTICLE_SOURCE / "figure_inference_latency.csv", largest_timing)

asset_rows = []
for asset_path in sorted(DIRS["article"].rglob("*")):
    if asset_path.is_file():
        asset_rows.append(
            {
                "path": asset_path.relative_to(DIRS["article"]).as_posix(),
                "bytes": asset_path.stat().st_size,
                "sha256": sha256_file(asset_path),
            }
        )
ARTICLE_ASSETS = pd.DataFrame(asset_rows)
atomic_csv(DIRS["article"] / "asset_inventory.csv", ARTICLE_ASSETS)
stage_complete("publication_assets", tables=len(publication_tables), figures=4)
print(f"Generated {len(publication_tables)} publication tables and four all-model figures.")


## Terminal acceptance and immutable manifest

Completion is recorded only after every projection, dataset count, profile, and artifact assertion passes. A cryptographic sidecar checks the final manifest itself.


In [ ]:
tau = float(ACTIVE["projection"]["tau"])
if bool(ID_PHYSICAL_FOLD.loc[ID_PHYSICAL_FOLD["prediction_type"] == "projected", "mass_violation_rate"].gt(0.0).any()):
    raise RuntimeError("At least one projected ID sample violates conservation.")
if bool(ID_PHYSICAL_FOLD.loc[ID_PHYSICAL_FOLD["prediction_type"] == "projected", "nonnegative_violation_rate"].gt(0.0).any()):
    raise RuntimeError("At least one projected ID sample violates non-negativity.")
if bool(OOD_PHYSICAL.loc[OOD_PHYSICAL["prediction_type"] == "projected", "mass_violation_rate"].gt(0.0).any()):
    raise RuntimeError("At least one projected OOD sample violates conservation.")
if bool(OOD_PHYSICAL.loc[OOD_PHYSICAL["prediction_type"] == "projected", "nonnegative_violation_rate"].gt(0.0).any()):
    raise RuntimeError("At least one projected OOD sample violates non-negativity.")
if len(ID_DATA) != ID_TARGET or any(
    len(OOD_DATASETS[regime]) != int(ACTIVE["simulation"]["ood"]["regimes"][regime]["n_samples"])
    for regime in ("mild", "severe")
):
    raise RuntimeError("An accepted mechanistic dataset has the wrong row count.")
expected_cardinalities = {
    "id_metrics": len(MODEL_NAMES) * len(SAMPLE_SIZES) * OUTER_FOLDS * 2,
    "id_timing": len(MODEL_NAMES) * len(SAMPLE_SIZES) * OUTER_FOLDS,
    "id_physics": len(MODEL_NAMES) * len(SAMPLE_SIZES) * OUTER_FOLDS * 2,
    "ood_metrics": len(MODEL_NAMES) * 2 * 2,
    "ood_timing": len(MODEL_NAMES) * 2,
}
observed_cardinalities = {
    "id_metrics": len(ID_METRICS),
    "id_timing": len(ID_TIMING),
    "id_physics": len(ID_PHYSICAL_FOLD),
    "ood_metrics": len(OOD_METRICS),
    "ood_timing": len(OOD_TIMING),
}
if observed_cardinalities != expected_cardinalities:
    raise RuntimeError(
        f"Benchmark row-cardinality mismatch: expected={expected_cardinalities}, "
        f"observed={observed_cardinalities}."
    )
expected_model_keys = set(MODEL_NAMES)
for frame_name, frame in {
    "ID_METRICS": ID_METRICS,
    "ID_TIMING": ID_TIMING,
    "ID_PHYSICAL_FOLD": ID_PHYSICAL_FOLD,
    "OOD_METRICS": OOD_METRICS,
    "OOD_TIMING": OOD_TIMING,
}.items():
    if set(frame["model_key"].astype(str)) != expected_model_keys:
        raise RuntimeError(f"{frame_name} does not contain the complete 13-model roster.")
for model_name in INTERPRETABLE_MODEL_NAMES:
    coefficient_path = DIRS["models"] / "interpretability" / f"{model_name}_standardized_coefficients.csv"
    coefficient_frame = pd.read_csv(coefficient_path)
    if len(coefficient_frame) != len(FEATURE_COLUMNS) or set(coefficient_frame["feature"]) != set(FEATURE_COLUMNS):
        raise RuntimeError(f"Incomplete standardized coefficient artifact for {model_name}.")
if requested_profile == "full":
    if not ARTICLE_ELIGIBLE:
        raise RuntimeError("A full run must use the complete article profile.")
    if len(MODEL_NAMES) != 13 or len(SAMPLE_SIZES) != 11 or OUTER_FOLDS != 5 or INNER_FOLDS != 4 or N_TRIALS != 100:
        raise RuntimeError("A full run does not satisfy the article benchmark dimensions.")
else:
    MANIFEST["article_eligible"] = False

for artifact_name, artifact in MANIFEST["artifacts"].items():
    artifact_path = RUN_ROOT / artifact["path"]
    if not artifact_path.is_file() or sha256_file(artifact_path) != artifact["sha256"]:
        raise RuntimeError(f"Registered artifact {artifact_name!r} is missing or has changed.")

inventory: dict[str, dict[str, Any]] = {}
excluded_names = {"manifest.json", "manifest.sha256", "COMPLETED.json"}
for artifact_path in sorted(RUN_ROOT.rglob("*")):
    if artifact_path.is_file() and artifact_path.name not in excluded_names and ".tmp" not in artifact_path.name:
        relative = artifact_path.relative_to(RUN_ROOT).as_posix()
        inventory[relative] = {
            "bytes": artifact_path.stat().st_size,
            "sha256": sha256_file(artifact_path),
        }
MANIFEST["artifact_inventory"] = inventory
MANIFEST["status"] = "complete"
MANIFEST["completed_utc"] = utc_now()
MANIFEST["article_eligible"] = bool(ARTICLE_ELIGIBLE)
save_manifest()
manifest_digest = sha256_file(manifest_path)
atomic_bytes(RUN_ROOT / "manifest.sha256", f"{manifest_digest}  manifest.json\n".encode("ascii"))
atomic_json(
    RUN_ROOT / "COMPLETED.json",
    {
        "run_id": RUN_ID,
        "profile": requested_profile,
        "article_eligible": bool(ARTICLE_ELIGIBLE),
        "manifest_sha256": manifest_digest,
        "completed_utc": MANIFEST["completed_utc"],
    },
)
print(
    json.dumps(
        {
            "status": "complete",
            "run_root": RUN_ROOT.relative_to(ROOT).as_posix(),
            "article_eligible": bool(ARTICLE_ELIGIBLE),
            "manifest_sha256": manifest_digest,
            "artifacts": len(inventory),
        },
        indent=2,
    )
)
